# ⬡ MISSÃO EVORTEX — Mission Control AI
### Sistema Inteligente de Monitoramento da Cápsula Espacial

---

Este notebook contém o sistema completo **MISSÃO EVORTEX** — um painel de controle futurista
para monitoramento ambiental de uma cápsula espacial experimental com **Arduino Uno**.

### O que está incluído

| Componente | Descrição |
|---|---|
| 🛰 **Interface web completa** | Dashboard futurista com sensores, gráficos, histórico e cápsula animada |
| 🚨 **Popups de alerta crítico** | Abre automaticamente com protocolo de resposta passo a passo |
| 💬 **Chatbot ARIA** | IA conversacional com base de dados completa sobre a missão |
| 📊 **Painel de análise** | Exibe condicionais, repetição, vetores e funções em tempo real |
| 🔌 **Web Serial API** | Conexão direta com o Arduino via navegador |

### Como usar

1. Execute a **Célula 1** — instala dependências e configura o ambiente
2. Execute a **Célula 2** — exibe a interface completa no notebook
3. Execute a **Célula 3** — testa o chatbot ARIA diretamente em Python
4. Execute a **Célula 4** — gera o arquivo `missao_evortex.html` para abrir no Chrome/Edge

> **Dica:** Para usar com Arduino real, abra o arquivo `.html` gerado na **Célula 4** no
> Google Chrome ou Microsoft Edge e clique em **Conectar Arduino**.

---
## 📦 Célula 1 — Configuração do Ambiente

Verifica a versão do Python, instala dependências e prepara o ambiente para exibir a interface.

In [ ]:
import sys
import os
import json
from IPython.display import display, HTML, Javascript, clear_output
from datetime import datetime

print("=" * 55)
print("  MISSION CONTROL AI — MISSÃO EVORTEX")
print("=" * 55)
print(f"  Python       : {sys.version.split()[0]}")
print(f"  Ambiente     : Google Colab / Jupyter")
print(f"  Data/hora    : {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
print("=" * 55)
print()
print("  Sensores monitorados:")
sensores = [
    ("A0", "Sensor de Gás",           "MQ-2 ou similar"),
    ("A1", "Sensor de Fogo",          "Fotodiodo IR — lógica invertida"),
    ("A3", "Sensor de Luminosidade",  "LDR — verifica iluminação"),
    ("A4", "Sensor de Temperatura",   "Termistor NTC β=3950"),
    ("D6", "Buzzer de Emergência",    "Sirene 600–1400 Hz"),
]
for pino, nome, desc in sensores:
    print(f"    [{pino}] {nome:<28} — {desc}")
print()
print("  ✅ Ambiente configurado. Execute a Célula 2.")

---
## 🖥️ Célula 2 — Interface EVORTEX no Notebook

Renderiza o painel completo dentro do Google Colab.

> **Nota:** Algumas funcionalidades como Web Serial API e animações Canvas
> funcionam melhor abrindo o arquivo `.html` diretamente no Chrome/Edge (veja Célula 4).
> No Colab, use o **Modo Simulação** para explorar todas as funcionalidades.

In [ ]:
from IPython.display import display, HTML

# Carrega o HTML completo da Missão EVORTEX
# Inclui: interface, sensores, gráficos, chatbot ARIA e popups de alerta

EVORTEX_HTML = open('missao_evortex.html', 'r', encoding='utf-8').read() if os.path.exists('missao_evortex.html') else None

if EVORTEX_HTML:
    display(HTML(f'<div style="width:100%;height:800px;border:1px solid #1e4180;border-radius:4px;overflow:hidden"><iframe srcdoc={repr(EVORTEX_HTML)} style="width:100%;height:100%;border:none"></iframe></div>'))
    print("✅ Interface carregada.")
else:
    # Fallback: exibe preview mínimo
    display(HTML("""
    <div style="background:#06080f;color:#00d4ff;font-family:'Courier New',monospace;
         padding:30px;border:1px solid #1e4180;border-radius:4px;text-align:center">
      <div style="font-size:1.8rem;font-weight:700;letter-spacing:.2em;
           text-shadow:0 0 20px #00d4ff">⬡ MISSÃO EVORTEX</div>
      <div style="font-size:.75rem;color:#5a7a9e;margin-top:8px;letter-spacing:.1em">
        SISTEMA INTELIGENTE DE MONITORAMENTO DA CÁPSULA ESPACIAL
      </div>
      <div style="margin-top:20px;color:#c4d4ec;font-size:.8rem">
        Execute a Célula 4 para gerar o arquivo HTML e abrir no Chrome/Edge
      </div>
    </div>
    """))
    print("⚠️  Arquivo HTML não encontrado. Execute a Célula 4 primeiro.")

---
## 💬 Célula 3 — Chatbot ARIA em Python

Versão Python do chatbot ARIA com a base de dados completa da missão.
Permite conversar com a ARIA diretamente no terminal do Colab.

**Tópicos que a ARIA conhece:**
`missão · objetivos · hardware · sensores · gás · fogo · luz · temperatura ·
buzzer · status · alertas · protocolos · código · interface · arduino · gráficos ·
análise · configurações · serial`

In [ ]:
# ══════════════════════════════════════════════════════════════
#  ARIA — Base de Dados Completa da Missão EVORTEX
#  Versão Python do chatbot integrado na interface web
# ══════════════════════════════════════════════════════════════

# ── Estado simulado dos sensores (atualizado pelo Arduino em modo real) ────────
estado_missao = {
    "gas":       None,   # valor bruto A0 (0–1023)
    "fogo":      None,   # valor bruto A1 (0–1023, invertido)
    "luz":       None,   # valor bruto A3 (0–1023)
    "temp_c":    None,   # temperatura calculada em °C
    "temp_raw":  None,   # valor bruto A4
    "buzzer":    False,  # estado do buzzer D6
    "conectado": False,  # Arduino conectado
    "modo":      "sim",  # 'real' ou 'sim'
}

# ── Limites configuráveis (espelham os padrões da interface) ──────────────────
limites = {
    "gas_lim":    200,   # alerta acima deste valor
    "fire_lim":   300,   # alerta abaixo deste valor (lógica invertida)
    "luz_lim":    50,    # falha abaixo deste valor
    "temp_warn":  30,    # atenção acima de 30°C
    "temp_crit":  40,    # crítico acima de 40°C
}

# ══════════════════════════════════════════════════════════════
#  BASE DE CONHECIMENTO COMPLETA
# ══════════════════════════════════════════════════════════════

def kb_missao():
    return """
🛰  O QUE É A MISSÃO EVORTEX?

A Missão EVORTEX é uma missão espacial experimental desenvolvida como projeto
acadêmico. O objetivo é simular o monitoramento ambiental interno de uma cápsula
espacial utilizando sensores reais conectados a um Arduino Uno.

O sistema monitora em tempo real as condições internas da cápsula:
  • Temperatura interna da cabine
  • Presença de fogo ou chamas
  • Vazamento de gás
  • Funcionamento do sistema de iluminação

Os dados são enviados pelo Arduino via porta serial USB e exibidos na interface
web em tempo real, com alertas automáticos, gráficos, histórico e protocolos.
"""

def kb_objetivos():
    return """
🎯 OBJETIVOS DA MISSÃO EVORTEX

Técnicos:
  • Monitorar continuamente 4 sensores ambientais da cápsula
  • Detectar situações de risco e emitir alertas automáticos
  • Acionar o buzzer de emergência (D6) em caso de falha
  • Transmitir telemetria em tempo real via porta serial (9600 baud)

Acadêmicos:
  • Demonstrar uso de estruturas condicionais, repetição, vetores e funções
  • Aplicar comunicação Arduino ↔ interface web (Web Serial API)
  • Construir um sistema completo de monitoramento com hardware real
  • Desenvolver uma IA conversacional (ARIA) com base de dados da missão
"""

def kb_hardware():
    return """
🔧 HARDWARE DA MISSÃO EVORTEX

Microcontrolador: Arduino Uno

Sensores conectados:
  A0 — Sensor de Gás        : MQ-2 ou similar (0–1023 raw)
  A1 — Sensor de Fogo       : Fotodiodo infravermelho (lógica INVERTIDA)
  A3 — Sensor de Luz        : LDR (Low Dependent Resistor)
  A4 — Sensor de Temperatura: Termistor NTC (β=3950, R₀=100kΩ, Rfix=6kΩ)

Atuadores:
  D6 — Buzzer piezoelétrico : sirene 600–1400 Hz (ativado por falha de luz)
  LED — Iluminação interna  : ligado diretamente no 5V (não controlado pelo Arduino)

Comunicação: USB Serial — 9600 baud
Interface  : Web Serial API (Google Chrome / Microsoft Edge)
"""

def kb_sensores():
    gl = limites['gas_lim']
    fl = limites['fire_lim']
    ll = limites['luz_lim']
    tw = limites['temp_warn']
    tc = limites['temp_crit']
    return f"""
🔬 SENSORES MONITORADOS — LIMITES EVORTEX

☁  GÁS (A0)
   Normal  : abaixo de {int(gl*0.7)} raw
   Atenção : {int(gl*0.7)} a {gl} raw
   Crítico : acima de {gl} raw

🔥 FOGO (A1) — LÓGICA INVERTIDA
   Normal  : acima de {fl+50} raw  (sem fogo)
   Atenção : {fl} a {fl+50} raw
   Crítico : abaixo de {fl} raw    (fogo detectado)
   ⚠  Valor ALTO = sem fogo | Valor BAIXO = fogo

💡 LUMINOSIDADE (A3)
   Funcionando : ≥ {ll} raw
   Falha       : < {ll} raw  → ativa buzzer D6 automaticamente

🌡  TEMPERATURA (A4 — NTC)
   Normal  : 18–30°C
   Atenção : 31–{tw}°C
   Crítico : acima de {tc}°C
   Parâmetros NTC: β=3950, R₀=100kΩ, Rfix=6kΩ, T₀=25°C
"""

def kb_gas():
    v = estado_missao['gas']
    return f"""
☁  SENSOR DE GÁS — PINO A0

O que mede : concentração de gás inflamável ou tóxico na câmara
Como funciona: valor analógico 0–1023. Maior valor = maior concentração

Limites:
  Normal  : < {int(limites['gas_lim']*0.7)} raw
  Atenção : {int(limites['gas_lim']*0.7)}–{limites['gas_lim']} raw
  Crítico : > {limites['gas_lim']} raw

Valor atual: {v if v is not None else 'Sem sinal (Arduino não conectado)'}

Se crítico:
  1. Interromper atividades na câmara
  2. Ativar ventilação forçada
  3. Identificar e isolar origem do vazamento
  4. Acionar sensor secundário para confirmação
  5. Se persistir: evacuar e selar o módulo
"""

def kb_fogo():
    v = estado_missao['fogo']
    return f"""
🔥 SENSOR DE FOGO — PINO A1

O que detecta: presença de chamas por radiação infravermelha

⚠  LÓGICA INVERTIDA:
  Valor ALTO (≈1023) = SEM FOGO  (situação normal)
  Valor BAIXO (≈0)   = FOGO DETECTADO (situação crítica)

Limites:
  Normal  : > {limites['fire_lim']+50} raw
  Atenção : {limites['fire_lim']}–{limites['fire_lim']+50} raw
  Crítico : < {limites['fire_lim']} raw

Valor atual: {v if v is not None else 'Sem sinal (Arduino não conectado)'}

Se crítico:
  1. Acionar extintor automático imediatamente
  2. Cortar alimentação elétrica do setor afetado
  3. Comunicar alerta à base terrestre
  4. Retirar tripulação do compartimento
  5. Se incontrolável: protocolo de evacuação total
"""

def kb_luz():
    v = estado_missao['luz']
    return f"""
💡 SENSOR DE LUMINOSIDADE — PINO A3

O que verifica: se a iluminação interna da cápsula está funcionando

Como funciona: lê o brilho via LDR.
  ≥ {limites['luz_lim']} raw → Iluminação funcionando
  <  {limites['luz_lim']} raw → FALHA DE ILUMINAÇÃO

Importante: o LED está ligado DIRETAMENTE no 5V.
O Arduino NÃO controla o LED — apenas verifica se há luz.

Valor atual: {v if v is not None else 'Sem sinal (Arduino não conectado)'}

Quando falha:
  → O buzzer D6 é ativado AUTOMATICAMENTE pelo Arduino
  → A interface exibe alerta vermelho pulsante
  → O popup de protocolo abre automaticamente

Protocolo de falha:
  1. Ativar iluminação de emergência (backup autônomo)
  2. Verificar fusíveis e conexões do circuito
  3. Inspecionar sensor LDR no pino A3
  4. Reduzir consumo elétrico geral
  5. Registrar falha e aguardar manutenção
"""

def kb_temperatura():
    v  = estado_missao['temp_c']
    vr = estado_missao['temp_raw']
    return f"""
🌡  SENSOR DE TEMPERATURA — PINO A4

Tipo: Termistor NTC (Negative Temperature Coefficient)

Parâmetros de calibração:
  Resistência nominal (R₀) : 100.000 Ω (a 25°C)
  Resistor fixo (Rfix)     : 6.000 Ω
  Coeficiente Beta (β)     : 3950
  Temperatura nominal (T₀) : 25°C

Fórmula (Steinhart-Hart simplificada / equação Beta):
  Rntc = Rfix × ((1023 / leitura) - 1)
  Tk   = 1 / ( 1/(T₀+273.15) + ln(Rntc/R₀)/β )
  Tc   = Tk - 273.15

Limites:
  Normal  : 18–30°C
  Atenção : 31–{limites['temp_warn']}°C
  Crítico : > {limites['temp_crit']}°C

Temperatura atual : {f"{v:.1f}°C" if v is not None else 'Sem sinal'}
Valor bruto A4    : {vr if vr is not None else '—'}

Protocolo térmico:
  1. Reduzir carga dos sistemas não essenciais
  2. Ativar sistema de resfriamento auxiliar
  3. Monitorar NTC a cada 30 segundos
  4. Verificar isolamento térmico do módulo
  5. Se > 45°C: iniciar protocolo de evacuação
"""

def kb_buzzer():
    estado = 'ATIVO — sirene em operação' if estado_missao['buzzer'] else 'Desligado — sistema estável'
    return f"""
🔔 BUZZER DE EMERGÊNCIA — PINO D6

O que é: buzzer piezoelétrico que emite sirene sonora de emergência

Quando ativa: Arduino ativa quando luminosidade < {limites['luz_lim']} (falha de iluminação)

Como funciona a sirene:
  O Arduino varia a frequência de 600 Hz até 1400 Hz ciclicamente.
  Sobe de 600 → 1400 Hz em passos de 10 Hz a cada 5ms.
  Depois desce de 1400 → 600 Hz — criando efeito de sirene.

Estado atual: {estado}

Na interface web:
  🔊 Botão "Som virtual" — simula o buzzer no navegador
  ✓  Botão "Confirmar"  — reconhece o alerta (não desativa o sensor físico)
"""

def kb_status():
    e = estado_missao
    l = limites
    # avaliar cada sensor
    def avaliar(val, lim, invertido=False):
        if val is None: return '— (sem sinal)'
        if invertido:
            if val < lim:       return f'{val} raw  ⚠ CRÍTICO'
            if val < lim*1.35:  return f'{val} raw  ⚡ ATENÇÃO'
            return f'{val} raw  ✅ NORMAL'
        else:
            if val >= lim:      return f'{val} raw  ⚠ CRÍTICO'
            if val >= lim*0.7:  return f'{val} raw  ⚡ ATENÇÃO'
            return f'{val} raw  ✅ NORMAL'

    gas_s  = avaliar(e['gas'],  l['gas_lim'])
    fire_s = avaliar(e['fogo'], l['fire_lim'], invertido=True)
    luz_s  = f"{e['luz']} raw  {'⚠ FALHA' if e['luz'] is not None and e['luz'] < l['luz_lim'] else '✅ OK'}" if e['luz'] is not None else '— (sem sinal)'
    tmp_s  = f"{e['temp_c']:.1f}°C  {'⚠ CRÍTICO' if e['temp_c'] >= l['temp_crit'] else '⚡ ATENÇÃO' if e['temp_c'] >= l['temp_warn'] else '✅ NORMAL'}" if e['temp_c'] is not None else '— (sem sinal)'
    buz_s  = '🔔 ATIVO' if e['buzzer'] else '🔕 Desligado'
    conn_s = '🟢 Conectado' if e['conectado'] else ('🟡 Simulação' if e['modo'] == 'sim' else '🔴 Offline')

    return f"""
📊 STATUS ATUAL DA MISSÃO EVORTEX

  ☁  Gás (A0)          : {gas_s}
  🔥 Fogo (A1)         : {fire_s}
  💡 Luz (A3)          : {luz_s}
  🌡  Temperatura (A4) : {tmp_s}
  🔔 Buzzer D6         : {buz_s}
  📡 Arduino           : {conn_s}
"""

def kb_alertas():
    e = estado_missao
    l = limites
    lista = []
    if e['gas']  is not None and e['gas']  >= l['gas_lim']:              lista.append(f"🔴 GÁS CRÍTICO: {e['gas']} raw (limite {l['gas_lim']})")
    elif e['gas'] is not None and e['gas'] >= l['gas_lim']*0.7:          lista.append(f"🟡 Gás em atenção: {e['gas']} raw")
    if e['fogo'] is not None and e['fogo'] < l['fire_lim']:               lista.append(f"🔴 FOGO DETECTADO: sensor {e['fogo']} raw (abaixo de {l['fire_lim']})")
    elif e['fogo'] is not None and e['fogo'] < l['fire_lim']*1.35:       lista.append(f"🟡 Fogo em atenção: sensor {e['fogo']} raw")
    if e['luz']  is not None and e['luz']  < l['luz_lim']:               lista.append(f"🔴 FALHA DE ILUMINAÇÃO: {e['luz']} raw (mínimo {l['luz_lim']})")
    if e['temp_c'] is not None and e['temp_c'] >= l['temp_crit']:        lista.append(f"🔴 TEMPERATURA CRÍTICA: {e['temp_c']:.1f}°C (limite {l['temp_crit']}°C)")
    elif e['temp_c'] is not None and e['temp_c'] >= l['temp_warn']:      lista.append(f"🟡 Temperatura elevada: {e['temp_c']:.1f}°C")
    if not lista:
        return "✅ Nenhum alerta ativo. Todos os sistemas dentro dos limites operacionais."
    return "Alertas detectados:
" + "
".join(f"  {a}" for a in lista)

def kb_protocolos():
    return """
📋 PROTOCOLOS DE EMERGÊNCIA — MISSÃO EVORTEX

🔥 PROTOCOLO DE INCÊNDIO (Sensor Fogo A1)
   1. Acionar extintor automático imediatamente
   2. Cortar alimentação elétrica do setor afetado
   3. Comunicar alerta à base terrestre
   4. Retirar tripulação do compartimento afetado
   5. Se incontrolável: iniciar evacuação total da cápsula

☁  PROTOCOLO DE GÁS (Sensor Gás A0)
   1. Interromper TODAS as atividades na câmara
   2. Ativar ventilação forçada imediatamente
   3. Identificar e isolar a origem do vazamento
   4. Acionar sensor secundário para confirmação
   5. Se persistir: evacuar e selar o módulo afetado

🌡  PROTOCOLO TÉRMICO (Sensor Temperatura A4)
   1. Reduzir carga dos sistemas não essenciais
   2. Ativar sistema de resfriamento auxiliar da câmara
   3. Monitorar sensor NTC (pino A4) a cada 30 segundos
   4. Verificar isolamento térmico do módulo principal
   5. Se temperatura > 45°C: iniciar protocolo de evacuação

💡 PROTOCOLO DE ILUMINAÇÃO (Sensor Luz A3)
   1. Ativar iluminação de emergência (backup autônomo)
   2. Verificar fusíveis e conexões do circuito elétrico
   3. Inspecionar sensor LDR no pino A3 (possível obstrução)
   4. Reduzir consumo elétrico geral da cápsula
   5. Registrar falha no histórico e aguardar manutenção
"""

def kb_codigo():
    return """
💻 CÓDIGO ARDUINO — MISSÃO EVORTEX

Linguagem     : C/C++ (Arduino IDE)
Baud rate     : 9600
Intervalo     : 500ms entre cada envio de telemetria

Formato enviado pelo Arduino:
  GAS:46,FOGO:987,LUZ:59,TEMP_RAW:57,TEMP:24.3,BUZZER:0

Funções principais do sketch:
  calcularTemperaturaCelsius(leitura)
    → Converte valor bruto A4 em °C via fórmula Steinhart-Hart (Beta)
    → Rejeita leituras saturadas (≤1 ou ≥1022) retornando NAN

  tocarSirene(agora)
    → Varre frequência 600–1400 Hz no buzzer D6
    → Intervalo de 5ms entre cada passo de 10 Hz

  desligarSirene()
    → Para o buzzer e reseta frequência para 600 Hz

  setup()
    → Inicia Serial em 9600 baud
    → Configura pinos dos sensores e buzzer

  loop()
    → Lê os 4 sensores analogicamente
    → Calcula temperatura via NTC
    → Decide se ativa/desativa buzzer (apenas por falha de luz)
    → A cada 500ms: envia linha compacta + bloco verbose
"""

def kb_interface():
    return """
🖥️  COMO USAR A INTERFACE EVORTEX

MODOS DE OPERAÇÃO:
  Real       → conecta ao Arduino via Web Serial API
  Simulação  → gera dados automaticamente sem Arduino

BARRA DE SIMULAÇÃO (aparece no modo Sim):
  ▶ Auto        — ciclo automático alternando todos os cenários
  🔥 Fogo        — sensor A1 cai para valor crítico
  ☁  Gás         — sensor A0 sobe acima do limite
  💡 Baixa Luz   — sensor A3 cai abaixo de 50
  🌡 Temp Alta   — temperatura sobe acima de 40°C
  📡 Falha Comm  — simula perda de sinal por 5 segundos
  ↺ Restaurar   — volta todos os sensores ao estado normal

PAINEL DOS SENSORES:
  4 cards com valor bruto, barra de intensidade e estado atual
  Verde = Normal | Amarelo = Atenção | Vermelho = Crítico

PAINEL DIREITO:
  Cápsula animada  — muda para vermelho em alertas
  Status da missão — verde/amarelo/vermelho com pulso
  Buzzer D6        — reflete estado do hardware
  ⚙ Config         — ajuste de todos os limites

GRÁFICOS:
  4 abas: Gás / Fogo / Luz / Temperatura
  Últimos 60 pontos em tempo real

HISTÓRICO:
  Todos os eventos registrados com horário, sensor e valor
  Filtros por sensor e nível | Exportação CSV | Cópia de relatório

PAINEL DE ANÁLISE (parte inferior):
  Mostra em tempo real as estruturas de programação sendo executadas
  Condicionais | Repetição | Vetores | Funções
"""

def kb_arduino():
    return """
📡 COMO CONECTAR O ARDUINO

Requisitos:
  • Navegador Google Chrome ou Microsoft Edge
  • Arduino Uno com o sketch EVORTEX gravado
  • Cabo USB conectado ao computador

Passo a passo:
  1. Conecte o Arduino via cabo USB
  2. Selecione o modo Real no topo da interface
  3. Clique em ⬡ Conectar Arduino
  4. Escolha a porta COM na janela que abrirá
  5. Aguarde os dados aparecerem nos cards

Formato serial aceito:
  GAS:46,FOGO:987,LUZ:59,TEMP_RAW:57,TEMP:24.3,BUZZER:0

Também aceita:
  • Verbose do Monitor Serial do Arduino IDE
  • JSON: {"gas":46,"fogo":987,"luz":59,"temperaturaC":24.3,"buzzer":false}

Se não conectar:
  • Verifique se o baud rate está em 9600
  • Confirme a porta COM correta no Gerenciador de Dispositivos
  • Use apenas Chrome ou Edge (Firefox não suporta Web Serial API)
"""

def kb_graficos():
    return """
📈 GRÁFICOS E HISTÓRICO

Gráficos em tempo real:
  • 4 gráficos de linha — um por sensor
  • Últimos 60 registros exibidos
  • Atualização a cada leitura
  • Abas: Gás / Fogo / Luz / Temperatura
  • Botão Limpar — apaga as séries

Histórico da Missão:
  • Registra eventos com: horário, sensor, valor, nível
  • Filtros por sensor: gás, fogo, luz, temp, sistema
  • Filtros por nível: info, atenção, crítico, ok
  • ⬇ CSV   — exporta tudo em planilha Excel
  • ⎘ Copiar — copia relatório para área de transferência
  • ACK      — marca alerta como reconhecido pelo operador
"""

def kb_serial():
    return """
🔌 WEB SERIAL API — COMUNICAÇÃO ARDUINO ↔ INTERFACE

O que é: API nativa dos navegadores Chrome/Edge que permite
comunicação serial direta com dispositivos USB como o Arduino.

Como funciona neste projeto:
  1. Clique em Conectar Arduino — navegador pede permissão para a porta COM
  2. Interface abre conexão em 9600 baud
  3. Loop contínuo lê o stream serial (while true)
  4. Cada linha é processada por processarDadosArduino()
  5. Cards, gráficos e chatbot atualizam em tempo real

Parser multi-formato (processarDadosArduino):
  • Compacto : GAS:46,FOGO:987,LUZ:59,TEMP_RAW:57,TEMP:24.3,BUZZER:0
  • Verbose  : linhas do Monitor Serial do Arduino IDE
  • JSON     : {"gas":46,"fogo":987,...}

Timeout de comunicação:
  Se não receber dados por 3 segundos → exibe banner de falha
  Quando comunicação volta → banner some e histórico registra

Compatível com: Google Chrome 89+ e Microsoft Edge 89+
"""

def kb_analise():
    return """
🔍 PAINEL DE ANÁLISE DE ESTRUTURAS DE PROGRAMAÇÃO

Localização: parte inferior da interface (role para baixo)

Mostra em tempo real como o código está sendo executado,
demonstrando as 4 estruturas de programação do projeto:

⎇ CONDICIONAIS (if/else):
  Cada bloco acende quando o critério é verdadeiro:
  • if gás ≥ limite          → alerta vermelho
  • if fogo < limite         → lógica invertida
  • if luz < limiteLuz       → falha de iluminação
  • if temp ≥ crítica        → temperatura crítica
  • else if temp ≥ atenção   → temperatura elevada
  • if raw ≤1 ou ≥1022       → NTC saturado → NaN
  • if Δt > timeout          → falha de comunicação

↺ REPETIÇÃO (loops):
  Contadores ao vivo de cada laço em execução:
  • loop() Arduino     — ciclos de telemetria
  • while(serial)      — bytes lidos da porta
  • forEach(campos)    — campos do pacote CSV
  • setInterval 1s     — verificação de comunicação e relógio

▦ VETORES (arrays):
  Visualização célula a célula dos arrays principais:
  • hist[]             — histórico de eventos (i/!/✕/✓)
  • chartData.gas[]    — série temporal do gás
  • chartData.temp[]   — série temporal da temperatura

ƒ FUNÇÕES (log de chamadas):
  Últimas 14 chamadas com: nome, argumentos e valor de retorno
  Ex: calcTempC(raw=57) → 24.3°C
"""

def kb_configuracoes():
    l = limites
    return f"""
⚙  CONFIGURAÇÕES — COMO AJUSTAR OS LIMITES

Localização: painel direito → aba ⚙ Config

Campos editáveis:
  Limite Gás (alerta acima)  : {l['gas_lim']} raw   (padrão: 200)
  Limite Fogo (alerta abaixo): {l['fire_lim']} raw   (padrão: 300)
  Limite Luz mínima          : {l['luz_lim']} raw    (padrão: 50 — igual ao Arduino)
  Temp. Atenção              : {l['temp_warn']}°C      (padrão: 30°C)
  Temp. Crítica              : {l['temp_crit']}°C      (padrão: 40°C)
  Timeout comunicação        : 3s       (padrão: 3 segundos)
  Volume alarme virtual      : 0–1      (padrão: 0.4)

Botão ↺ Restaurar padrões: volta todos os campos ao valor original

⚠  Atenção: o limite de luz (50) é o mesmo valor hardcoded no Arduino.
   Mantenha os dois sincronizados para evitar inconsistências.
"""

def kb_ajuda():
    return """
💬 ARIA — Assistente Inteligente da Missão EVORTEX

Tenho uma base de dados completa sobre todo este sistema.
Posso responder sobre os seguintes tópicos:

  missão        → o que é a Missão EVORTEX
  objetivos     → objetivos técnicos e acadêmicos
  hardware      → Arduino, sensores e componentes
  sensores      → todos os 4 sensores e seus limites
  gás           → sensor A0, limites, protocolo
  fogo          → sensor A1, lógica invertida, protocolo
  luz           → sensor A3, buzzer D6, protocolo
  temperatura   → sensor NTC A4, fórmula, protocolo
  buzzer        → como funciona a sirene do D6
  status        → dados em tempo real dos sensores
  alertas       → o que está crítico neste momento
  protocolos    → ações de emergência detalhadas
  código        → explicação do sketch Arduino
  interface     → como usar todos os botões
  arduino       → como conectar o hardware
  gráficos      → histórico e exportação
  análise       → painel de estruturas de programação
  configurações → como ajustar os limites
  serial        → Web Serial API

Digite qualquer um desses termos ou faça uma pergunta livre.
"""

# ══════════════════════════════════════════════════════════════
#  MOTOR NLP — interpreta linguagem natural
# ══════════════════════════════════════════════════════════════

import re

NLP_RULES = [
    (r'o que (é|eh|e|foi)|apresent|sobre (a|essa|esta) miss|evortex|qual.*miss', kb_missao),
    (r'objetiv|propósit|finalidad|para que|pra que serve',                       kb_objetivos),
    (r'hardware|componente|placa|circuito|montagem',                              kb_hardware),
    (r'^sensores?$|quais.*sensor|lista.*sensor|todos.*sensor',                   kb_sensores),
    (r'gás|gas|mq.?2|vazamento|vapores',                                         kb_gas),
    (r'fogo|incêndio|incendio|chama|fumaça|infravermelho',                       kb_fogo),
    (r'luz|ldr|luminosidade|iluminação|iluminacao|led',                          kb_luz),
    (r'temperatura|térmico|termico|calor|aquecimento|ntc|termistor',             kb_temperatura),
    (r'buzzer|sirene|alarme|som|barulho|d6|beep',                                kb_buzzer),
    (r'^status$|telemetria|dados.*atual|estado.*sensor',                          kb_status),
    (r'alerta|emergência|emergencia|crítico|critico|perigo|socorro',              kb_alertas),
    (r'protocol|resposta.*emergência|o que fazer|ação.*critica',                 kb_protocolos),
    (r'código|codigo|sketch|função|setup|loop|programação',                      kb_codigo),
    (r'como (usar|funciona|operar|mexer|navegar)|botões?|interface|painel',      kb_interface),
    (r'conectar|conexão|porta com|usb|baud',                                     kb_arduino),
    (r'gráfico|grafico|histórico|historico|csv|exportar|log',                    kb_graficos),
    (r'análise|analise|estrutura|condicional|vetor|repetição',                   kb_analise),
    (r'configuração|configuracao|limite|ajustar|editar|limite',                  kb_configuracoes),
    (r'serial|comunicação serial|transmissão|pacote|formato',                    kb_serial),
    (r'simul|demo|apresentação|sem arduino|teste',
     lambda: """🔄 MODO SIMULAÇÃO\n\nAcesse o modo Simulação no botão no topo da interface.\n\nBotões disponíveis:\n  ▶ Auto      — ciclos automáticos\n  🔥 Fogo      — simula chama\n  ☁  Gás       — simula vazamento\n  💡 Baixa Luz — simula falha de iluminação\n  🌡 Temp Alta  — temperatura crítica\n  📡 Falha Comm — perda de sinal\n  ↺ Restaurar  — volta ao normal"""),
]

EXATOS = {
    'missao': kb_missao, 'missão': kb_missao,
    'objetivos': kb_objetivos,
    'hardware': kb_hardware,
    'sensores': kb_sensores,
    'gas': kb_gas, 'gás': kb_gas,
    'fogo': kb_fogo,
    'luz': kb_luz,
    'temperatura': kb_temperatura,
    'buzzer': kb_buzzer,
    'status': kb_status,
    'alertas': kb_alertas,
    'protocolos': kb_protocolos,
    'codigo': kb_codigo, 'código': kb_codigo,
    'interface': kb_interface,
    'arduino': kb_arduino,
    'graficos': kb_graficos, 'gráficos': kb_graficos,
    'analise': kb_analise, 'análise': kb_analise,
    'configuracoes': kb_configuracoes, 'configurações': kb_configuracoes,
    'serial': kb_serial,
    'ajuda': kb_ajuda, 'help': kb_ajuda,
    'oi': lambda: 'Olá! Sou ARIA, IA de controle da Missão EVORTEX.\nDigite ajuda para ver tudo que sei.',
    'olá': lambda: 'Olá! Digite ajuda para ver minha base de conhecimento completa.',
    'ola': lambda: 'Olá! Digite ajuda para ver minha base de conhecimento completa.',
}

def aria_responder(msg):
    m = msg.strip().lower()
    if m in EXATOS:
        return EXATOS[m]()
    for padrao, fn in NLP_RULES:
        if re.search(padrao, msg, re.IGNORECASE):
            return fn()
    return (
        f'Não encontrei resposta específica para "{msg}".\n\n'
        'Tente uma dessas palavras-chave:\n'
        'missão · sensores · gás · fogo · luz · temperatura · buzzer ·\n'
        'status · alertas · protocolos · código · interface · arduino\n\n'
        'Ou faça uma pergunta mais específica.'
    )

# ══════════════════════════════════════════════════════════════
#  LOOP INTERATIVO DO CHATBOT
# ══════════════════════════════════════════════════════════════

SEP = '=' * 55
SEP_S = '-' * 55

print(SEP)
print('  💬 ARIA — Missão EVORTEX  |  Base de dados completa')
print(SEP)
print(aria_responder('ajuda'))
print()
print(SEP_S)
print('  Digite um tópico ou pergunta. "sair" para encerrar.')
print(SEP_S)

while True:
    try:
        entrada = input('\n  [OPERADOR] → ').strip()
    except (EOFError, KeyboardInterrupt):
        print('\n  [ARIA] Sessão encerrada.')
        break
    if not entrada:
        continue
    if entrada.lower() in ('sair', 'exit', 'quit'):
        print('  [ARIA] Encerrando. Boa missão, operador.')
        break
    resposta = aria_responder(entrada)
    print()
    print(f'  [ARIA]')
    for linha in resposta.strip().split('\n'):
        print(f'  {linha}')

---
## 💾 Célula 4 — Gerar o Arquivo HTML

Salva o arquivo `missao_evortex.html` no Colab e disponibiliza para download.
Abra-o no **Google Chrome** ou **Edge** para usar a interface completa com Arduino.

In [ ]:
# Grava o arquivo HTML completo da Missão EVORTEX
# Para usar com Arduino real: abra no Chrome/Edge

EVORTEX_HTML_CONTENT = '<!DOCTYPE html>\n<html lang="pt-BR">\n<head>\n<meta charset="UTF-8">\n<meta name="viewport" content="width=device-width, initial-scale=1.0">\n<title>MISSÃO EVORTEX</title>\n<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>\n<style>\n/* ═══════════════════════════════════════════\n   BASE\n═══════════════════════════════════════════ */\n*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}\n:root{\n  --bg:       #06080f;\n  --panel:    rgba(9,13,24,0.92);\n  --card:     rgba(11,16,28,0.95);\n  --border:   rgba(30,65,120,0.4);\n  --b-bright: rgba(0,210,255,0.55);\n  --t1:       #c4d4ec;\n  --t2:       #5a7a9e;\n  --t3:       #2e4260;\n  --cyan:     #00d4ff;\n  --cyandim:  rgba(0,212,255,0.12);\n  --green:    #00e06a;\n  --grndim:   rgba(0,224,106,0.10);\n  --yellow:   #f0b500;\n  --ylwdim:   rgba(240,181,0,0.10);\n  --orange:   #ff7318;\n  --red:      #ff2828;\n  --reddim:   rgba(255,40,40,0.13);\n  --mono:     \'Courier New\',monospace;\n  --ui:       \'Segoe UI\',system-ui,sans-serif;\n}\nbody{background:var(--bg);color:var(--t1);font-family:var(--ui);min-height:100vh;overflow-x:hidden}\n\n/* ── scrollbar ── */\n::-webkit-scrollbar{width:4px;height:4px}\n::-webkit-scrollbar-thumb{background:var(--border);border-radius:2px}\n\n/* ── canvas bg ── */\n#bg-canvas{position:fixed;inset:0;pointer-events:none;z-index:0;opacity:0.35}\n\n/* ── wrapper ── */\n.wrap{position:relative;z-index:1;display:flex;flex-direction:column;min-height:100vh}\n\n/* ═══════════════════════════════════════════\n   HEADER\n═══════════════════════════════════════════ */\nheader{\n  display:flex;align-items:center;justify-content:space-between;gap:14px;\n  padding:12px 24px;border-bottom:1px solid var(--border);\n  background:rgba(4,6,14,0.97);backdrop-filter:blur(10px);\n  position:sticky;top:0;z-index:100\n}\n.brand-name{\n  font-family:var(--mono);font-size:1.4rem;font-weight:700;letter-spacing:.2em;\n  color:var(--cyan);text-shadow:0 0 18px var(--cyan),0 0 36px rgba(0,212,255,.25);line-height:1\n}\n.brand-sub{font-size:.62rem;color:var(--t2);letter-spacing:.1em;text-transform:uppercase;margin-top:3px}\n.hdr-mid{display:flex;gap:8px;align-items:center;flex-wrap:wrap;justify-content:center}\n.hdr-right{display:flex;flex-direction:column;align-items:flex-end;gap:3px;min-width:170px}\n.hdr-row{display:flex;align-items:center;gap:7px;font-size:.62rem;color:var(--t2);font-family:var(--mono)}\n\n/* ── dot ── */\n.dot{width:7px;height:7px;border-radius:50%;background:var(--t3);flex-shrink:0;transition:all .3s}\n.dot.on {background:var(--green);box-shadow:0 0 7px var(--green);animation:blink 2s infinite}\n.dot.warn{background:var(--yellow);box-shadow:0 0 7px var(--yellow);animation:blink .9s infinite}\n.dot.err {background:var(--red);box-shadow:0 0 7px var(--red);animation:blink .45s infinite}\n@keyframes blink{0%,100%{opacity:1}50%{opacity:.25}}\n\n/* ── buttons ── */\n.btn{\n  padding:7px 16px;font-family:var(--mono);font-size:.68rem;letter-spacing:.08em;\n  text-transform:uppercase;border:1px solid;background:transparent;cursor:pointer;\n  transition:all .18s;white-space:nowrap;border-radius:2px;font-weight:600\n}\n.btn-cy{border-color:var(--cyan);color:var(--cyan)}\n.btn-cy:hover,.btn-cy.lit{background:var(--cyan);color:#000;box-shadow:0 0 14px rgba(0,212,255,.5)}\n.btn-dc{border-color:var(--t3);color:var(--t2)}\n.btn-dc:hover{border-color:var(--red);color:var(--red)}\n.btn-yl{border-color:var(--yellow);color:var(--yellow)}\n.btn-yl.lit{background:var(--yellow);color:#000;box-shadow:0 0 14px rgba(240,181,0,.45)}\n.btn-rd{border-color:var(--red);color:var(--red)}\n.btn-rd:hover{background:var(--red);color:#fff}\n.btn-gn{border-color:var(--green);color:var(--green)}\n.btn-gn:hover{background:var(--green);color:#000}\n.btn-sm{padding:5px 9px;font-size:.6rem}\n.btn-xs{padding:3px 7px;font-size:.58rem}\n\n/* ── mode toggle ── */\n.mode-tog{display:flex;border:1px solid var(--border);border-radius:2px;overflow:hidden}\n.mbt{\n  padding:6px 12px;font-family:var(--mono);font-size:.62rem;letter-spacing:.07em;\n  text-transform:uppercase;border:none;background:transparent;cursor:pointer;\n  color:var(--t3);transition:all .2s\n}\n.mbt.ar{background:var(--cyandim);color:var(--cyan)}\n.mbt.as{background:var(--ylwdim);color:var(--yellow)}\n\n/* ═══════════════════════════════════════════\n   BANNERS\n═══════════════════════════════════════════ */\n.banner{display:none;padding:7px 20px;font-family:var(--mono);font-size:.68rem;\n  letter-spacing:.1em;text-align:center;text-transform:uppercase}\n.banner.show{display:block}\n.ban-err{background:rgba(255,40,40,.12);color:var(--red);border-bottom:1px solid rgba(255,40,40,.3);animation:blink 1s infinite}\n\n/* ── sim bar ── */\n.sim-bar{display:none;flex-wrap:wrap;gap:6px;padding:9px 18px;\n  background:rgba(240,181,0,.04);border-bottom:1px solid rgba(240,181,0,.18)}\n.sim-bar.show{display:flex}\n.sim-lbl{width:100%;font-size:.58rem;color:var(--yellow);font-family:var(--mono);letter-spacing:.1em;text-transform:uppercase}\n\n/* ═══════════════════════════════════════════\n   LAYOUT GRID\n═══════════════════════════════════════════ */\n.main{display:grid;grid-template-columns:1fr 320px;gap:14px;padding:14px 20px;flex:1}\n.col-l,.col-r{display:flex;flex-direction:column;gap:14px}\n\n/* ═══════════════════════════════════════════\n   PANEL\n═══════════════════════════════════════════ */\n.panel{\n  background:var(--panel);border:1px solid var(--border);border-radius:3px;\n  backdrop-filter:blur(8px);box-shadow:0 4px 22px rgba(0,0,0,.55);overflow:hidden\n}\n.ph{display:flex;align-items:center;justify-content:space-between;\n  padding:9px 14px;border-bottom:1px solid var(--border);background:rgba(0,0,0,.18)}\n.ph-title{font-family:var(--mono);font-size:.65rem;letter-spacing:.14em;color:var(--t2);\n  text-transform:uppercase;display:flex;align-items:center;gap:7px}\n.ph-title .ac{color:var(--cyan)}\n.pb{padding:12px 14px}\n\n/* ═══════════════════════════════════════════\n   SENSOR CARDS GRID\n═══════════════════════════════════════════ */\n.sg{display:grid;grid-template-columns:repeat(2,1fr);gap:12px}\n.sc{\n  background:var(--card);border:1px solid var(--border);border-radius:3px;\n  padding:13px;position:relative;overflow:hidden;transition:border-color .3s,box-shadow .3s\n}\n.sc::before{content:\'\';position:absolute;top:0;left:0;right:0;height:2px;background:var(--cyan);opacity:.4;transition:all .3s}\n.sc.aw::before{background:var(--yellow);opacity:1}\n.sc.ae::before{background:var(--red);opacity:1;animation:blink .5s infinite}\n.sc.ae{border-color:rgba(255,40,40,.45);box-shadow:0 0 18px rgba(255,40,40,.12)}\n.sc.aw{border-color:rgba(240,181,0,.38);box-shadow:0 0 14px rgba(240,181,0,.08)}\n\n/* card header row */\n.sc-hdr{display:flex;justify-content:space-between;align-items:center;margin-bottom:9px}\n.sc-name{font-family:var(--mono);font-size:.62rem;letter-spacing:.13em;color:var(--t2);text-transform:uppercase}\n.badge{\n  font-family:var(--mono);font-size:.58rem;letter-spacing:.07em;padding:2px 7px;\n  border-radius:2px;text-transform:uppercase;border:1px solid\n}\n.b-ok  {background:var(--grndim);color:var(--green);border-color:rgba(0,224,106,.28)}\n.b-wn  {background:var(--ylwdim);color:var(--yellow);border-color:rgba(240,181,0,.28);animation:blink .9s infinite}\n.b-er  {background:var(--reddim);color:var(--red);border-color:rgba(255,40,40,.38);animation:blink .6s infinite}\n.b-off {background:rgba(30,45,70,.3);color:var(--t3);border-color:var(--border)}\n\n/* big value */\n.sc-val{font-family:var(--mono);font-size:2rem;font-weight:700;line-height:1;margin-bottom:3px}\n.vc{color:var(--cyan);text-shadow:0 0 10px rgba(0,212,255,.35)}\n.vg{color:var(--green);text-shadow:0 0 10px rgba(0,224,106,.35)}\n.vy{color:var(--yellow);text-shadow:0 0 10px rgba(240,181,0,.35)}\n.vr{color:var(--red);text-shadow:0 0 10px rgba(255,40,40,.4)}\n.v0{color:var(--t3)}\n\n.sc-unit{font-size:.6rem;color:var(--t2);margin-bottom:9px}\n\n/* bar */\n.bar-bg{background:rgba(14,22,42,.8);border-radius:2px;height:5px;overflow:hidden;margin-bottom:7px}\n.bar{height:100%;border-radius:2px;transition:width .45s,background .3s;width:0}\n.bar-c{background:linear-gradient(90deg,rgba(0,140,190,.6),var(--cyan))}\n.bar-g{background:linear-gradient(90deg,rgba(0,140,70,.6),var(--green))}\n.bar-y{background:linear-gradient(90deg,rgba(170,110,0,.6),var(--yellow))}\n.bar-r{background:linear-gradient(90deg,rgba(160,0,0,.6),var(--red))}\n\n/* meta row */\n.sc-raw{font-size:.58rem;color:var(--t3);font-family:var(--mono);margin-bottom:4px}\n.sc-meta{display:flex;justify-content:space-between;font-size:.58rem;color:var(--t3);font-family:var(--mono)}\n.trend-up{color:var(--red)}.trend-dn{color:var(--green)}.trend-eq{color:var(--t3)}\n\n/* mini stat row (temp) */\n.msr{display:flex;gap:8px;margin-top:7px}\n.ms{flex:1;background:rgba(0,0,0,.3);border:1px solid var(--border);border-radius:2px;padding:4px 7px}\n.ms-l{font-size:.52rem;color:var(--t3);font-family:var(--mono);letter-spacing:.06em}\n.ms-v{font-size:.72rem;font-family:var(--mono);color:var(--t1)}\n\n/* ═══════════════════════════════════════════\n   CAPSULE\n═══════════════════════════════════════════ */\n.cap-wrap{\n  position:relative;height:230px;\n  background:radial-gradient(ellipse at 50% 55%,rgba(0,25,55,.75) 0%,rgba(2,4,12,.96) 68%);\n  border:1px solid var(--border);border-radius:3px;overflow:hidden;\n  display:flex;align-items:center;justify-content:center\n}\n#cap-canvas{display:block}\n.scan{position:absolute;top:0;left:0;right:0;height:1px;\n  background:linear-gradient(90deg,transparent,rgba(0,210,255,.35),transparent);\n  animation:scanY 3s linear infinite}\n@keyframes scanY{0%{top:0}100%{top:100%}}\n.cap-msg{\n  position:absolute;bottom:10px;left:50%;transform:translateX(-50%);\n  font-family:var(--mono);font-size:.6rem;letter-spacing:.13em;\n  color:var(--cyan);text-shadow:0 0 7px var(--cyan);text-transform:uppercase;white-space:nowrap\n}\n\n/* ═══════════════════════════════════════════\n   MISSION STATUS\n═══════════════════════════════════════════ */\n.ms-box{\n  border-radius:3px;padding:13px 14px;text-align:center;\n  transition:all .35s;border:1px solid var(--border);position:relative;overflow:hidden\n}\n.ms-ok  {background:rgba(0,55,28,.22);border-color:rgba(0,200,80,.28)!important}\n.ms-wn  {background:rgba(55,38,0,.26);border-color:rgba(240,181,0,.38)!important}\n.ms-er  {background:rgba(55,0,0,.28);border-color:rgba(255,40,40,.55)!important;animation:pulse-box .8s infinite}\n@keyframes pulse-box{0%,100%{box-shadow:0 0 18px rgba(255,40,40,.15)}50%{box-shadow:0 0 36px rgba(255,40,40,.42)}}\n.ms-lbl{font-family:var(--mono);font-size:.57rem;letter-spacing:.18em;color:var(--t2);text-transform:uppercase;margin-bottom:5px}\n.ms-main{font-family:var(--mono);font-size:1rem;font-weight:700;letter-spacing:.07em;line-height:1.2;margin-bottom:3px}\n.ms-sub{font-size:.62rem;color:var(--t2);font-family:var(--mono)}\n\n/* ═══════════════════════════════════════════\n   BUZZER CARD\n═══════════════════════════════════════════ */\n.bz-card{border-radius:3px;padding:13px;transition:all .3s;border:1px solid}\n.bz-off{background:rgba(0,28,18,.18);border-color:rgba(0,100,55,.28)!important}\n.bz-on {background:rgba(55,0,0,.28);border-color:rgba(255,40,40,.48)!important}\n.bz-ind{\n  width:46px;height:46px;border-radius:50%;\n  display:flex;align-items:center;justify-content:center;\n  font-size:1.3rem;margin:0 auto 9px;transition:all .3s\n}\n.bz-ind-on {border:2px solid var(--red);background:rgba(255,40,40,.18);animation:pulse-bz .5s infinite;box-shadow:0 0 18px rgba(255,40,40,.38)}\n.bz-ind-off{border:2px solid var(--green);background:rgba(0,224,106,.08);box-shadow:0 0 9px rgba(0,224,106,.18)}\n@keyframes pulse-bz{0%,100%{transform:scale(1)}50%{transform:scale(1.08)}}\n.bz-lbl{font-family:var(--mono);font-size:.85rem;font-weight:700;text-align:center;margin-bottom:5px}\n.bz-rsn{font-size:.6rem;color:var(--t2);font-family:var(--mono);text-align:center;margin-bottom:9px;min-height:15px}\n.bz-time{text-align:center;font-size:.58rem;color:var(--t3);font-family:var(--mono);margin-bottom:9px;min-height:14px}\n.bz-btns{display:flex;gap:5px;flex-wrap:wrap;justify-content:center}\n\n/* ═══════════════════════════════════════════\n   TABS\n═══════════════════════════════════════════ */\n.tab-bar{display:flex;border-bottom:1px solid var(--border);background:rgba(0,0,0,.18)}\n.tb{\n  padding:9px 14px;font-family:var(--mono);font-size:.6rem;letter-spacing:.09em;\n  text-transform:uppercase;border:none;background:transparent;color:var(--t3);\n  cursor:pointer;border-bottom:2px solid transparent;transition:all .18s;margin-bottom:-1px\n}\n.tb:hover{color:var(--t1)}\n.tb.act{color:var(--cyan);border-bottom-color:var(--cyan)}\n.tc{display:none;padding:12px 14px}\n.tc.act{display:block}\n\n/* ═══════════════════════════════════════════\n   CHARTS\n═══════════════════════════════════════════ */\ncanvas.ch{max-height:115px}\n\n/* ═══════════════════════════════════════════\n   HISTORY\n═══════════════════════════════════════════ */\n.hlist{max-height:240px;overflow-y:auto}\n.ev{\n  display:flex;gap:8px;padding:6px 12px;\n  border-bottom:1px solid rgba(20,38,68,.35);font-family:var(--mono);\n  font-size:.6rem;align-items:flex-start;transition:background .18s\n}\n.ev:hover{background:rgba(0,212,255,.03)}\n.ev-t{color:var(--t3);white-space:nowrap;flex-shrink:0}\n.ev-b{padding:1px 5px;border-radius:2px;font-size:.53rem;white-space:nowrap;flex-shrink:0;letter-spacing:.04em;border:1px solid}\n.eb-inf{background:rgba(0,140,200,.18);color:var(--cyan);border-color:rgba(0,140,200,.3)}\n.eb-wn {background:rgba(240,181,0,.13);color:var(--yellow);border-color:rgba(240,181,0,.28)}\n.eb-cr {background:rgba(255,40,40,.13);color:var(--red);border-color:rgba(255,40,40,.3)}\n.eb-ok {background:rgba(0,224,106,.1);color:var(--green);border-color:rgba(0,224,106,.22)}\n.ev-msg{color:var(--t1);flex:1}\n.ev-ack{color:var(--t3);font-size:.53rem;cursor:pointer;white-space:nowrap;flex-shrink:0}\n.ev-ack:hover{color:var(--cyan)}\n.ev.acked{opacity:.45}\n.ev.acked .ev-ack{color:var(--green)}\n\n/* ═══════════════════════════════════════════\n   SETTINGS\n═══════════════════════════════════════════ */\n.cfgrid{display:grid;grid-template-columns:repeat(2,1fr);gap:8px}\n.cg{background:rgba(0,0,0,.28);border:1px solid var(--border);border-radius:2px;padding:9px}\n.cg-l{font-size:.57rem;color:var(--t3);font-family:var(--mono);letter-spacing:.07em;text-transform:uppercase;margin-bottom:4px}\n.ci{\n  width:100%;background:rgba(0,8,22,.8);border:1px solid var(--border);\n  color:var(--t1);padding:4px 7px;font-family:var(--mono);font-size:.68rem;\n  border-radius:2px;outline:none\n}\n.ci:focus{border-color:var(--cyan);box-shadow:0 0 7px rgba(0,212,255,.18)}\n\n/* ═══════════════════════════════════════════\n   STATS BAR (bottom)\n═══════════════════════════════════════════ */\n.sbar{display:flex;border-top:1px solid var(--border);background:rgba(3,5,13,.97);overflow-x:auto}\n.si{flex:1;min-width:90px;padding:7px 12px;border-right:1px solid var(--border);flex-shrink:0}\n.si-l{font-size:.52rem;color:var(--t3);font-family:var(--mono);letter-spacing:.07em;text-transform:uppercase}\n.si-v{font-size:.72rem;font-family:var(--mono);color:var(--t1)}\n\n/* ═══════════════════════════════════════════\n   ALERT OVERLAY\n═══════════════════════════════════════════ */\n#alert-overlay{\n  display:none;position:fixed;inset:0;pointer-events:none;z-index:50;\n  animation:overlay-pulse 1s infinite\n}\n#alert-overlay.show{display:block}\n@keyframes overlay-pulse{\n  0%,100%{border:2px solid transparent}\n  50%{border:2px solid rgba(255,40,40,.42)}\n}\n\n/* ═══════════════════════════════════════════\n   PAINEL DE ANÁLISE TÉCNICA\n═══════════════════════════════════════════ */\n.anl-section{\n  padding:14px 20px;border-top:1px solid var(--border);\n  background:rgba(4,6,14,.96)\n}\n.anl-title{\n  font-family:var(--mono);font-size:.68rem;letter-spacing:.18em;color:var(--cyan);\n  text-transform:uppercase;margin-bottom:12px;display:flex;align-items:center;gap:9px\n}\n.anl-title::before{content:\'◈\';color:var(--cyan);opacity:.7}\n.anl-grid{\n  display:grid;\n  grid-template-columns:repeat(4,1fr);\n  gap:10px\n}\n@media(max-width:1100px){.anl-grid{grid-template-columns:repeat(2,1fr)}}\n@media(max-width:580px) {.anl-grid{grid-template-columns:1fr}}\n\n/* cada bloco de conceito */\n.anl-block{\n  background:var(--card);border:1px solid var(--border);border-radius:3px;\n  overflow:hidden;display:flex;flex-direction:column\n}\n.anl-block-hdr{\n  padding:8px 11px;border-bottom:1px solid var(--border);\n  display:flex;align-items:center;gap:8px;background:rgba(0,0,0,.25)\n}\n.anl-block-icon{font-size:.95rem}\n.anl-block-name{font-family:var(--mono);font-size:.6rem;letter-spacing:.13em;color:var(--t2);text-transform:uppercase}\n.anl-block-count{\n  margin-left:auto;font-family:var(--mono);font-size:.72rem;font-weight:700;\n  color:var(--cyan);background:var(--cyandim);\n  padding:1px 7px;border-radius:10px;border:1px solid rgba(0,212,255,.2)\n}\n.anl-block-body{padding:10px 11px;flex:1;display:flex;flex-direction:column;gap:6px}\n\n/* linha de item dentro do bloco */\n.anl-item{\n  display:flex;align-items:flex-start;gap:7px;\n  font-family:var(--mono);font-size:.6rem;line-height:1.4;\n  padding:5px 8px;border-radius:2px;border-left:2px solid transparent;\n  background:rgba(0,0,0,.18);transition:background .2s,border-color .2s\n}\n.anl-item.active{\n  background:rgba(0,212,255,.06);border-left-color:var(--cyan)\n}\n.anl-item.active-warn{\n  background:rgba(240,181,0,.07);border-left-color:var(--yellow)\n}\n.anl-item.active-err{\n  background:rgba(255,40,40,.08);border-left-color:var(--red);\n  animation:blink-left .7s infinite\n}\n@keyframes blink-left{0%,100%{border-left-color:var(--red)}50%{border-left-color:transparent}}\n\n.anl-item-key{\n  color:var(--cyan);flex-shrink:0;width:110px;font-size:.58rem;\n  white-space:nowrap;overflow:hidden;text-overflow:ellipsis\n}\n.anl-item-desc{color:var(--t2);font-size:.58rem;flex:1}\n.anl-item-val{\n  margin-left:auto;color:var(--t1);font-size:.62rem;font-weight:700;\n  white-space:nowrap;flex-shrink:0\n}\n/* last-fired flash */\n.anl-item.flash{animation:flash-it .4s ease}\n@keyframes flash-it{0%{background:rgba(0,212,255,.18)}100%{background:rgba(0,212,255,.06)}}\n\n/* vetor mini-viz */\n.vec-row{display:flex;gap:3px;flex-wrap:wrap;margin-top:2px}\n.vec-cell{\n  width:18px;height:18px;border-radius:2px;border:1px solid var(--border);\n  display:flex;align-items:center;justify-content:center;\n  font-family:var(--mono);font-size:.44rem;color:var(--t3);\n  transition:background .3s,color .3s,border-color .3s;position:relative;cursor:default\n}\n.vec-cell.filled{background:rgba(0,212,255,.15);color:var(--cyan);border-color:rgba(0,212,255,.35)}\n.vec-cell.newest{background:rgba(0,212,255,.3);color:var(--cyan);border-color:var(--cyan);\n  box-shadow:0 0 5px rgba(0,212,255,.4)}\n.vec-cell.alert-c{background:rgba(255,40,40,.18);color:var(--red);border-color:rgba(255,40,40,.4)}\n\n/* fn call log */\n.fn-log{\n  max-height:130px;overflow-y:auto;display:flex;flex-direction:column;gap:3px\n}\n.fn-entry{\n  display:flex;align-items:center;gap:6px;padding:3px 6px;\n  border-radius:2px;background:rgba(0,0,0,.2);font-family:var(--mono);font-size:.57rem;\n  border-left:2px solid transparent;transition:all .15s\n}\n.fn-entry.new{border-left-color:var(--cyan);background:rgba(0,212,255,.07)}\n.fn-name{color:var(--cyan);flex-shrink:0}\n.fn-args{color:var(--t3);font-size:.54rem;flex:1;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}\n.fn-time{color:var(--t3);font-size:.52rem;margin-left:auto;flex-shrink:0}\n.fn-ret{color:var(--green);font-size:.54rem;flex-shrink:0}\n\n/* ═══════════════════════════════════════════\n   RESPONSIVE\n═══════════════════════════════════════════ */\n@media(max-width:1050px){\n  .main{grid-template-columns:1fr}\n  .col-r{order:-1}\n}\n@media(max-width:640px){\n  .sg{grid-template-columns:1fr}\n  .cfgrid{grid-template-columns:1fr}\n  header{flex-direction:column;align-items:flex-start}\n  .hdr-right{align-items:flex-start}\n  .brand-name{font-size:1.05rem}\n}\n\n/* ═══════════════════════════════════════════\n   POPUP DE ALERTA CRÍTICO\n═══════════════════════════════════════════ */\n.popup-overlay{\n  display:none;position:fixed;inset:0;z-index:500;\n  background:rgba(0,0,0,.75);backdrop-filter:blur(4px);\n  align-items:center;justify-content:center\n}\n.popup-overlay.show{display:flex}\n.popup-box{\n  background:rgba(10,14,26,.98);border:1px solid var(--red);\n  border-radius:4px;width:min(540px,95vw);max-height:85vh;\n  overflow-y:auto;box-shadow:0 0 40px rgba(255,40,40,.35);\n  animation:pop-in .2s ease\n}\n@keyframes pop-in{from{transform:scale(.93);opacity:0}to{transform:scale(1);opacity:1}}\n.popup-head{\n  display:flex;align-items:center;justify-content:space-between;\n  padding:14px 18px;border-bottom:1px solid rgba(255,40,40,.3);\n  background:rgba(255,40,40,.08)\n}\n.popup-title{\n  font-family:var(--mono);font-size:.8rem;font-weight:700;\n  letter-spacing:.12em;color:var(--red);text-transform:uppercase\n}\n.popup-close{\n  background:none;border:1px solid rgba(255,40,40,.4);color:var(--red);\n  cursor:pointer;padding:3px 10px;font-family:var(--mono);font-size:.7rem;\n  border-radius:2px;transition:all .15s\n}\n.popup-close:hover{background:var(--red);color:#fff}\n.popup-body{padding:16px 18px;display:flex;flex-direction:column;gap:12px}\n.popup-sensor-row{\n  display:flex;align-items:center;gap:10px;padding:8px 12px;\n  border-radius:3px;border:1px solid rgba(255,40,40,.25);\n  background:rgba(255,40,40,.06)\n}\n.popup-sensor-row.warn{\n  border-color:rgba(240,181,0,.3);background:rgba(240,181,0,.06)\n}\n.popup-sensor-icon{font-size:1.1rem;flex-shrink:0}\n.popup-sensor-name{\n  font-family:var(--mono);font-size:.65rem;color:var(--t2);\n  text-transform:uppercase;letter-spacing:.08em;flex-shrink:0;width:130px\n}\n.popup-sensor-val{\n  font-family:var(--mono);font-size:.8rem;font-weight:700;\n  color:var(--red);flex-shrink:0\n}\n.popup-sensor-val.warn-v{color:var(--yellow)}\n.popup-protocol{\n  background:rgba(0,0,0,.3);border:1px solid var(--border);\n  border-radius:3px;overflow:hidden\n}\n.popup-protocol-hdr{\n  padding:7px 12px;background:rgba(0,212,255,.07);\n  border-bottom:1px solid var(--border);\n  font-family:var(--mono);font-size:.62rem;color:var(--cyan);\n  letter-spacing:.1em;text-transform:uppercase;display:flex;align-items:center;gap:7px\n}\n.popup-protocol-steps{padding:10px 12px;display:flex;flex-direction:column;gap:5px}\n.popup-step{\n  font-family:var(--mono);font-size:.62rem;color:var(--t1);\n  display:flex;gap:8px;align-items:flex-start;line-height:1.5\n}\n.popup-step-num{\n  color:var(--cyan);flex-shrink:0;font-weight:700;\n  width:16px;text-align:right\n}\n.popup-confirm{\n  margin-top:4px;padding:10px;text-align:center;\n  border-top:1px solid var(--border)\n}\n.popup-confirm button{\n  padding:9px 28px;font-family:var(--mono);font-size:.72rem;\n  letter-spacing:.1em;text-transform:uppercase;border:1px solid var(--cyan);\n  color:var(--cyan);background:rgba(0,212,255,.08);cursor:pointer;\n  border-radius:2px;transition:all .18s\n}\n.popup-confirm button:hover{background:var(--cyan);color:#000}\n\n/* ═══════════════════════════════════════════\n   CHATBOT ARIA\n═══════════════════════════════════════════ */\n/* botão flutuante */\n#aria-fab{\n  position:fixed;bottom:24px;right:24px;z-index:400;\n  width:52px;height:52px;border-radius:50%;\n  background:rgba(0,212,255,.12);border:1.5px solid var(--cyan);\n  color:var(--cyan);font-size:1.3rem;cursor:pointer;\n  box-shadow:0 0 20px rgba(0,212,255,.25);\n  transition:all .2s;display:flex;align-items:center;justify-content:center\n}\n#aria-fab:hover{background:rgba(0,212,255,.22);box-shadow:0 0 30px rgba(0,212,255,.4)}\n#aria-fab .aria-notif{\n  position:absolute;top:-3px;right:-3px;\n  width:14px;height:14px;border-radius:50%;\n  background:var(--red);font-size:.5rem;font-family:var(--mono);\n  color:#fff;display:none;align-items:center;justify-content:center;\n  font-weight:700\n}\n#aria-fab .aria-notif.show{display:flex}\n\n/* janela do chat */\n#aria-window{\n  position:fixed;bottom:86px;right:24px;z-index:400;\n  width:min(380px,calc(100vw - 32px));height:520px;\n  background:rgba(9,13,24,.98);border:1px solid var(--border);\n  border-radius:4px;box-shadow:0 8px 40px rgba(0,0,0,.7);\n  display:none;flex-direction:column;overflow:hidden;\n  animation:chat-in .2s ease\n}\n#aria-window.open{display:flex}\n@keyframes chat-in{from{transform:translateY(16px);opacity:0}to{transform:translateY(0);opacity:1}}\n\n.aria-header{\n  display:flex;align-items:center;gap:10px;padding:11px 14px;\n  border-bottom:1px solid var(--border);background:rgba(0,0,0,.25);flex-shrink:0\n}\n.aria-avatar{\n  width:30px;height:30px;border-radius:50%;\n  background:rgba(0,212,255,.12);border:1.5px solid var(--cyan);\n  display:flex;align-items:center;justify-content:center;font-size:.9rem;flex-shrink:0\n}\n.aria-name{font-family:var(--mono);font-size:.72rem;font-weight:700;color:var(--cyan)}\n.aria-status-dot{width:6px;height:6px;border-radius:50%;background:var(--green);\n  box-shadow:0 0 5px var(--green);margin-left:auto;flex-shrink:0}\n.aria-close{\n  background:none;border:none;color:var(--t3);cursor:pointer;\n  font-size:.9rem;padding:2px 5px;transition:color .15s;line-height:1\n}\n.aria-close:hover{color:var(--t1)}\n\n/* mensagens */\n.aria-msgs{flex:1;overflow-y:auto;padding:12px;display:flex;flex-direction:column;gap:8px}\n.aria-msgs::-webkit-scrollbar{width:3px}\n.aria-msgs::-webkit-scrollbar-thumb{background:var(--border)}\n\n.msg{display:flex;gap:8px;max-width:92%;animation:msg-in .15s ease}\n@keyframes msg-in{from{opacity:0;transform:translateY(6px)}to{opacity:1;transform:translateY(0)}}\n.msg.bot{align-self:flex-start}\n.msg.usr{align-self:flex-end;flex-direction:row-reverse}\n\n.msg-bubble{\n  padding:8px 11px;border-radius:3px;\n  font-family:var(--mono);font-size:.63rem;line-height:1.55;\n  max-width:100%;word-break:break-word\n}\n.msg.bot .msg-bubble{\n  background:rgba(0,212,255,.07);border:1px solid rgba(0,212,255,.18);color:var(--t1)\n}\n.msg.usr .msg-bubble{\n  background:rgba(0,212,255,.13);border:1px solid rgba(0,212,255,.3);color:var(--cyan)\n}\n.msg.alert-msg .msg-bubble{\n  background:rgba(255,40,40,.09);border:1px solid rgba(255,40,40,.3);color:var(--t1)\n}\n.msg-icon{font-size:.85rem;flex-shrink:0;margin-top:2px}\n\n/* chips de comando rápido */\n.aria-chips{\n  display:flex;flex-wrap:wrap;gap:5px;padding:8px 12px;\n  border-top:1px solid var(--border);background:rgba(0,0,0,.15);flex-shrink:0\n}\n.chip{\n  padding:3px 9px;font-family:var(--mono);font-size:.57rem;letter-spacing:.06em;\n  text-transform:uppercase;border:1px solid var(--border);color:var(--t2);\n  background:transparent;cursor:pointer;border-radius:12px;transition:all .15s\n}\n.chip:hover{border-color:var(--cyan);color:var(--cyan);background:var(--cyandim)}\n\n/* input */\n.aria-input-row{\n  display:flex;gap:6px;padding:10px 12px;\n  border-top:1px solid var(--border);flex-shrink:0;background:rgba(0,0,0,.2)\n}\n#aria-input{\n  flex:1;background:rgba(0,10,30,.7);border:1px solid var(--border);\n  color:var(--t1);padding:7px 10px;font-family:var(--mono);font-size:.65rem;\n  border-radius:2px;outline:none;transition:border-color .15s\n}\n#aria-input:focus{border-color:var(--cyan)}\n#aria-send{\n  padding:7px 12px;font-family:var(--mono);font-size:.62rem;\n  border:1px solid var(--cyan);color:var(--cyan);background:rgba(0,212,255,.07);\n  cursor:pointer;border-radius:2px;transition:all .15s;letter-spacing:.05em\n}\n#aria-send:hover{background:var(--cyan);color:#000}\n\n/* typing indicator */\n.typing{display:flex;gap:4px;align-items:center;padding:2px 0}\n.typing span{\n  width:5px;height:5px;border-radius:50%;background:var(--cyan);opacity:.5;\n  animation:ty .9s infinite\n}\n.typing span:nth-child(2){animation-delay:.2s}\n.typing span:nth-child(3){animation-delay:.4s}\n@keyframes ty{0%,60%,100%{transform:translateY(0)}30%{transform:translateY(-5px);opacity:1}}\n</style>\n</head>\n<body>\n<canvas id="bg-canvas"></canvas>\n<div id="alert-overlay"></div>\n\n<div class="wrap">\n\n<!-- ════════════ HEADER ════════════ -->\n<header>\n  <div>\n    <div class="brand-name">⬡ MISSÃO EVORTEX</div>\n    <div class="brand-sub">Sistema Inteligente de Monitoramento da Cápsula Espacial</div>\n  </div>\n\n  <div class="hdr-mid">\n    <div class="mode-tog">\n      <button class="mbt ar" id="btn-mr" onclick="setMode(\'real\')">Real</button>\n      <button class="mbt"    id="btn-ms" onclick="setMode(\'sim\')">Simulação</button>\n    </div>\n    <button class="btn btn-cy" id="btn-conn"  onclick="conectarArduino()">⬡ Conectar Arduino</button>\n    <button class="btn btn-dc" id="btn-disc"  onclick="desconectarArduino()" style="display:none">✕ Desconectar</button>\n  </div>\n\n  <div class="hdr-right">\n    <div class="hdr-row"><div class="dot" id="dot"></div><span id="txt-status">Sistema offline</span></div>\n    <div class="hdr-row" id="txt-time" style="color:var(--cyan)">—</div>\n    <div class="hdr-row">T+ <span id="txt-up">00:00:00</span></div>\n    <div class="hdr-row">Último pacote: <span id="txt-pkt" style="color:var(--t1)">—</span></div>\n  </div>\n</header>\n\n<!-- banner falha comm -->\n<div class="banner ban-err" id="ban-comm">⚠ FALHA DE COMUNICAÇÃO COM O ARDUINO — TELEMETRIA CONGELADA</div>\n\n<!-- sim controls -->\n<div class="sim-bar" id="sim-bar">\n  <span class="sim-lbl">◈ Controles de simulação</span>\n  <button class="btn btn-yl btn-sm" onclick="simAuto()">▶ Auto</button>\n  <button class="btn btn-sm" style="border-color:var(--t3);color:var(--t2)" onclick="simStop()">⏸ Pausar</button>\n  <button class="btn btn-rd btn-sm" onclick="simFogo()">🔥 Fogo</button>\n  <button class="btn btn-sm" style="border-color:var(--orange);color:var(--orange)" onclick="simGas()">☁ Gás</button>\n  <button class="btn btn-sm" style="border-color:var(--yellow);color:var(--yellow)" onclick="simLuz()">💡 Baixa Luz</button>\n  <button class="btn btn-sm" style="border-color:var(--orange);color:var(--orange)" onclick="simTemp()">🌡 Temp Alta</button>\n  <button class="btn btn-yl btn-sm" onclick="simComm()">📡 Falha Comm</button>\n  <button class="btn btn-gn btn-sm" onclick="simRestore()">↺ Restaurar</button>\n</div>\n\n<!-- ════════════ MAIN ════════════ -->\n<div class="main">\n\n  <!-- ── LEFT COLUMN ── -->\n  <div class="col-l">\n\n    <!-- SENSOR CARDS -->\n    <div class="sg">\n\n      <!-- GÁS A0 -->\n      <div class="sc" id="card-gas">\n        <div class="sc-hdr">\n          <span class="sc-name">☁ Sensor de Gás — A0</span>\n          <span class="badge b-off" id="gas-badge">Sem sinal</span>\n        </div>\n        <div class="sc-val vc" id="gas-val">—</div>\n        <div class="sc-unit">valor bruto (0 – 1023)</div>\n        <div class="bar-bg"><div class="bar bar-c" id="gas-bar"></div></div>\n        <div class="sc-raw">\n          Bruto: <span id="gas-raw">—</span> &nbsp;|&nbsp;\n          Limite alerta: <span id="gas-lim-d">—</span> &nbsp;|&nbsp;\n          Tendência: <span id="gas-trend" class="trend-eq">—</span>\n        </div>\n        <div class="sc-meta">\n          <span id="gas-state-txt">—</span>\n          <span id="gas-upd">—</span>\n        </div>\n      </div>\n\n      <!-- FOGO A1  (lógica INVERTIDA: alto = sem fogo, baixo = fogo) -->\n      <div class="sc" id="card-fire">\n        <div class="sc-hdr">\n          <span class="sc-name">🔥 Sensor de Fogo — A1</span>\n          <span class="badge b-off" id="fire-badge">Sem sinal</span>\n        </div>\n        <div class="sc-val vc" id="fire-val">—</div>\n        <div class="sc-unit">valor bruto · lógica invertida (alto = sem fogo)</div>\n        <div class="bar-bg"><div class="bar bar-c" id="fire-bar"></div></div>\n        <div class="sc-raw">\n          Bruto: <span id="fire-raw">—</span> &nbsp;|&nbsp;\n          Limite alerta abaixo de: <span id="fire-lim-d">—</span>\n        </div>\n        <div class="sc-meta">\n          <span id="fire-state-txt">—</span>\n          <span id="fire-upd">—</span>\n        </div>\n      </div>\n\n      <!-- LUZ A3  — limiteLuz = 50 (hardcoded no Arduino, editável aqui) -->\n      <div class="sc" id="card-luz">\n        <div class="sc-hdr">\n          <span class="sc-name">💡 Luminosidade — A3</span>\n          <span class="badge b-off" id="luz-badge">Sem sinal</span>\n        </div>\n        <div class="sc-val vc" id="luz-val">—</div>\n        <div class="sc-unit">valor bruto (0 – 1023)</div>\n        <div class="bar-bg"><div class="bar bar-c" id="luz-bar"></div></div>\n        <div class="sc-raw">\n          Bruto: <span id="luz-raw">—</span> &nbsp;|&nbsp;\n          Limite mínimo: <span id="luz-lim-d">—</span>\n        </div>\n        <div class="sc-meta">\n          <!-- Espelha exatamente o que o Arduino imprime -->\n          <span id="luz-state-txt">—</span>\n          <span id="luz-upd">—</span>\n        </div>\n      </div>\n\n      <!-- TEMPERATURA A4  — cálculo NTC igual ao Arduino -->\n      <div class="sc" id="card-temp">\n        <div class="sc-hdr">\n          <span class="sc-name">🌡 Temperatura — A4</span>\n          <span class="badge b-off" id="temp-badge">Sem sinal</span>\n        </div>\n        <div class="sc-val vc" id="temp-val">—</div>\n        <div class="sc-unit">\n          °C &nbsp;|&nbsp; Bruto A4: <span id="temp-raw-d">—</span> &nbsp;|&nbsp;\n          NTC R: <span id="temp-ntc-r">—</span> Ω\n        </div>\n        <div class="bar-bg"><div class="bar bar-c" id="temp-bar"></div></div>\n        <div class="msr">\n          <div class="ms"><div class="ms-l">Mínima</div><div class="ms-v" id="temp-min">—</div></div>\n          <div class="ms"><div class="ms-l">Máxima</div><div class="ms-v" id="temp-max">—</div></div>\n          <div class="ms"><div class="ms-l">Média</div> <div class="ms-v" id="temp-avg">—</div></div>\n        </div>\n        <div class="sc-meta" style="margin-top:6px">\n          <span id="temp-state-txt">—</span>\n          <span id="temp-upd">—</span>\n        </div>\n      </div>\n\n    </div><!-- /sg -->\n\n    <!-- CHARTS -->\n    <div class="panel">\n      <div class="tab-bar">\n        <button class="tb act" onclick="chTab(\'gas\',this)">Gás</button>\n        <button class="tb"     onclick="chTab(\'fire\',this)">Fogo</button>\n        <button class="tb"     onclick="chTab(\'luz\',this)">Luz</button>\n        <button class="tb"     onclick="chTab(\'temp\',this)">Temperatura</button>\n        <div style="flex:1"></div>\n        <button class="btn btn-xs" style="margin:4px 8px;border-color:var(--t3);color:var(--t3)" onclick="clearCharts()">Limpar</button>\n      </div>\n      <div class="tc act" id="tc-gas"  style="padding:10px"><canvas class="ch" id="ch-gas"  height="115"></canvas></div>\n      <div class="tc"     id="tc-fire" style="padding:10px"><canvas class="ch" id="ch-fire" height="115"></canvas></div>\n      <div class="tc"     id="tc-luz"  style="padding:10px"><canvas class="ch" id="ch-luz"  height="115"></canvas></div>\n      <div class="tc"     id="tc-temp" style="padding:10px"><canvas class="ch" id="ch-temp" height="115"></canvas></div>\n    </div>\n\n    <!-- HISTORY -->\n    <div class="panel">\n      <div class="ph">\n        <div class="ph-title"><span class="ac">◈</span> Histórico da Missão</div>\n        <div style="display:flex;gap:5px;flex-wrap:wrap">\n          <select id="hf-s" class="ci" style="width:auto;padding:3px 5px;font-size:.58rem" onchange="renderHist()">\n            <option value="">Todos</option>\n            <option value="gas">Gás</option><option value="fogo">Fogo</option>\n            <option value="luz">Luz</option><option value="temp">Temp</option>\n            <option value="sis">Sistema</option>\n          </select>\n          <select id="hf-l" class="ci" style="width:auto;padding:3px 5px;font-size:.58rem" onchange="renderHist()">\n            <option value="">Todos níveis</option>\n            <option value="inf">Info</option><option value="wn">Atenção</option>\n            <option value="cr">Crítico</option><option value="ok">OK</option>\n          </select>\n          <button class="btn btn-xs btn-cy"  onclick="exportCSV()">⬇ CSV</button>\n          <button class="btn btn-xs" style="border-color:var(--t3);color:var(--t2)" onclick="copyReport()">⎘ Copiar</button>\n          <button class="btn btn-xs btn-rd"  onclick="clearHist()">✕ Limpar</button>\n        </div>\n      </div>\n      <div class="hlist" id="hlist">\n        <div style="padding:14px;text-align:center;color:var(--t3);font-family:var(--mono);font-size:.62rem">Aguardando eventos...</div>\n      </div>\n    </div>\n\n  </div><!-- /col-l -->\n\n  <!-- ── RIGHT COLUMN ── -->\n  <div class="col-r">\n\n    <!-- CAPSULE -->\n    <div class="cap-wrap">\n      <canvas id="cap-canvas" width="310" height="215"></canvas>\n      <div class="scan"></div>\n      <div class="cap-msg" id="cap-msg">OPERAÇÃO NORMAL</div>\n    </div>\n\n    <!-- MISSION STATUS -->\n    <div class="ms-box ms-ok" id="ms-box">\n      <div class="ms-lbl">Status geral da missão</div>\n      <div class="ms-main" id="ms-main" style="color:var(--green)">MISSÃO OPERANDO NORMALMENTE</div>\n      <div class="ms-sub"  id="ms-sub">Todos os parâmetros dentro dos limites</div>\n    </div>\n\n    <!-- BUZZER — D6, ativado apenas por falha de luz (igual ao Arduino) -->\n    <div class="bz-card bz-off" id="bz-card">\n      <div class="bz-ind bz-ind-off" id="bz-ind">🔕</div>\n      <div class="bz-lbl" id="bz-lbl" style="color:var(--green)">BUZZER DESLIGADO</div>\n      <div class="bz-rsn" id="bz-rsn">Iluminação OK — sirene em repouso</div>\n      <div class="bz-time" id="bz-time"></div>\n      <div class="bz-btns">\n        <button class="btn btn-xs btn-yl" id="btn-snd"   onclick="toggleSound()">🔊 Som virtual</button>\n        <button class="btn btn-xs btn-cy"                onclick="ackAlerts()">✓ Confirmar</button>\n      </div>\n    </div>\n\n    <!-- SETTINGS + INFO -->\n    <div class="panel">\n      <div class="tab-bar">\n        <button class="tb act" onclick="swTab(\'tcfg\',this)">⚙ Config</button>\n        <button class="tb"     onclick="swTab(\'tinf\',this)">ℹ Info</button>\n      </div>\n\n      <div class="tc act" id="tcfg">\n        <div class="cfgrid">\n          <div class="cg">\n            <div class="cg-l">Limite Gás (alerta acima)</div>\n            <input type="number" class="ci" id="cfg-gas" value="200" onchange="applyConfig()">\n          </div>\n          <div class="cg">\n            <div class="cg-l">Limite Fogo (alerta abaixo)</div>\n            <input type="number" class="ci" id="cfg-fire" value="300" onchange="applyConfig()">\n          </div>\n          <div class="cg">\n            <div class="cg-l">Limite Luz mínima — Arduino usa 50</div>\n            <input type="number" class="ci" id="cfg-luz" value="50" onchange="applyConfig()">\n          </div>\n          <div class="cg">\n            <div class="cg-l">Temp. Atenção (°C)</div>\n            <input type="number" class="ci" id="cfg-tw" value="30" onchange="applyConfig()">\n          </div>\n          <div class="cg">\n            <div class="cg-l">Temp. Crítica (°C)</div>\n            <input type="number" class="ci" id="cfg-tc" value="40" onchange="applyConfig()">\n          </div>\n          <div class="cg">\n            <div class="cg-l">Timeout comm (s)</div>\n            <input type="number" class="ci" id="cfg-to" value="3" onchange="applyConfig()">\n          </div>\n          <div class="cg">\n            <div class="cg-l">Volume alarme (0–1)</div>\n            <input type="number" class="ci" id="cfg-vol" value="0.4" min="0" max="1" step="0.1" onchange="applyConfig()">\n          </div>\n        </div>\n        <div style="margin-top:9px;text-align:right">\n          <button class="btn btn-sm" style="border-color:var(--t3);color:var(--t2)" onclick="resetConfig()">↺ Restaurar padrões</button>\n        </div>\n      </div>\n\n      <div class="tc" id="tinf">\n        <div style="display:flex;flex-direction:column;gap:7px;font-family:var(--mono);font-size:.6rem;color:var(--t2)">\n          <div style="color:var(--cyan)">Central de Controle · Sistema EVORTEX</div>\n          <div>Telemetria em tempo real via Web Serial API</div>\n          <hr style="border-color:var(--border)">\n          <div style="color:var(--t1)">Pinos do Arduino:</div>\n          <div>A0 → Sensor de Gás (MQ-x)</div>\n          <div>A1 → Sensor de Fogo (lógica invertida)</div>\n          <div>A3 → Sensor de Luz (LDR/módulo)</div>\n          <div>A4 → Temperatura NTC (β=3950, Rfix=6kΩ, R0=100kΩ)</div>\n          <div>D6 → Buzzer — sirene ativada por falha de luz</div>\n          <hr style="border-color:var(--border)">\n          <div style="color:var(--t1)">Formato serial aceito:</div>\n          <div style="background:rgba(0,0,0,.4);padding:5px 7px;border-radius:2px;color:var(--cyan);font-size:.56rem;word-break:break-all">\n            GAS:46,FOGO:987,LUZ:59,TEMP_RAW:57,TEMP:24.3,BUZZER:0\n          </div>\n          <div>Também aceita o verbose do Monitor Serial e JSON.</div>\n          <div>Baud: 9600 · Chrome / Edge com flag experimental ativada</div>\n        </div>\n      </div>\n    </div>\n\n  </div><!-- /col-r -->\n</div><!-- /main -->\n\n<!-- STATS BAR -->\n<div class="sbar">\n  <div class="si"><div class="si-l">Conexão</div>   <div class="si-v" id="st-conn" style="color:var(--t3)">Offline</div></div>\n  <div class="si"><div class="si-l">Modo</div>       <div class="si-v" id="st-mode" style="color:var(--cyan)">Real</div></div>\n  <div class="si"><div class="si-l">Alertas</div>    <div class="si-v" id="st-alrt">0</div></div>\n  <div class="si"><div class="si-l">Sensores</div>   <div class="si-v" id="st-sens">0 / 4</div></div>\n  <div class="si"><div class="si-l">Taxa</div>       <div class="si-v" id="st-freq">—</div></div>\n  <div class="si"><div class="si-l">Buzzer D6</div>  <div class="si-v" id="st-buz" style="color:var(--green)">Desligado</div></div>\n  <div class="si"><div class="si-l">Missão T+</div>  <div class="si-v" id="st-up">00:00:00</div></div>\n</div>\n\n<!-- ════════════════════════════════════════\n     PAINEL DE ANÁLISE DE ESTRUTURAS\n     Demonstra condicionais, repetição,\n     vetores e funções em tempo real.\n════════════════════════════════════════ -->\n<div class="anl-section">\n  <div class="anl-title">Análise de Estruturas de Programação — Execução em Tempo Real</div>\n  <div class="anl-grid">\n\n    <!-- BLOCO 1: CONDICIONAIS -->\n    <div class="anl-block">\n      <div class="anl-block-hdr">\n        <span class="anl-block-icon">⎇</span>\n        <span class="anl-block-name">Estruturas Condicionais</span>\n        <span class="anl-block-count" id="anl-if-count">0</span>\n      </div>\n      <div class="anl-block-body">\n        <!-- cada if/else do sistema, acende quando disparado -->\n        <div class="anl-item" id="anl-if-gas" title="if (valorGas >= limiteGas)">\n          <span class="anl-item-key">if gas ≥ limite</span>\n          <span class="anl-item-desc">Detecta gás acima do limiar</span>\n          <span class="anl-item-val" id="anl-if-gas-v">—</span>\n        </div>\n        <div class="anl-item" id="anl-if-fogo" title="if (valorFogo < limiteFogo)">\n          <span class="anl-item-key">if fogo &lt; limite</span>\n          <span class="anl-item-desc">Sensor invertido: baixo = chama</span>\n          <span class="anl-item-val" id="anl-if-fogo-v">—</span>\n        </div>\n        <div class="anl-item" id="anl-if-luz" title="falhaIluminacao = valorLuz < limiteLuz">\n          <span class="anl-item-key">if luz &lt; limiteLuz</span>\n          <span class="anl-item-desc">falhaIluminacao — ativa buzzer D6</span>\n          <span class="anl-item-val" id="anl-if-luz-v">—</span>\n        </div>\n        <div class="anl-item" id="anl-if-temp-c" title="if (tempC >= tempCrit)">\n          <span class="anl-item-key">if temp ≥ crítica</span>\n          <span class="anl-item-desc">Temperatura acima do limite crítico</span>\n          <span class="anl-item-val" id="anl-if-tc-v">—</span>\n        </div>\n        <div class="anl-item" id="anl-if-temp-w" title="else if (tempC >= tempWarn)">\n          <span class="anl-item-key">else if temp ≥ atenção</span>\n          <span class="anl-item-desc">Zona de alerta de temperatura</span>\n          <span class="anl-item-val" id="anl-if-tw-v">—</span>\n        </div>\n        <div class="anl-item" id="anl-if-ntc" title="if (leitura <= 1 || leitura >= 1022)">\n          <span class="anl-item-key">if raw ≤1 ou ≥1022</span>\n          <span class="anl-item-desc">Rejeita NTC saturado → retorna NaN</span>\n          <span class="anl-item-val" id="anl-if-ntc-v">—</span>\n        </div>\n        <div class="anl-item" id="anl-if-comm" title="if (dt > commTimeout)">\n          <span class="anl-item-key">if Δt &gt; timeout</span>\n          <span class="anl-item-desc">Falha de comunicação serial</span>\n          <span class="anl-item-val" id="anl-if-comm-v">—</span>\n        </div>\n      </div>\n    </div>\n\n    <!-- BLOCO 2: REPETIÇÃO -->\n    <div class="anl-block">\n      <div class="anl-block-hdr">\n        <span class="anl-block-icon">↺</span>\n        <span class="anl-block-name">Estruturas de Repetição</span>\n        <span class="anl-block-count" id="anl-loop-count">0</span>\n      </div>\n      <div class="anl-block-body">\n        <div class="anl-item active" id="anl-lp-main">\n          <span class="anl-item-key">loop() Arduino</span>\n          <span class="anl-item-desc">Ciclo principal de leitura (500 ms)</span>\n          <span class="anl-item-val" id="anl-lp-main-v">0×</span>\n        </div>\n        <div class="anl-item active" id="anl-lp-serial">\n          <span class="anl-item-key">while(serial)</span>\n          <span class="anl-item-desc">Lê stream serial continuamente</span>\n          <span class="anl-item-val" id="anl-lp-serial-v">0 bytes</span>\n        </div>\n        <div class="anl-item" id="anl-lp-parse">\n          <span class="anl-item-key">forEach(campos)</span>\n          <span class="anl-item-desc">Percorre campos do pacote CSV</span>\n          <span class="anl-item-val" id="anl-lp-parse-v">0 campos</span>\n        </div>\n        <div class="anl-item" id="anl-lp-hist">\n          <span class="anl-item-key">forEach(eventos)</span>\n          <span class="anl-item-desc">Renderiza lista do histórico</span>\n          <span class="anl-item-val" id="anl-lp-hist-v">0 eventos</span>\n        </div>\n        <div class="anl-item" id="anl-lp-chart">\n          <span class="anl-item-key">forEach(gráficos)</span>\n          <span class="anl-item-desc">Atualiza 4 gráficos Chart.js</span>\n          <span class="anl-item-val" id="anl-lp-chart-v">0×</span>\n        </div>\n        <div class="anl-item active" id="anl-lp-check">\n          <span class="anl-item-key">setInterval(checkComm)</span>\n          <span class="anl-item-desc">Vigilância de comunicação (1 s)</span>\n          <span class="anl-item-val" id="anl-lp-check-v">0×</span>\n        </div>\n        <div class="anl-item active" id="anl-lp-tick">\n          <span class="anl-item-key">setInterval(tick)</span>\n          <span class="anl-item-desc">Relógio e uptime (1 s)</span>\n          <span class="anl-item-val" id="anl-lp-tick-v">0×</span>\n        </div>\n      </div>\n    </div>\n\n    <!-- BLOCO 3: VETORES -->\n    <div class="anl-block">\n      <div class="anl-block-hdr">\n        <span class="anl-block-icon">▦</span>\n        <span class="anl-block-name">Vetores (Arrays)</span>\n        <span class="anl-block-count" id="anl-vec-count">0 itens</span>\n      </div>\n      <div class="anl-block-body">\n\n        <div style="font-family:var(--mono);font-size:.57rem;color:var(--t2);margin-bottom:3px">\n          hist[] — histórico de eventos (últimos 10 visíveis)\n        </div>\n        <div class="vec-row" id="vec-hist"></div>\n\n        <div style="font-family:var(--mono);font-size:.57rem;color:var(--t2);margin:6px 0 3px">\n          chartData.gas[] — série temporal de gás (últimos 20)\n        </div>\n        <div class="vec-row" id="vec-gas"></div>\n\n        <div style="font-family:var(--mono);font-size:.57rem;color:var(--t2);margin:6px 0 3px">\n          chartData.temp[] — série temporal de temperatura\n        </div>\n        <div class="vec-row" id="vec-temp"></div>\n\n        <div style="margin-top:8px;display:flex;gap:8px;font-family:var(--mono);font-size:.57rem;color:var(--t3)">\n          <span>hist: <b id="vec-hist-sz" style="color:var(--t1)">0</b></span>\n          <span>gas: <b id="vec-gas-sz" style="color:var(--t1)">0</b></span>\n          <span>fire: <b id="vec-fire-sz" style="color:var(--t1)">0</b></span>\n          <span>luz: <b id="vec-luz-sz" style="color:var(--t1)">0</b></span>\n          <span>temp: <b id="vec-temp-sz" style="color:var(--t1)">0</b></span>\n        </div>\n        <div style="font-family:var(--mono);font-size:.57rem;color:var(--t3);margin-top:4px">\n          Capacidade máx: <b style="color:var(--t2)">60 pts / gráfico · 600 eventos hist</b>\n        </div>\n      </div>\n    </div>\n\n    <!-- BLOCO 4: FUNÇÕES -->\n    <div class="anl-block">\n      <div class="anl-block-hdr">\n        <span class="anl-block-icon">ƒ</span>\n        <span class="anl-block-name">Funções — Chamadas Recentes</span>\n        <span class="anl-block-count" id="anl-fn-count">0 chamadas</span>\n      </div>\n      <div class="anl-block-body" style="padding:8px 10px">\n        <div class="fn-log" id="fn-log"></div>\n        <div style="margin-top:8px;border-top:1px solid var(--border);padding-top:7px;\n          display:grid;grid-template-columns:1fr 1fr;gap:4px">\n          <div style="font-family:var(--mono);font-size:.57rem;color:var(--t3)">calcTempC(raw) →</div>\n          <div style="font-family:var(--mono);font-size:.6rem;color:var(--green)" id="anl-fn-ntc-ret">—</div>\n          <div style="font-family:var(--mono);font-size:.57rem;color:var(--t3)">updateLuz(v) →</div>\n          <div style="font-family:var(--mono);font-size:.6rem" id="anl-fn-luz-ret">—</div>\n          <div style="font-family:var(--mono);font-size:.57rem;color:var(--t3)">updateFire(v) →</div>\n          <div style="font-family:var(--mono);font-size:.6rem" id="anl-fn-fire-ret">—</div>\n          <div style="font-family:var(--mono);font-size:.57rem;color:var(--t3)">setBuzzer(on) →</div>\n          <div style="font-family:var(--mono);font-size:.6rem" id="anl-fn-buz-ret">—</div>\n        </div>\n      </div>\n    </div>\n\n  </div>\n</div>\n\n<!-- ════════════════════════════════════════\n     POPUP DE ALERTA CRÍTICO\n════════════════════════════════════════ -->\n<div class="popup-overlay" id="popup-overlay">\n  <div class="popup-box" id="popup-box">\n    <div class="popup-head">\n      <div class="popup-title" id="popup-title">🚨 ALERTA CRÍTICO</div>\n      <button class="popup-close" onclick="fecharPopup()">✕ Fechar</button>\n    </div>\n    <div class="popup-body" id="popup-body">\n      <!-- preenchido dinamicamente pelo JS -->\n    </div>\n    <div class="popup-confirm">\n      <button onclick="fecharPopup()">✓ Alerta reconhecido — Fechar</button>\n    </div>\n  </div>\n</div>\n\n<!-- ════════════════════════════════════════\n     CHATBOT ARIA — botão flutuante\n════════════════════════════════════════ -->\n<button id="aria-fab" onclick="ariaToggle()" title="Falar com ARIA">\n  💬\n  <div class="aria-notif" id="aria-notif">!</div>\n</button>\n\n<!-- janela do chat -->\n<div id="aria-window">\n  <div class="aria-header">\n    <div class="aria-avatar">🤖</div>\n    <div>\n      <div class="aria-name">ARIA</div>\n      <div style="font-size:.55rem;color:var(--t2);font-family:var(--mono)">IA de Controle — EVORTEX</div>\n    </div>\n    <div class="aria-status-dot" id="aria-status-dot"></div>\n    <button class="aria-close" onclick="ariaToggle()">✕</button>\n  </div>\n\n  <div class="aria-msgs" id="aria-msgs"></div>\n\n  <!-- chips de acesso rápido -->\n  <div class="aria-chips" id="aria-chips">\n    <button class="chip" onclick="ariaChip(\'missão\')">🛰 missão</button>\n    <button class="chip" onclick="ariaChip(\'status\')">📊 status</button>\n    <button class="chip" onclick="ariaChip(\'alertas\')">🚨 alertas</button>\n    <button class="chip" onclick="ariaChip(\'sensores\')">🔬 sensores</button>\n    <button class="chip" onclick="ariaChip(\'protocolos\')">📋 protocolos</button>\n    <button class="chip" onclick="ariaChip(\'hardware\')">🔧 hardware</button>\n    <button class="chip" onclick="ariaChip(\'interface\')">🖥️ interface</button>\n    <button class="chip" onclick="ariaChip(\'arduino\')">📡 arduino</button>\n    <button class="chip" onclick="ariaChip(\'ajuda\')">❓ ajuda</button>\n  </div>\n\n  <div class="aria-input-row">\n    <input id="aria-input" placeholder="Pergunte algo à ARIA..." onkeydown="if(event.key===\'Enter\')ariaSend()">\n    <button id="aria-send" onclick="ariaSend()">Enviar</button>\n  </div>\n</div>\n\n</div><!-- /wrap -->\n\n<script>\n/* ════════════════════════════════════════════════════════════\n   MISSÃO EVORTEX — JavaScript\n   Espelha fielmente o código Arduino fornecido:\n     A0 = Gás\n     A1 = Fogo (lógica invertida)\n     A3 = Luz (limiteLuz = 50, buzzer ativado se luz < 50)\n     A4 = Temperatura NTC (β=3950, Rfix=6kΩ, R0=100kΩ, T0=25°C)\n     D6 = Buzzer (sirene varredura 600–1400 Hz, ativada SOMENTE por falha de luz)\n════════════════════════════════════════════════════════════ */\n\n// ─── Parâmetros NTC idênticos ao Arduino ───────────────────\nconst NTC = { Rfix:6000, R0:100000, T0:25, beta:3950 };\n\n// ─── Config editável ───────────────────────────────────────\nconst C = {\n  gasLim:  200,   // alerta acima deste valor\n  fireLim: 300,   // alerta abaixo deste valor (lógica invertida)\n  luzLim:  50,    // falhaIluminacao = luz < 50  (igual ao Arduino)\n  tempW:   30,    // atenção\n  tempC:   40,    // crítico\n  commTO:  3000,  // ms sem dado = falha de comm\n  vol:     0.4,\n};\n\n// ─── Estado ────────────────────────────────────────────────\nconst S = {\n  mode:\'real\', connected:false,\n  port:null, reader:null,\n  t0: Date.now(), lastPkt:null, commFailed:false,\n  alertCnt:0, ackd:false,\n  buzzer:false, buzzerAt:null, buzzerReason:\'\',\n  sndEnabled:false, sndMuted:false,\n  simRunning:false, simTick:null, simAutoTick:null,\n  // sensor values\n  gas:null, gasPrev:null,\n  fire:null,\n  luz:null,\n  tempC:null, tempRaw:null,\n  tempMin:Infinity, tempMax:-Infinity, tempSum:0, tempN:0,\n  _prevPktTs: null,\n};\n\n// ─── History ───────────────────────────────────────────────\nlet hist = [];\n\n// ─── Charts ────────────────────────────────────────────────\nconst CD = { gas:{l:[],d:[]}, fire:{l:[],d:[]}, luz:{l:[],d:[]}, temp:{l:[],d:[]} };\nconst MAXPT = 60;\nlet charts = {};\n\n// ─── Audio ─────────────────────────────────────────────────\nlet actx = null, alarmOsc = null;\n\n/* ════════════════════════════════════════════════════════════\n   BACKGROUND CANVAS\n════════════════════════════════════════════════════════════ */\n(function initBg() {\n  const cv = document.getElementById(\'bg-canvas\');\n  const cx = cv.getContext(\'2d\');\n  let stars=[], W, H;\n  function resize() {\n    W=cv.width=innerWidth; H=cv.height=innerHeight;\n    stars=Array.from({length:110},()=>({\n      x:Math.random()*W, y:Math.random()*H,\n      r:Math.random()*1.1+.2, a:Math.random()*.5+.1, tw:Math.random()*Math.PI*2\n    }));\n  }\n  function draw() {\n    cx.clearRect(0,0,W,H);\n    cx.strokeStyle=\'rgba(0,140,240,.04)\'; cx.lineWidth=.5;\n    for(let x=0;x<W;x+=58){cx.beginPath();cx.moveTo(x,0);cx.lineTo(x,H);cx.stroke()}\n    for(let y=0;y<H;y+=58){cx.beginPath();cx.moveTo(0,y);cx.lineTo(W,y);cx.stroke()}\n    stars.forEach(s=>{\n      s.tw+=.018;\n      const a=s.a*(.6+.4*Math.sin(s.tw));\n      cx.beginPath(); cx.arc(s.x,s.y,s.r,0,Math.PI*2);\n      cx.fillStyle=`rgba(200,225,255,${a})`; cx.fill();\n    });\n    requestAnimationFrame(draw);\n  }\n  resize(); addEventListener(\'resize\',resize); draw();\n})();\n\n/* ════════════════════════════════════════════════════════════\n   CAPSULE CANVAS\n════════════════════════════════════════════════════════════ */\nlet capAlert = false;\n(function initCap() {\n  const cv = document.getElementById(\'cap-canvas\');\n  const cx = cv.getContext(\'2d\');\n  let t=0;\n  function frame() {\n    t+=.022;\n    const float=Math.sin(t)*5.5;\n    cx.clearRect(0,0,cv.width,cv.height);\n    const mx=cv.width/2, my=cv.height/2+float;\n    const ap=capAlert?(.5+.5*Math.sin(t*6)):0;\n\n    // Alert glow\n    if(capAlert){\n      const g=cx.createRadialGradient(mx,my,28,mx,my,105);\n      g.addColorStop(0,`rgba(255,0,0,${.04+ap*.14})`);\n      g.addColorStop(1,\'transparent\');\n      cx.fillStyle=g; cx.fillRect(0,0,cv.width,cv.height);\n    }\n\n    cx.save(); cx.translate(mx,my);\n\n    // Orbit rings\n    cx.strokeStyle=capAlert?`rgba(255,55,55,${.18+ap*.28})`:\'rgba(0,175,215,.1)\';\n    cx.lineWidth=1;\n    cx.beginPath(); cx.ellipse(0,0,96,26,0,0,Math.PI*2); cx.stroke();\n    cx.strokeStyle=capAlert?`rgba(255,75,75,${.08+ap*.18})`:\'rgba(0,210,255,.06)\';\n    cx.beginPath(); cx.ellipse(0,0,112,34,.18,0,Math.PI*2); cx.stroke();\n\n    // Body\n    const bg=cx.createLinearGradient(-46,-65,46,65);\n    bg.addColorStop(0,\'rgba(28,42,68,.92)\');\n    bg.addColorStop(.45,\'rgba(18,30,54,.96)\');\n    bg.addColorStop(1,\'rgba(7,11,26,.99)\');\n    cx.beginPath();\n    cx.moveTo(-30,28); cx.lineTo(-36,-18);\n    cx.bezierCurveTo(-36,-66,36,-66,36,-18);\n    cx.lineTo(30,28); cx.bezierCurveTo(30,48,-30,48,-30,28);\n    cx.closePath();\n    cx.fillStyle=bg; cx.fill();\n    cx.strokeStyle=capAlert?`rgba(255,45,45,${.6+ap*.4})`:\'rgba(0,195,235,.38)\';\n    cx.lineWidth=1.2; cx.stroke();\n\n    // Dome window\n    const dg=cx.createRadialGradient(-7,-37,2,-7,-37,20);\n    dg.addColorStop(0,capAlert?`rgba(255,90,90,.3)`:`rgba(0,215,255,.22)`);\n    dg.addColorStop(1,\'transparent\');\n    cx.beginPath(); cx.ellipse(-7,-36,17,21,0,0,Math.PI*2);\n    cx.fillStyle=dg; cx.fill();\n    cx.strokeStyle=capAlert?\'rgba(255,70,70,.45)\':\'rgba(0,210,255,.3)\';\n    cx.lineWidth=.8; cx.stroke();\n\n    // Panel lines\n    cx.strokeStyle=\'rgba(35,65,110,.5)\'; cx.lineWidth=.6;\n    [-13,0,13].forEach(x=>{cx.beginPath();cx.moveTo(x,-8);cx.lineTo(x*1.12,26);cx.stroke()});\n    cx.beginPath();cx.moveTo(-32,2);cx.lineTo(32,2);cx.stroke();\n\n    // Tech lights\n    [{x:18,y:-52,c:capAlert?\'#ff2828\':\'#00e06a\'},{x:-18,y:-52,c:\'#00d4ff\'},\n     {x:26,y:14,c:capAlert?\'#ff7318\':\'#00d4ff\'},{x:-26,y:14,c:\'#00e06a\'}].forEach(l=>{\n      cx.shadowBlur=6; cx.shadowColor=l.c;\n      cx.beginPath(); cx.arc(l.x,l.y,2.2,0,Math.PI*2);\n      cx.fillStyle=l.c; cx.fill(); cx.shadowBlur=0;\n    });\n\n    // Thrusters\n    [[-18,48],[0,53],[18,48]].forEach(([ox,oy])=>{\n      cx.fillStyle=\'rgba(22,32,56,.92)\'; cx.strokeStyle=\'rgba(35,70,120,.5)\'; cx.lineWidth=.7;\n      cx.beginPath();cx.moveTo(ox-5,oy-5);cx.lineTo(ox+5,oy-5);cx.lineTo(ox+4,oy+9);cx.lineTo(ox-4,oy+9);cx.closePath();\n      cx.fill(); cx.stroke();\n      const eg=cx.createLinearGradient(ox,oy,ox,oy+18);\n      eg.addColorStop(0,\'rgba(0,175,255,.22)\'); eg.addColorStop(1,\'transparent\');\n      cx.fillStyle=eg;\n      cx.beginPath();cx.moveTo(ox-4,oy+9);cx.lineTo(ox+4,oy+9);\n      cx.lineTo(ox+3,oy+19+Math.sin(t*3)*3.5);cx.lineTo(ox-3,oy+19+Math.sin(t*3+1)*3.5);\n      cx.closePath(); cx.fill();\n    });\n\n    // Solar panels\n    [[38,-8,1],[-56,-8,-1]].forEach(([px,py,sign])=>{\n      cx.fillStyle=\'rgba(12,22,46,.9)\'; cx.strokeStyle=\'rgba(0,115,195,.45)\'; cx.lineWidth=.8;\n      cx.fillRect(px,py-18,sign*18,36); cx.strokeRect(px,py-18,sign*18,36);\n      cx.strokeStyle=\'rgba(0,145,215,.25)\';\n      for(let r=0;r<4;r++){cx.beginPath();cx.moveTo(px,py-18+r*9);cx.lineTo(px+sign*18,py-18+r*9);cx.stroke()}\n      cx.beginPath();cx.moveTo(px+sign*9,py-18);cx.lineTo(px+sign*9,py+18);cx.stroke();\n      const rg=cx.createLinearGradient(px,0,px+sign*18,0);\n      rg.addColorStop(0,\'transparent\');rg.addColorStop(.5,\'rgba(0,195,255,.07)\');rg.addColorStop(1,\'transparent\');\n      cx.fillStyle=rg; cx.fillRect(px,py-18,sign*18,36);\n    });\n\n    cx.restore();\n    requestAnimationFrame(frame);\n  }\n  frame();\n})();\n\n/* ════════════════════════════════════════════════════════════\n   CHARTS\n════════════════════════════════════════════════════════════ */\nfunction initCharts() {\n  const opts=(label,color)=>({\n    type:\'line\',\n    data:{labels:[],datasets:[{\n      label, data:[],\n      borderColor:color,\n      backgroundColor:color.replace(\'rgb(\',\'rgba(\').replace(\')\',\',0.07)\'),\n      borderWidth:1.5, pointRadius:0, tension:.4, fill:true\n    }]},\n    options:{\n      responsive:true, maintainAspectRatio:false, animation:{duration:250},\n      plugins:{legend:{display:false}},\n      scales:{\n        x:{ticks:{color:\'#2e4260\',font:{size:8},maxTicksLimit:8,maxRotation:0},\n           grid:{color:\'rgba(30,65,120,.13)\'},border:{color:\'rgba(30,65,120,.28)\'}},\n        y:{ticks:{color:\'#2e4260\',font:{size:8}},\n           grid:{color:\'rgba(30,65,120,.13)\'},border:{color:\'rgba(30,65,120,.28)\'}},\n      }\n    }\n  });\n  charts.gas  = new Chart(g(\'ch-gas\'),  opts(\'Gás\',\'rgb(0,212,255)\'));\n  charts.fire = new Chart(g(\'ch-fire\'), opts(\'Fogo\',\'rgb(255,115,24)\'));\n  charts.luz  = new Chart(g(\'ch-luz\'),  opts(\'Luz\',\'rgb(240,181,0)\'));\n  charts.temp = new Chart(g(\'ch-temp\'), opts(\'°C\',\'rgb(0,224,106)\'));\n}\n\nfunction addPt(key, val) {\n  ANL.chartUpdates++;\n  g(\'anl-lp-chart-v\').textContent = ANL.chartUpdates+\'×\';\n  const ts = new Date().toLocaleTimeString(\'pt-BR\',{hour:\'2-digit\',minute:\'2-digit\',second:\'2-digit\'});\n  CD[key].l.push(ts); CD[key].d.push(val);\n  if(CD[key].l.length>MAXPT){CD[key].l.shift();CD[key].d.shift()}\n  charts[key].data.labels=CD[key].l;\n  charts[key].data.datasets[0].data=CD[key].d;\n  charts[key].update(\'none\');\n}\n\nfunction clearCharts() {\n  [\'gas\',\'fire\',\'luz\',\'temp\'].forEach(k=>{\n    CD[k].l=[];CD[k].d=[];\n    charts[k].data.labels=[];charts[k].data.datasets[0].data=[];charts[k].update();\n  });\n  addEvt(\'sis\',\'inf\',\'Gráficos limpos\',null);\n}\n\n/* ════════════════════════════════════════════════════════════\n   SENSOR UPDATES\n   Cada função espelha a lógica exata do Arduino.\n════════════════════════════════════════════════════════════ */\n\n// ── Gás (A0) ───────────────────────────────────────────────\nfunction updateGas(raw) {\n  fnLog(\'updateGas\',\'raw=\'+raw);\n  if(raw===null||isNaN(raw)){setCard(\'gas\',null);return}\n  const v=Math.round(raw);\n  S.gasPrev=S.gas; S.gas=v;\n  const pct=Math.min(100,(v/1023)*100);\n\n  // estado\n  let st=\'ok\', bc=\'b-ok\', cc=\'\', txt=\'Normal\';\n  if(v>=C.gasLim){st=\'er\';bc=\'b-er\';cc=\'ae\';txt=\'GÁS DETECTADO\'}\n  else if(v>=C.gasLim*.7){st=\'wn\';bc=\'b-wn\';cc=\'aw\';txt=\'Atenção\'}\n\n  // cor do valor\n  const vc = st===\'er\'?\'vr\':st===\'wn\'?\'vy\':\'vg\';\n  const bc2 = st===\'er\'?\'bar-r\':st===\'wn\'?\'bar-y\':\'bar-g\';\n\n  g(\'gas-val\').textContent=v;\n  g(\'gas-val\').className=\'sc-val \'+vc;\n  g(\'gas-badge\').textContent=txt;\n  g(\'gas-badge\').className=\'badge \'+bc;\n  g(\'gas-bar\').className=\'bar \'+bc2;\n  g(\'gas-bar\').style.width=pct+\'%\';\n  g(\'gas-raw\').textContent=v;\n  g(\'gas-lim-d\').textContent=C.gasLim;\n  g(\'gas-state-txt\').textContent=txt;\n  g(\'gas-upd\').textContent=ts();\n  g(\'card-gas\').className=\'sc \'+cc;\n\n  // tendência\n  let tr=\'—\',tc2=\'trend-eq\';\n  if(S.gasPrev!==null){\n    if(v>S.gasPrev+1){tr=\'▲\';tc2=\'trend-up\'}\n    else if(v<S.gasPrev-1){tr=\'▼\';tc2=\'trend-dn\'}\n    else{tr=\'→\'}\n  }\n  const tel=g(\'gas-trend\'); tel.textContent=tr; tel.className=tc2;\n\n  addPt(\'gas\',v);\n  if(st===\'er\') trigAlert(\'gas\',\'cr\',\'Gás detectado: \'+v,v);\n  else if(st===\'wn\') trigAlert(\'gas\',\'wn\',\'Gás se aproximando do limite: \'+v,v);\n}\n\n// ── Fogo (A1) — lógica invertida ───────────────────────────\n// Arduino: valor alto = ausência de fogo; valor baixo = presença de chama\nfunction updateFire(raw) {\n  fnLog(\'updateFire\',\'raw=\'+raw);\n  if(raw===null||isNaN(raw)){setCard(\'fire\',null);return}\n  const v=Math.round(raw);\n  S.fire=v;\n  // A barra mostra "intensidade de fogo" = quanto mais baixo o valor, mais cheia\n  const pct=Math.min(100,Math.max(0,((1023-v)/1023)*100));\n\n  let st=\'ok\',bc=\'b-ok\',cc=\'\',txt=\'Sem chama\';\n  if(v<C.fireLim){st=\'er\';bc=\'b-er\';cc=\'ae\';txt=\'FOGO DETECTADO\'}\n  else if(v<C.fireLim*1.35){st=\'wn\';bc=\'b-wn\';cc=\'aw\';txt=\'Atenção\'}\n\n  const vc=st===\'er\'?\'vr\':st===\'wn\'?\'vy\':\'vg\';\n  const bc2=st===\'er\'?\'bar-r\':st===\'wn\'?\'bar-y\':\'bar-g\';\n\n  g(\'fire-val\').textContent=v;\n  g(\'fire-val\').className=\'sc-val \'+vc;\n  g(\'fire-badge\').textContent=txt;\n  g(\'fire-badge\').className=\'badge \'+bc;\n  g(\'fire-bar\').className=\'bar \'+bc2;\n  g(\'fire-bar\').style.width=pct+\'%\';\n  g(\'fire-raw\').textContent=v;\n  g(\'fire-lim-d\').textContent=C.fireLim;\n  g(\'fire-state-txt\').textContent=txt;\n  g(\'fire-upd\').textContent=ts();\n  g(\'card-fire\').className=\'sc \'+cc;\n\n  addPt(\'fire\',v);\n  if(st===\'er\') trigAlert(\'fogo\',\'cr\',\'FOGO DETECTADO — sensor: \'+v,v);\n}\n\n// ── Luz (A3) — lógica idêntica ao Arduino ──────────────────\n// falhaIluminacao = valorLuz < limiteLuz (50)\n// Buzzer ATIVADO se falhaIluminacao\nfunction updateLuz(raw) {\n  fnLog(\'updateLuz\',\'raw=\'+raw);\n  if(raw===null||isNaN(raw)){setCard(\'luz\',null);return}\n  const v=Math.round(raw);\n  S.luz=v;\n  const pct=Math.min(100,(v/1023)*100);\n\n  // Exatamente o que o Arduino decide\n  const falha = v < C.luzLim;\n\n  let st,bc,cc,txt,detail;\n  if(falha){\n    st=\'er\';bc=\'b-er\';cc=\'ae\';\n    txt=\'FALHA NA ILUMINAÇÃO\';\n    detail=\'LUZ ABAIXO DO LIMITE — BUZZER ATIVADO\';\n  } else {\n    st=\'ok\';bc=\'b-ok\';cc=\'\';\n    txt=\'Iluminação funcionando\';\n    detail=\'NÍVEL DE LUZ ADEQUADO\';\n  }\n\n  const vc=falha?\'vr\':\'vg\';\n  const bc2=falha?\'bar-r\':\'bar-g\';\n\n  g(\'luz-val\').textContent=v;\n  g(\'luz-val\').className=\'sc-val \'+vc;\n  g(\'luz-badge\').textContent=txt;\n  g(\'luz-badge\').className=\'badge \'+bc;\n  g(\'luz-bar\').className=\'bar \'+bc2;\n  g(\'luz-bar\').style.width=pct+\'%\';\n  g(\'luz-raw\').textContent=v;\n  g(\'luz-lim-d\').textContent=C.luzLim;\n  // espelha exatamente as strings do Serial.println do Arduino\n  g(\'luz-state-txt\').textContent = falha\n    ? \'ILUMINACAO: FALHA - LUZ ABAIXO DE \'+C.luzLim\n    : \'ILUMINACAO: FUNCIONANDO\';\n  g(\'luz-upd\').textContent=ts();\n  g(\'card-luz\').className=\'sc \'+cc;\n\n  addPt(\'luz\',v);\n\n  // Buzzer: ativado SOMENTE por falha de luz (igual ao Arduino)\n  if(falha) {\n    trigAlert(\'luz\',\'cr\',\'Falha de iluminação — luz: \'+v,v);\n    setBuzzer(true, \'Luz < \'+C.luzLim+\' — ILUMINACAO: FALHA\');\n  } else {\n    // Buzzer desligado SE não houver outra causa (o Arduino usa só luz)\n    setBuzzer(false, \'\');\n  }\n}\n\n// ── Temperatura (A4) — cálculo NTC igual ao Arduino ────────\n// calcularTemperaturaCelsius(leitura) com mesmos parâmetros\nfunction calcTempC(leitura) {\n  if(leitura<=1||leitura>=1022) {\n    fnLog(\'calcTempC\', \'raw=\'+leitura, \'NaN — fora do range\');\n    return NaN;\n  }\n  const Rntc = NTC.Rfix * ((1023.0/leitura)-1.0);\n  const Tk   = 1.0 / ((1.0/(NTC.T0+273.15)) + (Math.log(Rntc/NTC.R0)/NTC.beta));\n  const result = Tk - 273.15;\n  fnLog(\'calcTempC\', \'raw=\'+leitura+\', R=\'+Math.round(Rntc)+\'Ω\', result.toFixed(2)+\'°C\');\n  return result;\n}\n\nfunction updateTemp(tempC, rawVal) {\n  // Se o Arduino já enviou o valor calculado, usa direto.\n  // Se enviou só raw, calcula igual ao Arduino.\n  let tC = tempC;\n  let raw = rawVal;\n\n  if((tC===null||isNaN(tC)) && raw!==null && !isNaN(raw)){\n    tC = calcTempC(raw);\n  }\n\n  if(raw!==null && !isNaN(raw)){\n    S.tempRaw = Math.round(raw);\n    const Rntc = NTC.Rfix * ((1023.0/raw)-1.0);\n    g(\'temp-raw-d\').textContent = S.tempRaw;\n    g(\'temp-ntc-r\').textContent = isFinite(Rntc) ? Math.round(Rntc).toLocaleString(\'pt-BR\') : \'—\';\n  }\n\n  if(tC===null||isNaN(tC)){\n    g(\'temp-val\').textContent=\'ERRO\';\n    g(\'temp-val\').className=\'sc-val v0\';\n    g(\'temp-badge\').textContent=\'Erro no sensor\';\n    g(\'temp-badge\').className=\'badge b-off\';\n    g(\'temp-bar\').style.width=\'0\';\n    g(\'temp-state-txt\').textContent=\'ERRO NO SENSOR\';\n    g(\'temp-upd\').textContent=ts();\n    g(\'card-temp\').className=\'sc\';\n    trigAlert(\'temp\',\'wn\',\'Erro no sensor de temperatura — valor fora do range\',null);\n    return;\n  }\n\n  S.tempC=tC;\n  if(tC<S.tempMin)S.tempMin=tC;\n  if(tC>S.tempMax)S.tempMax=tC;\n  S.tempSum+=tC; S.tempN++;\n\n  const pct=Math.min(100,Math.max(0,((tC-(-10))/70)*100));\n\n  let st=\'ok\',bc=\'b-ok\',cc=\'\',txt=\'Temperatura normal\';\n  if(tC>=C.tempC){st=\'er\';bc=\'b-er\';cc=\'ae\';txt=\'TEMPERATURA CRÍTICA\'}\n  else if(tC>=C.tempW){st=\'wn\';bc=\'b-wn\';cc=\'aw\';txt=\'Temperatura elevada\'}\n\n  const vc=st===\'er\'?\'vr\':st===\'wn\'?\'vy\':\'vg\';\n  const bc2=st===\'er\'?\'bar-r\':st===\'wn\'?\'bar-y\':\'bar-g\';\n\n  g(\'temp-val\').textContent=tC.toFixed(1)+\' °C\';\n  g(\'temp-val\').className=\'sc-val \'+vc;\n  g(\'temp-badge\').textContent=txt;\n  g(\'temp-badge\').className=\'badge \'+bc;\n  g(\'temp-bar\').className=\'bar \'+bc2;\n  g(\'temp-bar\').style.width=pct+\'%\';\n  g(\'temp-state-txt\').textContent=txt;\n  g(\'temp-upd\').textContent=ts();\n  g(\'card-temp\').className=\'sc \'+cc;\n  g(\'temp-min\').textContent=S.tempMin===Infinity?\'—\':S.tempMin.toFixed(1)+\'°C\';\n  g(\'temp-max\').textContent=S.tempMax===-Infinity?\'—\':S.tempMax.toFixed(1)+\'°C\';\n  g(\'temp-avg\').textContent=S.tempN?((S.tempSum/S.tempN).toFixed(1)+\'°C\'):\'—\';\n\n  addPt(\'temp\',parseFloat(tC.toFixed(2)));\n  if(st===\'er\') trigAlert(\'temp\',\'cr\',\'Temperatura crítica: \'+tC.toFixed(1)+\'°C\',tC.toFixed(1));\n  else if(st===\'wn\') trigAlert(\'temp\',\'wn\',\'Temperatura elevada: \'+tC.toFixed(1)+\'°C\',tC.toFixed(1));\n}\n\nfunction setCard(sensor, val) {\n  // offline / sem sinal\n  const ids={gas:\'gas\',fire:\'fire\',luz:\'luz\',temp:\'temp\'};\n  const n=ids[sensor];\n  if(!n)return;\n  g(n+\'-val\').textContent=\'—\'; g(n+\'-val\').className=\'sc-val v0\';\n  g(n+\'-badge\').textContent=\'Sem sinal\'; g(n+\'-badge\').className=\'badge b-off\';\n  g(n+\'-bar\').style.width=\'0\';\n  g(\'card-\'+sensor).className=\'sc\';\n}\n\n/* ════════════════════════════════════════════════════════════\n   BUZZER — D6\n   No Arduino: ativado SOMENTE quando luz < limiteLuz.\n   A interface respeita esse estado mas também reage a outros\n   sensores críticos para feedback visual.\n════════════════════════════════════════════════════════════ */\nfunction setBuzzer(on, reason) {\n  fnLog(\'setBuzzer\', on?\'true\':\'false\', on?\'ATIVO\':\'desligado\');\n  const wasOn=S.buzzer;\n  S.buzzer=on;\n  if(on && !wasOn){ S.buzzerAt=Date.now(); S.buzzerReason=reason; }\n  if(!on){ S.buzzerAt=null; S.buzzerReason=\'\'; }\n  if(on && reason) S.buzzerReason=reason;\n\n  const ind=g(\'bz-ind\');\n  const lbl=g(\'bz-lbl\');\n  const card=g(\'bz-card\');\n\n  if(on){\n    ind.className=\'bz-ind bz-ind-on\'; ind.textContent=\'🔔\';\n    lbl.textContent=\'BUZZER ATIVO — SIRENE\'; lbl.style.color=\'var(--red)\';\n    card.className=\'bz-card bz-on\';\n    g(\'bz-rsn\').textContent=S.buzzerReason||\'Alerta detectado\';\n    g(\'st-buz\').textContent=\'ATIVO\'; g(\'st-buz\').style.color=\'var(--red)\';\n    if(!wasOn && S.sndEnabled && !S.sndMuted) playAlarm();\n  } else {\n    ind.className=\'bz-ind bz-ind-off\'; ind.textContent=\'🔕\';\n    lbl.textContent=\'BUZZER DESLIGADO\'; lbl.style.color=\'var(--green)\';\n    card.className=\'bz-card bz-off\';\n    g(\'bz-rsn\').textContent=\'Iluminação OK — sirene em repouso\';\n    g(\'st-buz\').textContent=\'Desligado\'; g(\'st-buz\').style.color=\'var(--green)\';\n    stopAlarm();\n  }\n}\n\n/* ════════════════════════════════════════════════════════════\n   MISSION STATUS GERAL\n════════════════════════════════════════════════════════════ */\nfunction updateStatus() {\n  fnLog(\'updateStatus\',\'—\');\n  const gasErr  = S.gas!==null  && S.gas>=C.gasLim;\n  const fireErr = S.fire!==null && S.fire<C.fireLim;\n  // Buzzer físico: apenas luz\n  const luzErr  = S.luz!==null  && S.luz<C.luzLim;\n  const tempEr  = S.tempC!==null && S.tempC>=C.tempC;\n  const tempWn  = S.tempC!==null && S.tempC>=C.tempW;\n  const commErr = S.commFailed;\n\n  const isCrit = gasErr||fireErr||luzErr||tempEr||commErr;\n  const isWarn = !isCrit&&(tempWn||(S.gas!==null&&S.gas>=C.gasLim*.7));\n\n  const box=g(\'ms-box\'), main=g(\'ms-main\'), sub=g(\'ms-sub\'), cmsg=g(\'cap-msg\'), ov=g(\'alert-overlay\');\n\n  if(isCrit){\n    box.className=\'ms-box ms-er\';\n    main.style.color=\'var(--red)\';\n    main.textContent=\'ALERTA CRÍTICO NA CÁPSULA\';\n    sub.textContent = commErr?\'Falha de comunicação com o Arduino\'\n      : luzErr?\'Falha de iluminação — BUZZER ATIVADO\'\n      : fireErr?\'FOGO DETECTADO\'\n      : gasErr?\'Gás acima do limite\'\n      : \'Temperatura crítica\';\n    capAlert=true; cmsg.textContent=\'ALERTA CRÍTICO\'; cmsg.style.color=\'var(--red)\';\n    ov.className=\'overlay show\';\n  } else if(isWarn){\n    box.className=\'ms-box ms-wn\';\n    main.style.color=\'var(--yellow)\';\n    main.textContent=\'ATENÇÃO: PARÂMETRO FORA DA FAIXA IDEAL\';\n    sub.textContent=tempWn?`Temperatura elevada: ${S.tempC.toFixed(1)}°C`:\'Gás se aproximando do limite\';\n    capAlert=false; cmsg.textContent=\'PARÂMETRO FORA DA FAIXA\'; cmsg.style.color=\'var(--yellow)\';\n    ov.className=\'overlay\';\n  } else {\n    box.className=\'ms-box ms-ok\';\n    main.style.color=\'var(--green)\';\n    main.textContent=\'MISSÃO OPERANDO NORMALMENTE\';\n    sub.textContent=\'Todos os parâmetros dentro dos limites\';\n    capAlert=false; cmsg.textContent=\'OPERAÇÃO NORMAL\'; cmsg.style.color=\'var(--cyan)\';\n    ov.className=\'overlay\';\n  }\n\n  // sensores ativos\n  let a=0;\n  if(S.gas!==null)a++; if(S.fire!==null)a++;\n  if(S.luz!==null)a++; if(S.tempC!==null)a++;\n  g(\'st-sens\').textContent=a+\' / 4\';\n}\n\n/* ════════════════════════════════════════════════════════════\n   ALERT HISTORY\n════════════════════════════════════════════════════════════ */\nconst _throttle={};\nfunction trigAlert(sensor, level, msg, val) {\n  const key=sensor+\':\'+level;\n  const now=Date.now();\n  if(_throttle[key]&&(now-_throttle[key])<4500)return;\n  _throttle[key]=now;\n  addEvt(sensor,level,msg,val);\n  S.alertCnt++;\n  g(\'st-alrt\').textContent=S.alertCnt;\n}\n\nfunction addEvt(sensor,level,msg,val) {\n  const ev={id:Date.now()+Math.random(),time:ts(),sensor,level,msg,val,buz:S.buzzer,ack:false};\n  hist.unshift(ev);\n  if(hist.length>600)hist.pop();\n  renderHist();\n}\n\nfunction renderHist() {\n  ANL.histRenders++;\n  g(\'anl-lp-hist-v\').textContent = ANL.histRenders+\' renders\';\n  const sf=g(\'hf-s\').value, lf=g(\'hf-l\').value;\n  const filtered=hist.filter(e=>{\n    if(sf&&e.sensor!==sf)return false;\n    if(lf&&e.level!==lf)return false;\n    return true;\n  });\n  if(!filtered.length){\n    g(\'hlist\').innerHTML=\'<div style="padding:13px;text-align:center;color:var(--t3);font-family:var(--mono);font-size:.62rem">Sem eventos.</div>\';\n    return;\n  }\n  const bc={\'inf\':\'eb-inf\',\'wn\':\'eb-wn\',\'cr\':\'eb-cr\',\'ok\':\'eb-ok\'};\n  const ln={\'inf\':\'INFO\',\'wn\':\'ATENÇÃO\',\'cr\':\'CRÍTICO\',\'ok\':\'OK\'};\n  g(\'hlist\').innerHTML=filtered.slice(0,120).map(e=>`\n    <div class="ev${e.ack?\' acked\':\'\'}" id="ev${e.id}">\n      <span class="ev-t">${e.time}</span>\n      <span class="ev-b ${bc[e.level]||\'eb-inf\'}">${ln[e.level]||e.level}</span>\n      <span class="ev-msg">${e.msg}${e.val!==null&&e.val!==undefined?\' [\'+e.val+\']\':\'\'}</span>\n      <span class="ev-ack" onclick="ackEv(${e.id})">${e.ack?\'✓ ACK\':\'ACK\'}</span>\n    </div>`).join(\'\');\n}\nfunction ackEv(id){const e=hist.find(x=>Math.abs(x.id-id)<1);if(e){e.ack=true;renderHist()}}\nfunction clearHist(){hist=[];renderHist();addEvt(\'sis\',\'inf\',\'Histórico limpo\',null)}\nfunction ackAlerts(){S.ackd=true;addEvt(\'sis\',\'inf\',\'Alertas confirmados pelo operador\',null)}\n\nfunction exportCSV(){\n  const rows=[\'Horário,Sensor,Nível,Mensagem,Valor,Buzzer,ACK\'];\n  hist.forEach(e=>rows.push(`"${e.time}","${e.sensor}","${e.level}","${e.msg}","${e.val??\'\'}","${e.buz?\'Sim\':\'Não\'}","${e.ack?\'Sim\':\'Não\'}"`));\n  const a=document.createElement(\'a\');\n  a.href=URL.createObjectURL(new Blob([rows.join(\'\\n\')],{type:\'text/csv\'}));\n  a.download=\'historico_evortex.csv\'; a.click();\n}\nfunction copyReport(){\n  navigator.clipboard.writeText(hist.map(e=>`${e.time} | ${e.level.toUpperCase()} | ${e.sensor} | ${e.msg}`).join(\'\\n\'))\n    .then(()=>addEvt(\'sis\',\'inf\',\'Relatório copiado para área de transferência\',null))\n    .catch(()=>{});\n}\n\n/* ════════════════════════════════════════════════════════════\n   PARSER — aceita os três formatos\n════════════════════════════════════════════════════════════ */\n// Buffer para modo verbose (Serial.print linha a linha)\nconst VB={gas:null,fire:null,luz:null,raw:null,tempC:null,buz:false,in:false};\n\nfunction processarDadosArduino(linha) {\n  if(!linha||!linha.trim())return;\n  linha=linha.trim();\n\n  // ── JSON ──────────────────────────────────────────────────\n  if(linha.startsWith(\'{\')){\n    try{\n      const j=JSON.parse(linha);\n      const gas  =j.gas   ??j.GAS   ??null;\n      const fire =j.fogo  ??j.FOGO  ??null;\n      const luz  =j.luz   ??j.LUZ   ??null;\n      const raw  =j.temperaturaBruta??j.TEMP_RAW??null;\n      const tC   =j.temperaturaC??j.TEMP??null;\n      const buz  =!!(j.buzzer??j.BUZZER);\n      dispatch(gas,fire,luz,raw,tC,buz);\n    }catch{}\n    return;\n  }\n\n  // ── Compacto CSV: GAS:xx,FOGO:xx,... ────────────────────\n  if(/^GAS:\\d/i.test(linha)&&linha.includes(\',\')){\n    let gas=null,fire=null,luz=null,raw=null,tC=null,buz=false;\n    linha.split(\',\').forEach(p=>{\n      const i=p.indexOf(\':\'); if(i<0)return;\n      const k=p.slice(0,i).trim().toUpperCase();\n      const v=p.slice(i+1).trim();\n      const n=parseFloat(v);\n      if(k===\'GAS\')      gas=n;\n      else if(k===\'FOGO\')fire=n;\n      else if(k===\'LUZ\') luz=n;\n      else if(k===\'TEMP_RAW\')raw=n;\n      else if(k===\'TEMP\'){if(v!==\'nan\'&&v!==\'NAN\')tC=n;}\n      else if(k===\'BUZZER\')buz=n===1||v===\'true\'||v===\'ATIVADO\';\n    });\n    dispatch(gas,fire,luz,raw,tC,buz);\n    return;\n  }\n\n  // ── Verbose (Serial.println do Arduino) ─────────────────\n  const U=linha.toUpperCase();\n\n  // início de bloco\n  if(U.includes(\'TELEMETRIA\')||U.includes(\'MISSAO EVORTEX\')){\n    flushVB(); VB.in=true; return;\n  }\n  // fim de bloco\n  if(U.startsWith(\'---\')||U.startsWith(\'===\')){\n    flushVB(); return;\n  }\n\n  if(!VB.in) VB.in=true; // aceita mesmo sem cabeçalho\n\n  // "Sensor de gas A0: 46"\n  if(U.includes(\'GAS\')&&!U.includes(\'DETECTADO\')&&!U.includes(\'ACIMA\')){\n    const m=linha.match(/:\\s*([\\d.]+)/); if(m)VB.gas=parseFloat(m[1]);\n  }\n  // "Sensor de fogo A1: 987"\n  else if(U.includes(\'FOGO\')&&!U.includes(\'DETECTADO\')){\n    const m=linha.match(/:\\s*([\\d.]+)/); if(m)VB.fire=parseFloat(m[1]);\n  }\n  // "Sensor de luz A3: 59"\n  else if(U.includes(\'LUZ\')&&!U.includes(\'ILUMINACAO\')&&!U.includes(\'ABAIXO\')&&!U.includes(\'NIVEL\')){\n    const m=linha.match(/:\\s*([\\d.]+)/); if(m)VB.luz=parseFloat(m[1]);\n  }\n  // "Temperatura bruta A4: 57"\n  else if(U.includes(\'BRUTA\')){\n    const m=linha.match(/:\\s*([\\d.]+)/); if(m)VB.raw=parseFloat(m[1]);\n  }\n  // "Temperatura: 24.3 graus Celsius"\n  else if(U.startsWith(\'TEMPERATURA\')&&!U.includes(\'BRUTA\')){\n    if(U.includes(\'ERRO\')){VB.tempC=null;}\n    else{\n      const m=linha.match(/([\\d.]+)\\s*grau/i)||linha.match(/:\\s*([\\d.]+)/);\n      if(m)VB.tempC=parseFloat(m[1]);\n    }\n  }\n  // "BUZZER: ATIVADO" / "BUZZER: DESLIGADO"\n  else if(U.includes(\'BUZZER\')){\n    VB.buz=U.includes(\'ATIVADO\')||U.includes(\'1\');\n  }\n}\n\nfunction flushVB(){\n  if(VB.gas===null&&VB.fire===null&&VB.luz===null&&VB.tempC===null&&VB.raw===null)return;\n  dispatch(VB.gas,VB.fire,VB.luz,VB.raw,VB.tempC,VB.buz);\n  VB.gas=null;VB.fire=null;VB.luz=null;VB.raw=null;VB.tempC=null;VB.buz=false;\n}\n\nfunction dispatch(gas,fire,luz,raw,tC,buz){\n  let any=false;\n  if(gas !==null&&!isNaN(gas)) {updateGas(gas);  any=true}\n  if(fire!==null&&!isNaN(fire)){updateFire(fire); any=true}\n  if(luz !==null&&!isNaN(luz)) {updateLuz(luz);   any=true}\n  // temperatura: prefere valor já calculado; se não, usa raw\n  if((tC!==null&&!isNaN(tC))||(raw!==null&&!isNaN(raw))){\n    updateTemp(tC,raw); any=true;\n  }\n  // Buzzer do Arduino (D6): apenas luz decide — mas repassa o estado\n  // O updateLuz já chama setBuzzer com a lógica correta.\n  // Este dispatch só loga se o Arduino sinalizar divergência.\n  if(!any)return;\n\n  ANL.serialBytes += 50; // estimate; exact tracking via readSerial\n\n  const now=Date.now();\n  const freq=S._prevPktTs?`${(1000/(now-S._prevPktTs)).toFixed(1)} Hz`:\'—\';\n  g(\'st-freq\').textContent=freq;\n  g(\'txt-pkt\').textContent=ts();\n  S.lastPkt=now; S._prevPktTs=now;\n  S.commFailed=false; S.ackd=false;\n  g(\'ban-comm\').className=\'banner\';\n  updateStatus();\n  updateAnlPanel();\n  fnLog(\'dispatch\',\'gas,fire,luz,temp\',\'ok\');\n}\n\n/* ════════════════════════════════════════════════════════════\n   SERIAL — Web Serial API\n════════════════════════════════════════════════════════════ */\nasync function conectarArduino(){\n  if(!(\'serial\' in navigator)){\n    alert(\'Web Serial API não disponível.\\nUse Google Chrome ou Edge.\\nNo modo simulação o sistema funciona sem Arduino.\');\n    return;\n  }\n  try{\n    S.port=await navigator.serial.requestPort();\n    await S.port.open({baudRate:9600});\n    S.connected=true; S.lastPkt=Date.now();\n    setConnUI(true);\n    addEvt(\'sis\',\'ok\',\'Arduino conectado via Web Serial API (9600 baud)\',null);\n    readSerial();\n    S._commCheck=setInterval(checkComm,1000);\n  }catch(e){\n    if(e.name!==\'NotFoundError\')addEvt(\'sis\',\'wn\',\'Erro ao conectar: \'+e.message,null);\n  }\n}\n\nasync function readSerial(){\n  const dec=new TextDecoderStream();\n  S.port.readable.pipeThrough(dec);\n  S.reader=dec.readable.getReader();\n  let buf=\'\';\n  try{\n    while(true){\n      const{value,done}=await S.reader.read();\n      if(done)break;\n      buf+=value;\n      ANL.serialBytes += value.length;\n      g(\'anl-lp-serial-v\').textContent = ANL.serialBytes.toLocaleString(\'pt-BR\')+\' bytes\';\n      const lines=buf.split(\'\\n\'); buf=lines.pop();\n      lines.forEach(l=>processarDadosArduino(l));\n    }\n  }catch{}\n}\n\nasync function desconectarArduino(){\n  clearInterval(S._commCheck);\n  if(S.reader){try{await S.reader.cancel()}catch{}S.reader=null}\n  if(S.port){try{await S.port.close()}catch{}S.port=null}\n  S.connected=false; setConnUI(false);\n  addEvt(\'sis\',\'inf\',\'Arduino desconectado pelo operador\',null);\n}\n\nfunction setConnUI(on){\n  const bc=g(\'btn-conn\'), bd=g(\'btn-disc\'), d=g(\'dot\');\n  if(on){\n    bc.textContent=\'⬡ Arduino Conectado\'; bc.classList.add(\'lit\');\n    bd.style.display=\'inline-block\';\n    d.className=\'dot on\'; g(\'txt-status\').textContent=\'Sistema online\';\n    g(\'st-conn\').textContent=\'Conectado\'; g(\'st-conn\').style.color=\'var(--green)\';\n  } else {\n    bc.textContent=\'⬡ Conectar Arduino\'; bc.classList.remove(\'lit\');\n    bd.style.display=\'none\';\n    d.className=\'dot\'; g(\'txt-status\').textContent=\'Sistema offline\';\n    g(\'st-conn\').textContent=\'Offline\'; g(\'st-conn\').style.color=\'var(--t3)\';\n  }\n}\n\nfunction checkComm(){\n  ANL.commChecks++;\n  g(\'anl-lp-check-v\').textContent = ANL.commChecks+\'×\';\n  fnLog(\'checkComm\',\'Δt=\'+(S.lastPkt?(Date.now()-S.lastPkt)+\'ms\':\'—\'));\n  if(S.mode===\'sim\')return;\n  if(!S.connected)return;\n  const dt=S.lastPkt?Date.now()-S.lastPkt:Infinity;\n  if(dt>C.commTO){\n    if(!S.commFailed){\n      S.commFailed=true;\n      g(\'ban-comm\').className=\'banner ban-err show\';\n      addEvt(\'sis\',\'cr\',\'Falha de comunicação — telemetria congelada há \'+(dt/1000).toFixed(0)+\'s\',null);\n      updateStatus();\n    }\n  } else if(S.commFailed){\n    S.commFailed=false;\n    g(\'ban-comm\').className=\'banner\';\n    addEvt(\'sis\',\'ok\',\'Comunicação com Arduino restaurada\',null);\n    updateStatus();\n  }\n}\n\n/* ════════════════════════════════════════════════════════════\n   MODE\n════════════════════════════════════════════════════════════ */\nfunction setMode(m){\n  S.mode=m;\n  g(\'btn-mr\').className=\'mbt\'+(m===\'real\'?\' ar\':\'\');\n  g(\'btn-ms\').className=\'mbt\'+(m===\'sim\'?\' as\':\'\');\n  g(\'sim-bar\').className=\'sim-bar\'+(m===\'sim\'?\' show\':\'\');\n  g(\'st-mode\').textContent=m===\'sim\'?\'Simulação\':\'Real\';\n  g(\'st-mode\').style.color=m===\'sim\'?\'var(--yellow)\':\'var(--cyan)\';\n  if(m===\'sim\'){\n    g(\'dot\').className=\'dot on\';\n    g(\'txt-status\').textContent=\'Sistema online (simulação)\';\n    addEvt(\'sis\',\'inf\',\'Modo simulação ativado\',null);\n  } else {\n    simStop();\n    addEvt(\'sis\',\'inf\',\'Modo real ativado\',null);\n    if(!S.connected){g(\'dot\').className=\'dot\';g(\'txt-status\').textContent=\'Sistema offline\'}\n  }\n}\n\n/* ════════════════════════════════════════════════════════════\n   SIMULATION ENGINE\n   Valores espelham as faixas reais observadas no Arduino.\n════════════════════════════════════════════════════════════ */\nlet sGas=52, sFire=982, sLuz=60, sTemp=24.2;\n\nfunction simTick(){\n  // deriva natural\n  sGas  +=(Math.random()-.48)*5;    sGas =Math.max(10,Math.min(1010,sGas));\n  sFire +=(Math.random()-.5)*10;    sFire=Math.max(0,Math.min(1023,sFire));\n  sLuz  +=(Math.random()-.48)*4;    sLuz =Math.max(0,Math.min(1023,sLuz));\n  sTemp +=(Math.random()-.49)*.35;  sTemp=Math.max(-5,Math.min(80,sTemp));\n  const buz = sLuz < C.luzLim;\n  // emite no mesmo formato compacto que o Arduino\n  processarDadosArduino(\n    `GAS:${Math.round(sGas)},FOGO:${Math.round(sFire)},LUZ:${Math.round(sLuz)},TEMP_RAW:${Math.round(sTemp*2)},TEMP:${sTemp.toFixed(1)},BUZZER:${buz?1:0}`\n  );\n}\n\nfunction simStart(){\n  if(S.simRunning)return;\n  S.simRunning=true;\n  S.simTick=setInterval(simTick,500);\n  setMode(\'sim\');\n}\nfunction simStop(){\n  S.simRunning=false;\n  clearInterval(S.simTick); S.simTick=null;\n  clearInterval(S.simAutoTick); S.simAutoTick=null;\n}\n\nfunction simAuto(){\n  simStart();\n  clearInterval(S.simAutoTick);\n  const phases=[\'normal\',\'gas\',\'fogo\',\'luz\',\'temp\',\'comm\'];\n  let pi=0;\n  S.simAutoTick=setInterval(()=>{\n    pi=(pi+1)%phases.length;\n    const p=phases[pi];\n    addEvt(\'sis\',\'inf\',\'Simulação auto: fase "\'+p+\'"\',null);\n    if(p===\'gas\')     simGas();\n    else if(p===\'fogo\') simFogo();\n    else if(p===\'luz\')  simLuz();\n    else if(p===\'temp\') simTemp();\n    else if(p===\'comm\') simComm();\n    else                simRestore();\n  },7000);\n}\n\nfunction simGas()   { simStart(); sGas=C.gasLim+100+Math.random()*100; addEvt(\'gas\',\'cr\',\'Sim: gás detectado\',Math.round(sGas)); }\nfunction simFogo()  { simStart(); sFire=Math.max(5,C.fireLim-120-Math.random()*80); addEvt(\'fogo\',\'cr\',\'Sim: fogo detectado\',Math.round(sFire)); }\nfunction simLuz()   { simStart(); sLuz=Math.max(0,C.luzLim-20-Math.random()*20); addEvt(\'luz\',\'cr\',\'Sim: falha de iluminação\',Math.round(sLuz)); }\nfunction simTemp()  { simStart(); sTemp=C.tempC+4+Math.random()*8; addEvt(\'temp\',\'cr\',\'Sim: temperatura crítica\',sTemp.toFixed(1)); }\nfunction simComm()  {\n  simStop();\n  S.lastPkt=Date.now()-(C.commTO+1500);\n  S.commFailed=true;\n  g(\'ban-comm\').className=\'banner ban-err show\';\n  addEvt(\'sis\',\'cr\',\'Sim: falha de comunicação\',null);\n  updateStatus();\n  setTimeout(()=>{\n    S.commFailed=false;\n    g(\'ban-comm\').className=\'banner\';\n    addEvt(\'sis\',\'ok\',\'Sim: comunicação restaurada\',null);\n    updateStatus(); simStart();\n  },5000);\n}\nfunction simRestore(){\n  sGas=45+Math.random()*25; sFire=965+Math.random()*55;\n  sLuz=55+Math.random()*18; sTemp=22.5+Math.random()*4;\n  S.commFailed=false; g(\'ban-comm\').className=\'banner\';\n  addEvt(\'sis\',\'ok\',\'Sistema restaurado para parâmetros normais\',null);\n}\n\n/* ════════════════════════════════════════════════════════════\n   ALARM SOUND\n════════════════════════════════════════════════════════════ */\nfunction toggleSound(){\n  if(!S.sndEnabled){\n    S.sndEnabled=true; S.sndMuted=false;\n    g(\'btn-snd\').textContent=\'🔊 Som: Ativo\';\n    g(\'btn-snd\').style.color=\'var(--green)\';\n    if(S.buzzer)playAlarm();\n    addEvt(\'sis\',\'inf\',\'Som do alarme virtual ativado\',null);\n  } else if(!S.sndMuted){\n    S.sndMuted=true;\n    g(\'btn-snd\').textContent=\'🔇 Mudo\';\n    g(\'btn-snd\').style.color=\'var(--t3)\';\n    stopAlarm();\n  } else {\n    S.sndEnabled=false; S.sndMuted=false;\n    g(\'btn-snd\').textContent=\'🔊 Som virtual\';\n    g(\'btn-snd\').style.color=\'var(--yellow)\';\n    stopAlarm();\n  }\n}\nfunction playAlarm(){\n  if(!S.sndEnabled||S.sndMuted)return;\n  try{\n    actx=actx||new(window.AudioContext||window.webkitAudioContext)();\n    if(alarmOsc){try{alarmOsc.stop()}catch{}alarmOsc=null}\n    const o=actx.createOscillator(), gn=actx.createGain();\n    o.connect(gn); gn.connect(actx.destination);\n    o.type=\'square\';\n    o.frequency.setValueAtTime(880,actx.currentTime);\n    o.frequency.setValueAtTime(660,actx.currentTime+.18);\n    o.frequency.setValueAtTime(880,actx.currentTime+.36);\n    gn.gain.setValueAtTime(Math.min(.25,C.vol*.25),actx.currentTime);\n    o.start(); alarmOsc=o;\n    setTimeout(()=>{\n      if(alarmOsc){try{alarmOsc.stop()}catch{}alarmOsc=null}\n      if(S.buzzer&&!S.sndMuted)playAlarm();\n    },540);\n  }catch{}\n}\nfunction stopAlarm(){\n  if(alarmOsc){try{alarmOsc.stop()}catch{}alarmOsc=null}\n}\n\n/* ════════════════════════════════════════════════════════════\n   CONFIG\n════════════════════════════════════════════════════════════ */\nfunction applyConfig(){\n  C.gasLim  = +g(\'cfg-gas\').value||200;\n  C.fireLim = +g(\'cfg-fire\').value||300;\n  C.luzLim  = +g(\'cfg-luz\').value||50;\n  C.tempW   = +g(\'cfg-tw\').value||30;\n  C.tempC   = +g(\'cfg-tc\').value||40;\n  C.commTO  = (+g(\'cfg-to\').value||3)*1000;\n  C.vol     = +g(\'cfg-vol\').value||0.4;\n  updateStatus();\n}\nfunction resetConfig(){\n  g(\'cfg-gas\').value=200; g(\'cfg-fire\').value=300; g(\'cfg-luz\').value=50;\n  g(\'cfg-tw\').value=30;   g(\'cfg-tc\').value=40;   g(\'cfg-to\').value=3;\n  g(\'cfg-vol\').value=0.4;\n  applyConfig();\n  addEvt(\'sis\',\'inf\',\'Configurações restauradas para os padrões\',null);\n}\n\n/* ════════════════════════════════════════════════════════════\n   UI HELPERS\n════════════════════════════════════════════════════════════ */\nconst g = id => document.getElementById(id);\nconst ts = () => new Date().toLocaleTimeString(\'pt-BR\');\n\nfunction swTab(id,btn){\n  const panel=btn.closest(\'.panel\');\n  panel.querySelectorAll(\'.tc\').forEach(t=>t.classList.remove(\'act\'));\n  panel.querySelectorAll(\'.tb\').forEach(b=>b.classList.remove(\'act\'));\n  g(id).classList.add(\'act\'); btn.classList.add(\'act\');\n}\nfunction chTab(key,btn){\n  [\'gas\',\'fire\',\'luz\',\'temp\'].forEach(k=>g(\'tc-\'+k).classList.remove(\'act\'));\n  btn.closest(\'.panel\').querySelectorAll(\'.tb\').forEach(b=>b.classList.remove(\'act\'));\n  g(\'tc-\'+key).classList.add(\'act\'); btn.classList.add(\'act\');\n}\n\n/* ════════════════════════════════════════════════════════════\n   CLOCK / UPTIME\n════════════════════════════════════════════════════════════ */\nfunction tick(){\n  ANL.tickCycles++;\n  g(\'anl-lp-tick-v\').textContent = ANL.tickCycles+\'×\';\n  const now=new Date();\n  g(\'txt-time\').textContent=now.toLocaleString(\'pt-BR\');\n  const s=Math.floor((Date.now()-S.t0)/1000);\n  const up=`${String(Math.floor(s/3600)).padStart(2,\'0\')}:${String(Math.floor((s%3600)/60)).padStart(2,\'0\')}:${String(s%60).padStart(2,\'0\')}`;\n  g(\'txt-up\').textContent=up; g(\'st-up\').textContent=up;\n  if(S.buzzerAt) g(\'bz-time\').textContent=`Ativo há ${Math.floor((Date.now()-S.buzzerAt)/1000)}s`;\n  else g(\'bz-time\').textContent=\'\';\n}\n\n/* ════════════════════════════════════════════════════════════\n   MOTOR DE ANÁLISE — exibe em tempo real o uso de:\n   condicionais, repetição, vetores e funções\n════════════════════════════════════════════════════════════ */\n\n// ── Contadores globais ──────────────────────────────────────\nconst ANL = {\n  ifFired:   0,  // total de condicionais disparadas\n  loopCycles:0,  // ciclos de loop (pacotes recebidos)\n  fnCalls:   0,  // total de chamadas de funções rastreadas\n  serialBytes:0, // bytes acumulados lidos da serial\n  commChecks: 0, // vezes que checkComm rodou\n  tickCycles: 0, // vezes que tick() rodou\n  chartUpdates:0,// atualizações de gráfico\n  histRenders: 0,// renderizações do histórico\n};\n\n// ── Log de funções (máx 30 entradas) ───────────────────────\nconst FN_LOG = [];\nconst FN_MAX = 30;\n\nfunction fnLog(name, args, ret) {\n  ANL.fnCalls++;\n  FN_LOG.unshift({ name, args: args||\'\', ret: ret!==undefined?ret:\'\', t: ts() });\n  if(FN_LOG.length > FN_MAX) FN_LOG.pop();\n  g(\'anl-fn-count\').textContent = ANL.fnCalls + \' chamadas\';\n  renderFnLog();\n}\n\nfunction renderFnLog() {\n  const el = g(\'fn-log\');\n  if(!el) return;\n  el.innerHTML = FN_LOG.slice(0,14).map((e,i)=>`\n    <div class="fn-entry${i===0?\' new\':\'\'}">\n      <span class="fn-name">${e.name}()</span>\n      <span class="fn-args">${e.args}</span>\n      ${e.ret!==\'\'?`<span class="fn-ret">→ ${e.ret}</span>`:\'\'}\n      <span class="fn-time">${e.t}</span>\n    </div>`).join(\'\');\n}\n\n// ── Actualiza painel de condicionais ───────────────────────\nfunction anlUpdateIf(id, fired, valStr, level) {\n  const el = g(id);\n  if(!el) return;\n  const clsMap = { err:\'active-err\', warn:\'active-warn\', ok:\'active\', off:\'\' };\n  el.className = \'anl-item \' + (fired ? (clsMap[level]||\'active\') : \'\');\n  // flash quando muda para ativo\n  if(fired) { el.classList.add(\'flash\'); setTimeout(()=>el.classList.remove(\'flash\'), 450); }\n  const vel = g(id+\'-v\');\n  if(vel) vel.textContent = valStr || \'—\';\n  if(fired) { ANL.ifFired++; g(\'anl-if-count\').textContent = ANL.ifFired; }\n}\n\n// ── Actualiza células de vetor ──────────────────────────────\nfunction renderVecCells(containerId, arr, isAlert) {\n  const el = g(containerId);\n  if(!el) return;\n  const slice = arr.slice(-20);\n  el.innerHTML = slice.map((v,i)=>{\n    const isNew = i===slice.length-1;\n    const alert = isAlert && isAlert(v);\n    const cls = alert?\'vec-cell alert-c\': isNew?\'vec-cell newest\':\'vec-cell filled\';\n    const disp = typeof v===\'number\' ? (v>99?Math.round(v): v.toFixed?v.toFixed(0):v) : \'?\';\n    return `<div class="${cls}" title="${v}">${disp}</div>`;\n  }).join(\'\');\n}\n\nfunction renderVecHistCells() {\n  const el = g(\'vec-hist\');\n  if(!el) return;\n  const slice = hist.slice(0,20);\n  el.innerHTML = slice.map((e,i)=>{\n    const cls = e.level===\'cr\'?\'vec-cell alert-c\': e.level===\'wn\'?\'vec-cell alert-c\': i===0?\'vec-cell newest\':\'vec-cell filled\';\n    const lbl = {inf:\'i\',wn:\'!\',cr:\'✕\',ok:\'✓\'}[e.level]||\'?\';\n    return `<div class="${cls}" title="${e.msg}">${lbl}</div>`;\n  }).join(\'\');\n}\n\n// ── Função central chamada a cada pacote ───────────────────\nfunction updateAnlPanel() {\n  // contadores de loop\n  ANL.loopCycles++;\n  g(\'anl-lp-main-v\').textContent   = ANL.loopCycles + \'×\';\n  g(\'anl-lp-serial-v\').textContent = ANL.serialBytes.toLocaleString(\'pt-BR\') + \' bytes\';\n  g(\'anl-lp-chart-v\').textContent  = ANL.chartUpdates + \'×\';\n  g(\'anl-lp-hist-v\').textContent   = ANL.histRenders + \' renders\';\n  g(\'anl-lp-check-v\').textContent  = ANL.commChecks + \'×\';\n  g(\'anl-lp-tick-v\').textContent   = ANL.tickCycles + \'×\';\n  g(\'anl-loop-count\').textContent  = ANL.loopCycles + \'×\';\n\n  // highlight laços ativos\n  [\'anl-lp-main\',\'anl-lp-serial\',\'anl-lp-check\',\'anl-lp-tick\'].forEach(id=>{\n    const e=g(id); if(e){ e.classList.add(\'flash\'); setTimeout(()=>e.classList.remove(\'flash\'),400); }\n  });\n  if(ANL.loopCycles>0){ const e=g(\'anl-lp-parse\'); if(e)e.className=\'anl-item active\'; }\n\n  // ── condicionais ────────────────────────────────────────\n  // gas\n  const gasF  = S.gas!==null && S.gas>=C.gasLim;\n  const gasW  = S.gas!==null && !gasF && S.gas>=C.gasLim*.7;\n  anlUpdateIf(\'anl-if-gas\',  gasF,  S.gas!==null?`${S.gas} ≥ ${C.gasLim}`:\'—\', gasF?\'err\': gasW?\'warn\':\'off\');\n  // fogo\n  const fireF = S.fire!==null && S.fire<C.fireLim;\n  anlUpdateIf(\'anl-if-fogo\', fireF, S.fire!==null?`${S.fire} < ${C.fireLim}`:\'—\', fireF?\'err\':\'off\');\n  // luz\n  const luzF  = S.luz!==null && S.luz<C.luzLim;\n  anlUpdateIf(\'anl-if-luz\',  luzF,  S.luz!==null?`${S.luz} < ${C.luzLim}`:\'—\', luzF?\'err\':\'off\');\n  g(\'anl-fn-luz-ret\').textContent = luzF?\'FALHA\':\'OK\';\n  g(\'anl-fn-luz-ret\').style.color = luzF?\'var(--red)\':\'var(--green)\';\n  // fogo retorno\n  g(\'anl-fn-fire-ret\').textContent = fireF?\'FOGO\':\'Sem chama\';\n  g(\'anl-fn-fire-ret\').style.color = fireF?\'var(--red)\':\'var(--green)\';\n  // temp crítica\n  const tempCF = S.tempC!==null && S.tempC>=C.tempC;\n  const tempWF = S.tempC!==null && !tempCF && S.tempC>=C.tempW;\n  anlUpdateIf(\'anl-if-temp-c\', tempCF, S.tempC!==null?`${S.tempC.toFixed(1)}°C ≥ ${C.tempC}°C`:\'—\', tempCF?\'err\':\'off\');\n  anlUpdateIf(\'anl-if-temp-w\', tempWF, S.tempC!==null?`${S.tempC.toFixed(1)}°C ≥ ${C.tempW}°C`:\'—\', tempWF?\'warn\':\'off\');\n  // NTC range check\n  const ntcF  = S.tempRaw!==null && (S.tempRaw<=1 || S.tempRaw>=1022);\n  anlUpdateIf(\'anl-if-ntc\', ntcF, S.tempRaw!==null?`raw=${S.tempRaw}`:\'—\', ntcF?\'err\':\'off\');\n  // comm\n  const commF = S.commFailed;\n  anlUpdateIf(\'anl-if-comm\', commF, commF?\'timeout\':\'OK\', commF?\'err\':\'off\');\n\n  // ── NTC fn retorno ───────────────────────────────────────\n  if(S.tempC!==null){\n    g(\'anl-fn-ntc-ret\').textContent = S.tempC.toFixed(2)+\'°C\';\n    g(\'anl-fn-ntc-ret\').style.color = tempCF?\'var(--red)\':tempWF?\'var(--yellow)\':\'var(--green)\';\n  }\n  // buzzer\n  g(\'anl-fn-buz-ret\').textContent = S.buzzer?\'ATIVO\':\'false\';\n  g(\'anl-fn-buz-ret\').style.color = S.buzzer?\'var(--red)\':\'var(--green)\';\n\n  // ── vetores ─────────────────────────────────────────────\n  const total = hist.length + CD.gas.d.length + CD.fire.d.length + CD.luz.d.length + CD.temp.d.length;\n  g(\'anl-vec-count\').textContent = total + \' itens\';\n  g(\'vec-hist-sz\').textContent   = hist.length;\n  g(\'vec-gas-sz\').textContent    = CD.gas.d.length;\n  g(\'vec-fire-sz\').textContent   = CD.fire.d.length;\n  g(\'vec-luz-sz\').textContent    = CD.luz.d.length;\n  g(\'vec-temp-sz\').textContent   = CD.temp.d.length;\n\n  renderVecCells(\'vec-gas\',  CD.gas.d,  v=>v>=C.gasLim);\n  renderVecCells(\'vec-temp\', CD.temp.d, v=>v>=C.tempC);\n  renderVecHistCells();\n\n  // ── campos do último pacote ──────────────────────────────\n  const campos = [S.gas,S.fire,S.luz,S.tempC].filter(v=>v!==null).length;\n  g(\'anl-lp-parse-v\').textContent = campos + \' campos\';\n}\n\n\ndocument.addEventListener(\'DOMContentLoaded\',()=>{\n  initCharts();\n  applyConfig();\n  addEvt(\'sis\',\'inf\',\'Sistema EVORTEX iniciado — aguardando dados\',null);\n  setInterval(tick,1000); tick();\n  setInterval(checkComm,1000);\n  renderFnLog();\n  updateAnlPanel();\n  ariaInit(); // inicia o chatbot\n});\n\n/* ════════════════════════════════════════════════════════════\n   POPUP DE ALERTA CRÍTICO\n   Abre automaticamente quando um sensor entra em estado crítico.\n   Exibe quais sistemas falharam + protocolo de ação passo a passo.\n════════════════════════════════════════════════════════════ */\n\n// Protocolos de resposta para cada sensor (índice = posição do sensor)\nconst PROTOCOLOS = {\n  gas: {\n    icon: \'☁\',\n    nome: \'Sensor de Gás\',\n    titulo: \'Protocolo de Gás\',\n    passos: [\n      \'Interromper imediatamente todas as atividades na cápsula\',\n      \'Verificar a origem do vazamento de gás pelo inspetor visual\',\n      \'Ativar o sistema de ventilação forçada da câmara\',\n      \'Acionar o sensor de gás secundário para confirmação\',\n      \'Se nível persistir: evacuar compartimento e isolar módulo\',\n    ]\n  },\n  fogo: {\n    icon: \'🔥\',\n    nome: \'Sensor de Fogo\',\n    titulo: \'Protocolo de Incêndio\',\n    passos: [\n      \'EMERGÊNCIA: acionar extintor automático imediatamente\',\n      \'Cortar alimentação elétrica dos circuitos do setor afetado\',\n      \'Acionar alarme sonoro e comunicar a base terrestre\',\n      \'Retirar a tripulação do compartimento afetado\',\n      \'Monitorar propagação — se incontrolável: protocolo de evacuação\',\n    ]\n  },\n  luz: {\n    icon: \'💡\',\n    nome: \'Sistema de Iluminação\',\n    titulo: \'Protocolo de Falha de Luz\',\n    passos: [\n      \'Ativar iluminação de emergência (backup autônomo)\',\n      \'Verificar fusíveis e conexões do circuito de iluminação\',\n      \'Checar o sensor LDR no pino A3 — possível obstrução física\',\n      \'Reduzir consumo geral para estabilizar o sistema elétrico\',\n      \'Se falha persistir: registrar evento e aguardar manutenção\',\n    ]\n  },\n  temp: {\n    icon: \'🌡\',\n    nome: \'Temperatura Interna\',\n    titulo: \'Protocolo Térmico\',\n    passos: [\n      \'Reduzir imediatamente carga dos sistemas não essenciais\',\n      \'Ativar o sistema de resfriamento auxiliar da câmara\',\n      \'Verificar isolamento térmico e status do sensor NTC (pino A4)\',\n      \'Monitorar a temperatura a cada 30 segundos até estabilizar\',\n      \'Se temperatura > 45°C: iniciar protocolo de evacuação de emergência\',\n    ]\n  }\n};\n\n// Controla quais alertas já dispararam popup (evita spam)\nconst _popupJaAberto = { gas:false, fogo:false, luz:false, temp:false };\n\nfunction abrirPopupAlerta(sensoresAlerta) {\n  /* sensoresAlerta = array de strings: [\'gas\',\'fogo\',...] */\n  const body = g(\'popup-box\').querySelector(\'.popup-body\');\n  const titulo = g(\'popup-title\');\n  body.innerHTML = \'\';\n\n  const criticos = sensoresAlerta.filter(s => s.nivel === \'crit\');\n  const atencoes = sensoresAlerta.filter(s => s.nivel === \'warn\');\n\n  titulo.textContent = criticos.length\n    ? `🚨 ${criticos.length > 1 ? \'MÚLTIPLOS ALERTAS CRÍTICOS\' : \'ALERTA CRÍTICO\'}`\n    : \'⚠️ ATENÇÃO — PARÂMETRO FORA DO NORMAL\';\n\n  // Resumo dos sensores afetados\n  const resumo = document.createElement(\'div\');\n  resumo.style.cssText = \'display:flex;flex-direction:column;gap:6px;margin-bottom:4px\';\n  sensoresAlerta.forEach(s => {\n    const row = document.createElement(\'div\');\n    row.className = \'popup-sensor-row\' + (s.nivel === \'warn\' ? \' warn\' : \'\');\n    const isCrit = s.nivel === \'crit\';\n    row.innerHTML = `\n      <span class="popup-sensor-icon">${PROTOCOLOS[s.key]?.icon || \'⚠\'}</span>\n      <span class="popup-sensor-name">${PROTOCOLOS[s.key]?.nome || s.key}</span>\n      <span class="popup-sensor-val${isCrit ? \'\' : \' warn-v\'}">${s.valor}</span>\n      <span style="margin-left:auto;font-family:var(--mono);font-size:.58rem;\n        color:${isCrit ? \'var(--red)\' : \'var(--yellow)\'}">\n        ${isCrit ? \'● CRÍTICO\' : \'● ATENÇÃO\'}\n      </span>`;\n    resumo.appendChild(row);\n  });\n  body.appendChild(resumo);\n\n  // Protocolos de ação apenas para os críticos\n  criticos.forEach(s => {\n    const proto = PROTOCOLOS[s.key];\n    if (!proto) return;\n    const bloco = document.createElement(\'div\');\n    bloco.className = \'popup-protocol\';\n    bloco.innerHTML = `\n      <div class="popup-protocol-hdr">▶ ${proto.titulo}</div>\n      <div class="popup-protocol-steps">\n        ${proto.passos.map((p, i) => `\n          <div class="popup-step">\n            <span class="popup-step-num">${i+1}.</span>\n            <span>${p}</span>\n          </div>`).join(\'\')}\n      </div>`;\n    body.appendChild(bloco);\n  });\n\n  // Recomendação ARIA\n  const rec = document.createElement(\'div\');\n  rec.style.cssText = \'padding:8px 12px;background:rgba(0,212,255,.06);border:1px solid rgba(0,212,255,.18);border-radius:3px;font-family:var(--mono);font-size:.62rem;color:var(--t2)\';\n  rec.innerHTML = `<span style="color:var(--cyan)">💬 ARIA:</span> ${_ariaRecAlerta(criticos, atencoes)}`;\n  body.appendChild(rec);\n\n  g(\'popup-overlay\').classList.add(\'show\');\n}\n\nfunction fecharPopup() {\n  g(\'popup-overlay\').classList.remove(\'show\');\n}\n\n// Fecha o popup clicando fora\ng(\'popup-overlay\') && document.getElementById(\'popup-overlay\').addEventListener(\'click\', function(e) {\n  if (e.target === this) fecharPopup();\n});\n\nfunction _ariaRecAlerta(criticos, atencoes) {\n  if (criticos.length >= 3) return \'Emergência múltipla detectada. Ativar modo de segurança total imediatamente e priorizar suporte à vida.\';\n  const c = criticos[0];\n  if (!c) return `${atencoes.length} sistema(s) em atenção. Monitorar de perto e preparar contingência.`;\n  const msgs = {\n    fogo: \'Incêndio detectado. Siga o Protocolo de Incêndio agora — não aguarde.\',\n    gas:  \'Gás detectado. Ventilação forçada e isolamento do módulo são prioridade.\',\n    luz:  \'Falha de iluminação ativa o buzzer D6. Verifique o circuito e ative o backup.\',\n    temp: \'Temperatura crítica. Ative o resfriamento auxiliar e reduza a carga dos sistemas.\',\n  };\n  return msgs[c.key] || \'Sensor crítico ativo. Consulte o protocolo correspondente.\';\n}\n\n// Hook nas funções existentes de atualização de sensores para disparar popup\n// Sobrescreve levemente o updateStatus() já existente\nconst _origUpdateStatus = updateStatus;\nupdateStatus = function() {\n  _origUpdateStatus();\n  _verificarPopupNecessario();\n};\n\nlet _lastPopupHash = \'\';\nfunction _verificarPopupNecessario() {\n  const alertas = [];\n\n  if (S.gas !== null) {\n    if (S.gas >= C.gasLim) alertas.push({ key:\'gas\', nivel:\'crit\', valor: S.gas+\' raw\' });\n    else if (S.gas >= C.gasLim * 0.7) alertas.push({ key:\'gas\', nivel:\'warn\', valor: S.gas+\' raw\' });\n  }\n  if (S.fire !== null) {\n    if (S.fire < C.fireLim) alertas.push({ key:\'fogo\', nivel:\'crit\', valor: S.fire+\' raw\' });\n    else if (S.fire < C.fireLim * 1.35) alertas.push({ key:\'fogo\', nivel:\'warn\', valor: S.fire+\' raw\' });\n  }\n  if (S.luz !== null) {\n    if (S.luz < C.luzLim) alertas.push({ key:\'luz\', nivel:\'crit\', valor: S.luz+\' raw\' });\n  }\n  if (S.tempC !== null) {\n    if (S.tempC >= C.tempC) alertas.push({ key:\'temp\', nivel:\'crit\', valor: S.tempC.toFixed(1)+\'°C\' });\n    else if (S.tempC >= C.tempW) alertas.push({ key:\'temp\', nivel:\'warn\', valor: S.tempC.toFixed(1)+\'°C\' });\n  }\n\n  const hash = alertas.map(a=>a.key+a.nivel).sort().join(\',\');\n  const temCritico = alertas.some(a => a.nivel === \'crit\');\n\n  // Abre popup apenas quando o conjunto de críticos muda\n  if (temCritico && hash !== _lastPopupHash) {\n    _lastPopupHash = hash;\n    abrirPopupAlerta(alertas);\n    // notifica o chatbot ARIA\n    ariaReceberAlertas(alertas);\n  }\n  if (!temCritico) _lastPopupHash = \'\';\n}\n\n\n/* ════════════════════════════════════════════════════════════\n   CHATBOT ARIA\n   Interface conversacional que conhece o estado atual da missão.\n   Responde comandos de texto e perguntas sobre sensores, alertas\n   e como usar a interface do sistema EVORTEX.\n════════════════════════════════════════════════════════════ */\n\nlet _ariaOpen    = false;\nlet _ariaHistory = [];      // histórico da conversa (para contexto)\nlet _ariaTyping  = false;\n\n// Base de conhecimento da ARIA — respostas locais sem IA externa\n/* ════════════════════════════════════════════════════════════\n   BASE DE CONHECIMENTO COMPLETA DA ARIA\n   Contém tudo sobre a Missão EVORTEX:\n   — o que é a missão, objetivos, hardware, sensores,\n     limites, protocolos, interface, alertas, Arduino,\n     código, simulação, histórico, gráficos e mais.\n════════════════════════════════════════════════════════════ */\n\nconst ARIA_KB = {\n\n  // ─────────────────────────────────────────────────────────\n  // O QUE É A MISSÃO EVORTEX\n  // ─────────────────────────────────────────────────────────\n  missao: () => `\n<b>🛰 O que é a Missão EVORTEX?</b><br><br>\nA <b>Missão EVORTEX</b> é uma missão espacial experimental desenvolvida como projeto acadêmico. O objetivo é simular o monitoramento ambiental interno de uma <b>cápsula espacial</b> utilizando sensores reais conectados a um <b>Arduino Uno</b>.<br><br>\nO sistema monitora em tempo real as condições internas da cápsula que seriam críticas em uma missão real:<br>\n• Temperatura interna da cabine<br>\n• Presença de fogo ou chamas<br>\n• Vazamento de gás<br>\n• Funcionamento do sistema de iluminação<br><br>\nOs dados são enviados pelo Arduino via porta serial USB e exibidos nesta interface web em tempo real, com alertas automáticos, gráficos, histórico e protocolos de resposta.`,\n\n  // ─────────────────────────────────────────────────────────\n  // OBJETIVOS DA MISSÃO\n  // ─────────────────────────────────────────────────────────\n  objetivos: () => `\n<b>🎯 Objetivos da Missão EVORTEX:</b><br><br>\n<b>Técnicos:</b><br>\n• Monitorar continuamente 4 sensores ambientais da cápsula<br>\n• Detectar situações de risco e emitir alertas automáticos<br>\n• Acionar o buzzer de emergência (D6) em caso de falha<br>\n• Transmitir telemetria em tempo real via porta serial<br><br>\n<b>Acadêmicos:</b><br>\n• Demonstrar uso de estruturas condicionais, repetição, vetores e funções<br>\n• Aplicar comunicação Arduino ↔ interface web (Web Serial API)<br>\n• Construir um sistema completo de monitoramento com hardware real<br><br>\n<b>Resultado esperado:</b><br>\nUm painel de controle funcional que simula a central de monitoramento de uma missão espacial real.`,\n\n  // ─────────────────────────────────────────────────────────\n  // HARDWARE — ARDUINO E COMPONENTES\n  // ─────────────────────────────────────────────────────────\n  hardware: () => `\n<b>🔧 Hardware da Missão EVORTEX:</b><br><br>\n<b>Microcontrolador:</b> Arduino Uno<br><br>\n<b>Sensores conectados:</b><br>\n• <b>A0</b> — Sensor de Gás (MQ-2 ou similar)<br>\n• <b>A1</b> — Sensor de Fogo (fotodiodo infravermelho)<br>\n• <b>A3</b> — Sensor de Luminosidade (LDR)<br>\n• <b>A4</b> — Sensor de Temperatura (termistor NTC)<br><br>\n<b>Atuadores:</b><br>\n• <b>D6</b> — Buzzer piezoelétrico (sirene de emergência)<br>\n• LED de iluminação ligado diretamente no 5V (não controlado pelo Arduino)<br><br>\n<b>Comunicação:</b> USB Serial, 9600 baud<br>\n<b>Interface:</b> Web Serial API no Chrome/Edge`,\n\n  // ─────────────────────────────────────────────────────────\n  // SENSORES — EXPLICAÇÃO COMPLETA\n  // ─────────────────────────────────────────────────────────\n  sensores: () => `\n<b>🔬 Sensores monitorados — limites EVORTEX:</b><br><br>\n☁ <b>Gás — Pino A0</b><br>\n&nbsp;&nbsp;Mede concentração de gás na câmara (0–1023 raw)<br>\n&nbsp;&nbsp;Normal: &lt;${Math.round(C.gasLim*0.7)} | Atenção: ${Math.round(C.gasLim*0.7)}–${C.gasLim} | Crítico: &gt;${C.gasLim}<br><br>\n🔥 <b>Fogo — Pino A1 (lógica invertida)</b><br>\n&nbsp;&nbsp;Detecta chamas por infravermelho (0–1023 raw)<br>\n&nbsp;&nbsp;⚠ Invertido: valor ALTO = sem fogo | valor BAIXO = fogo<br>\n&nbsp;&nbsp;Normal: &gt;${C.fireLim+50} | Atenção: ${C.fireLim}–${C.fireLim+50} | Crítico: &lt;${C.fireLim}<br><br>\n💡 <b>Luminosidade — Pino A3</b><br>\n&nbsp;&nbsp;Verifica se a iluminação interna está funcionando<br>\n&nbsp;&nbsp;Funcionando: ≥${C.luzLim} | Falha: &lt;${C.luzLim}<br>\n&nbsp;&nbsp;O LED está ligado direto no 5V — o sensor só verifica<br><br>\n🌡 <b>Temperatura — Pino A4 (termistor NTC)</b><br>\n&nbsp;&nbsp;Mede temperatura interna (β=3950, R₀=100kΩ, Rfix=6kΩ)<br>\n&nbsp;&nbsp;Normal: 18–30°C | Atenção: 31–35°C | Crítico: &gt;35°C`,\n\n  // ─────────────────────────────────────────────────────────\n  // SENSOR ESPECÍFICO: GÁS\n  // ─────────────────────────────────────────────────────────\n  gas: () => `\n<b>☁ Sensor de Gás — Pino A0</b><br><br>\n<b>O que mede:</b> concentração de gás inflamável ou tóxico na câmara da cápsula.<br><br>\n<b>Como funciona:</b> envia um valor analógico de 0 a 1023. Quanto maior o valor, maior a concentração de gás.<br><br>\n<b>Limites configurados:</b><br>\n• Normal: abaixo de ${Math.round(C.gasLim*0.7)} raw<br>\n• Atenção: ${Math.round(C.gasLim*0.7)} a ${C.gasLim} raw<br>\n• Crítico: acima de ${C.gasLim} raw<br><br>\n<b>Valor atual:</b> ${S.gas !== null ? S.gas+\' raw\' : \'Sem sinal\'}<br><br>\n<b>Se crítico:</b> ativar ventilação forçada, identificar origem, isolar módulo afetado e acionar sensor secundário.`,\n\n  // ─────────────────────────────────────────────────────────\n  // SENSOR ESPECÍFICO: FOGO\n  // ─────────────────────────────────────────────────────────\n  fogo: () => `\n<b>🔥 Sensor de Fogo — Pino A1</b><br><br>\n<b>O que detecta:</b> presença de chamas por radiação infravermelha.<br><br>\n<b>⚠ Lógica INVERTIDA:</b><br>\n• Valor ALTO (próximo de 1023) = <b>sem fogo</b> — normal<br>\n• Valor BAIXO (próximo de 0) = <b>fogo detectado</b> — crítico<br><br>\n<b>Limites configurados:</b><br>\n• Normal: acima de ${C.fireLim+50} raw<br>\n• Atenção: ${C.fireLim} a ${C.fireLim+50} raw<br>\n• Crítico: abaixo de ${C.fireLim} raw<br><br>\n<b>Valor atual:</b> ${S.fire !== null ? S.fire+\' raw\' : \'Sem sinal\'}<br><br>\n<b>Se crítico:</b> acionar extintor automático, cortar energia do setor, comunicar base e retirar tripulação.`,\n\n  // ─────────────────────────────────────────────────────────\n  // SENSOR ESPECÍFICO: LUZ\n  // ─────────────────────────────────────────────────────────\n  luz: () => `\n<b>💡 Sensor de Luminosidade — Pino A3</b><br><br>\n<b>O que verifica:</b> se a iluminação interna da cápsula está funcionando.<br><br>\n<b>Como funciona:</b> lê o brilho ambiente via LDR. Se cair abaixo de ${C.luzLim}, indica falha de iluminação.<br><br>\n<b>Importante:</b> o LED de iluminação está ligado <b>diretamente no 5V</b> — o Arduino não controla o LED, apenas verifica se há luz com o sensor.<br><br>\n<b>Limites:</b><br>\n• Funcionando: ≥${C.luzLim} raw<br>\n• Falha: &lt;${C.luzLim} raw<br><br>\n<b>Valor atual:</b> ${S.luz !== null ? S.luz+\' raw\' : \'Sem sinal\'}<br><br>\n<b>Se falha:</b> o buzzer D6 é ativado automaticamente. Ligar backup de emergência e inspecionar o circuito.`,\n\n  // ─────────────────────────────────────────────────────────\n  // SENSOR ESPECÍFICO: TEMPERATURA\n  // ─────────────────────────────────────────────────────────\n  temperatura: () => `\n<b>🌡 Sensor de Temperatura — Pino A4</b><br><br>\n<b>Tipo:</b> Termistor NTC (Negative Temperature Coefficient)<br><br>\n<b>Parâmetros de calibração:</b><br>\n• Resistência nominal: 100kΩ (a 25°C)<br>\n• Resistor fixo divisor: 6kΩ<br>\n• Coeficiente Beta: 3950<br><br>\n<b>Fórmula:</b> Steinhart-Hart simplificada via equação Beta<br><br>\n<b>Limites:</b><br>\n• Normal: 18–30°C<br>\n• Atenção: 31–${C.tempW}°C<br>\n• Crítico: acima de ${C.tempC}°C<br><br>\n<b>Temperatura atual:</b> ${S.tempC !== null ? S.tempC.toFixed(1)+\'°C\' : \'Sem sinal\'}<br>\n<b>Valor bruto:</b> ${S.tempRaw !== null ? S.tempRaw+\' raw (A4)\' : \'—\'}`,\n\n  // ─────────────────────────────────────────────────────────\n  // BUZZER\n  // ─────────────────────────────────────────────────────────\n  buzzer: () => `\n<b>🔔 Buzzer de Emergência — Pino D6</b><br><br>\n<b>O que é:</b> buzzer piezoelétrico que emite sirene sonora.<br><br>\n<b>Quando ativa:</b> o Arduino ativa automaticamente o buzzer quando a luminosidade cai abaixo de ${C.luzLim} (falha de iluminação).<br><br>\n<b>Como funciona a sirene:</b> o Arduino varia a frequência de 600 Hz a 1400 Hz ciclicamente (efeito de subida e descida), com intervalo de 5ms entre cada passo — criando o som de sirene característico.<br><br>\n<b>Estado atual:</b> ${S.buzzer ? \'🔔 ATIVO — sirene em operação\' : \'🔕 Desligado — sistema estável\'}<br><br>\n<b>Na interface:</b> o botão <b>🔊 Som virtual</b> simula o buzzer no navegador. <b>✓ Confirmar</b> reconhece o alerta sem desativar o sensor físico.`,\n\n  // ─────────────────────────────────────────────────────────\n  // STATUS EM TEMPO REAL\n  // ─────────────────────────────────────────────────────────\n  status: () => {\n    const gasV  = S.gas   !== null ? `${S.gas} raw`           : \'— (sem sinal)\';\n    const fireV = S.fire  !== null ? `${S.fire} raw`          : \'— (sem sinal)\';\n    const luzV  = S.luz   !== null ? `${S.luz} raw`           : \'— (sem sinal)\';\n    const tmpV  = S.tempC !== null ? `${S.tempC.toFixed(1)}°C (raw: ${S.tempRaw??\'—\'})` : \'— (sem sinal)\';\n    const buz   = S.buzzer ? \'🔔 ATIVO\' : \'🔕 Desligado\';\n    const conn  = S.connected ? \'🟢 Arduino conectado\' : (S.mode===\'sim\' ? \'🟡 Modo simulação\' : \'🔴 Offline — sem dados\');\n    const mode  = S.mode === \'sim\' ? \'Simulação\' : \'Real\';\n    // Conta alertas\n    const nCrit = [\n      S.gas  >= C.gasLim,\n      S.fire !== null && S.fire < C.fireLim,\n      S.luz  !== null && S.luz  < C.luzLim,\n      S.tempC!== null && S.tempC >= C.tempC,\n    ].filter(Boolean).length;\n    return `\n<b>📊 Telemetria em tempo real — EVORTEX:</b><br><br>\n☁ <b>Gás (A0):</b>           ${gasV}<br>\n🔥 <b>Fogo (A1):</b>          ${fireV}<br>\n💡 <b>Luz (A3):</b>           ${luzV}<br>\n🌡 <b>Temperatura (A4):</b>   ${tmpV}<br>\n🔔 <b>Buzzer D6:</b>          ${buz}<br>\n📡 <b>Conexão:</b>            ${conn}<br>\n🔄 <b>Modo:</b>              ${mode}<br>\n⏱ <b>Uptime:</b>            ${g(\'txt-up\').textContent}<br>\n🚨 <b>Alertas críticos:</b>   ${nCrit > 0 ? nCrit+\' ativo(s)\' : \'Nenhum\'}`;\n  },\n\n  // ─────────────────────────────────────────────────────────\n  // ALERTAS ATIVOS\n  // ─────────────────────────────────────────────────────────\n  alertas: () => {\n    const lista = [];\n    if (S.gas  !== null && S.gas  >= C.gasLim)                        lista.push(`🔴 <b>Gás CRÍTICO:</b> ${S.gas} raw (limite ${C.gasLim})`);\n    else if (S.gas !== null && S.gas >= C.gasLim*0.7)                  lista.push(`🟡 <b>Gás em atenção:</b> ${S.gas} raw`);\n    if (S.fire !== null && S.fire <  C.fireLim)                        lista.push(`🔴 <b>Fogo DETECTADO:</b> sensor ${S.fire} raw (abaixo de ${C.fireLim})`);\n    else if (S.fire !== null && S.fire < C.fireLim*1.35)               lista.push(`🟡 <b>Fogo em atenção:</b> sensor ${S.fire} raw`);\n    if (S.luz  !== null && S.luz  <  C.luzLim)                         lista.push(`🔴 <b>Falha de iluminação:</b> ${S.luz} raw (mínimo ${C.luzLim})`);\n    if (S.tempC!== null && S.tempC >= C.tempC)                         lista.push(`🔴 <b>Temperatura CRÍTICA:</b> ${S.tempC.toFixed(1)}°C (limite ${C.tempC}°C)`);\n    else if (S.tempC !== null && S.tempC >= C.tempW)                   lista.push(`🟡 <b>Temperatura elevada:</b> ${S.tempC.toFixed(1)}°C`);\n    if (S.commFailed)                                                   lista.push(`🔴 <b>Falha de comunicação</b> com o Arduino`);\n    if (!lista.length) return \'✅ <b>Nenhum alerta ativo.</b> Todos os sistemas dentro dos limites operacionais.\';\n    return `<b>Alertas detectados agora:</b><br><br>` + lista.join(\'<br>\');\n  },\n\n  // ─────────────────────────────────────────────────────────\n  // PROTOCOLOS DE EMERGÊNCIA COMPLETOS\n  // ─────────────────────────────────────────────────────────\n  protocolos: () => `\n<b>📋 Protocolos de emergência da Missão EVORTEX:</b><br><br>\n🔥 <b>Protocolo de Incêndio (Fogo A1)</b><br>\n1. Acionar extintor automático imediatamente<br>\n2. Cortar alimentação elétrica do setor afetado<br>\n3. Comunicar alerta à base terrestre<br>\n4. Retirar tripulação do compartimento<br>\n5. Monitorar propagação — se incontrolável: evacuação<br><br>\n☁ <b>Protocolo de Gás (Gás A0)</b><br>\n1. Interromper todas as atividades na câmara<br>\n2. Ativar ventilação forçada<br>\n3. Identificar e isolar a origem do vazamento<br>\n4. Acionar sensor secundário para confirmação<br>\n5. Se persistir: evacuar e selar o módulo<br><br>\n🌡 <b>Protocolo Térmico (Temp A4)</b><br>\n1. Reduzir carga dos sistemas não essenciais<br>\n2. Ativar sistema de resfriamento auxiliar<br>\n3. Monitorar sensor NTC (A4) a cada 30 segundos<br>\n4. Verificar isolamento térmico do módulo<br>\n5. Se &gt;45°C: iniciar protocolo de evacuação<br><br>\n💡 <b>Protocolo de Iluminação (Luz A3)</b><br>\n1. Ativar iluminação de emergência (backup autônomo)<br>\n2. Verificar fusíveis e conexões do circuito<br>\n3. Inspecionar sensor LDR no pino A3<br>\n4. Reduzir consumo elétrico geral<br>\n5. Registrar falha no histórico e aguardar manutenção`,\n\n  // ─────────────────────────────────────────────────────────\n  // CÓDIGO ARDUINO\n  // ─────────────────────────────────────────────────────────\n  codigo: () => `\n<b>💻 Código Arduino da Missão EVORTEX:</b><br><br>\n<b>Linguagem:</b> C/C++ (Arduino IDE)<br>\n<b>Baud rate:</b> 9600<br>\n<b>Intervalo de envio:</b> 500ms<br><br>\n<b>Formato enviado pelo Arduino:</b><br>\n<code>GAS:46,FOGO:987,LUZ:59,TEMP_RAW:57,TEMP:24.3,BUZZER:0</code><br><br>\n<b>Funções principais do sketch:</b><br>\n• <code>calcularTemperaturaCelsius(raw)</code> — converte A4 em °C via fórmula NTC<br>\n• <code>tocarSirene(agora)</code> — varre frequência 600–1400Hz no buzzer<br>\n• <code>desligarSirene()</code> — para o buzzer e reseta frequência<br>\n• <code>setup()</code> — inicia Serial e configura pinos<br>\n• <code>loop()</code> — lê sensores, decide alertas, envia telemetria<br><br>\n<b>Lógica do buzzer:</b> ativado apenas quando <code>valorLuz &lt; 50</code>`,\n\n  // ─────────────────────────────────────────────────────────\n  // INTERFACE — COMO USAR O SISTEMA\n  // ─────────────────────────────────────────────────────────\n  interface: () => `\n<b>🖥️ Como usar a interface EVORTEX:</b><br><br>\n<b>Modos de operação:</b><br>\n• <b>Real</b> — recebe dados do Arduino via USB<br>\n• <b>Simulação</b> — gera dados automaticamente sem Arduino<br><br>\n<b>Barra de simulação (modo Sim):</b><br>\n• <b>▶ Auto</b> — ciclo automático alternando cenários<br>\n• 🔥 Fogo — simula detecção de chama<br>\n• ☁ Gás — simula gás acima do limite<br>\n• 💡 Baixa Luz — simula falha de iluminação<br>\n• 🌡 Temp Alta — simula temperatura crítica<br>\n• 📡 Falha Comm — simula perda de sinal<br>\n• ↺ Restaurar — volta ao estado normal<br><br>\n<b>Painel direito:</b><br>\n• Cápsula animada — muda para vermelho em alertas<br>\n• Status da missão — verde/amarelo/vermelho<br>\n• Buzzer — reflete o estado do D6<br>\n• ⚙ Config — ajuste os limites de cada sensor`,\n\n  // ─────────────────────────────────────────────────────────\n  // CONECTAR O ARDUINO\n  // ─────────────────────────────────────────────────────────\n  arduino: () => `\n<b>📡 Como conectar o Arduino:</b><br><br>\n<b>Requisitos:</b><br>\n• Navegador Google Chrome ou Microsoft Edge<br>\n• Arduino com o sketch EVORTEX gravado<br>\n• Cabo USB conectado ao computador<br><br>\n<b>Passo a passo:</b><br>\n1. Conecte o Arduino via cabo USB<br>\n2. Selecione o modo <b>Real</b> no topo da tela<br>\n3. Clique em <b>⬡ Conectar Arduino</b><br>\n4. Escolha a porta COM na janela que abrirá<br>\n5. Aguarde os dados aparecerem nos cards<br><br>\n<b>Formato aceito:</b><br>\n<code>GAS:46,FOGO:987,LUZ:59,TEMP_RAW:57,TEMP:24.3,BUZZER:0</code><br><br>\n<b>Se não conectar:</b> verifique se o Arduino está na porta correta e se o baud rate está em 9600.`,\n\n  // ─────────────────────────────────────────────────────────\n  // GRÁFICOS E HISTÓRICO\n  // ─────────────────────────────────────────────────────────\n  graficos: () => `\n<b>📈 Gráficos e Histórico da Missão EVORTEX:</b><br><br>\n<b>Gráficos em tempo real:</b><br>\n• 4 gráficos de linha — um para cada sensor<br>\n• Mostram os últimos 60 registros<br>\n• Atualizam a cada leitura do Arduino<br>\n• Abas: <b>Gás / Fogo / Luz / Temperatura</b><br>\n• Botão <b>Limpar</b> — apaga o histórico do gráfico<br><br>\n<b>Histórico da Missão:</b><br>\n• Registra todos os eventos com horário e valor<br>\n• Filtre por sensor (gás, fogo, luz, temp, sistema)<br>\n• Filtre por nível (info, atenção, crítico)<br>\n• <b>⬇ CSV</b> — exporta tudo em planilha<br>\n• <b>⎘ Copiar</b> — copia o relatório para área de transferência<br>\n• <b>ACK</b> — marca um alerta como reconhecido`,\n\n  // ─────────────────────────────────────────────────────────\n  // PAINEL DE ANÁLISE DE ESTRUTURAS\n  // ─────────────────────────────────────────────────────────\n  analise: () => `\n<b>🔍 Painel de Análise de Estruturas (parte inferior):</b><br><br>\nEste painel mostra em tempo real como o código está sendo executado, demonstrando as estruturas de programação:<br><br>\n<b>⎇ Condicionais:</b> cada <code>if/else</code> do sistema acende quando ativo<br>\n• if gás ≥ limite, if fogo &lt; limite, if luz &lt; limiteLuz<br>\n• if temp ≥ crítica, else if temp ≥ atenção<br>\n• if raw ≤1 ou ≥1022 (verifica NTC saturado)<br><br>\n<b>↺ Repetição:</b> contadores de cada loop em execução<br>\n• loop() Arduino, while(serial), forEach(campos)<br>\n• setInterval(checkComm), setInterval(tick)<br><br>\n<b>▦ Vetores:</b> células visuais mostrando os arrays<br>\n• hist[], chartData.gas[], chartData.temp[]<br><br>\n<b>ƒ Funções:</b> log das últimas 14 chamadas com argumentos e retorno`,\n\n  // ─────────────────────────────────────────────────────────\n  // CONFIGURAÇÕES\n  // ─────────────────────────────────────────────────────────\n  configuracoes: () => `\n<b>⚙ Configurações — como ajustar os limites:</b><br><br>\nClique na aba <b>⚙ Config</b> no painel direito.<br><br>\n<b>Limites editáveis:</b><br>\n• <b>Limite Gás</b> — valor raw acima = alerta (padrão: 200)<br>\n• <b>Limite Fogo</b> — valor raw abaixo = alerta (padrão: 300)<br>\n• <b>Limite Luz mínima</b> — igual ao Arduino: 50 raw<br>\n• <b>Temp. Atenção</b> — padrão: 30°C<br>\n• <b>Temp. Crítica</b> — padrão: 40°C<br>\n• <b>Timeout comm</b> — segundos sem dado = falha (padrão: 3s)<br>\n• <b>Volume alarme</b> — volume do som virtual (0 a 1)<br><br>\n<b>Restaurar padrões:</b> botão <b>↺ Restaurar padrões</b><br><br>\n<b>⚠ Atenção:</b> o limite de luz (50) é o mesmo hardcoded no Arduino — mantenha sincronizado.`,\n\n  // ─────────────────────────────────────────────────────────\n  // WEB SERIAL API\n  // ─────────────────────────────────────────────────────────\n  serial: () => `\n<b>🔌 Web Serial API — como funciona a comunicação:</b><br><br>\nA <b>Web Serial API</b> permite que o navegador se comunique diretamente com dispositivos seriais como o Arduino.<br><br>\n<b>Como funciona aqui:</b><br>\n1. Ao clicar em <b>Conectar Arduino</b>, o navegador pede permissão para acessar a porta COM<br>\n2. A interface abre a conexão em 9600 baud<br>\n3. O sistema lê o stream serial continuamente em loop<br>\n4. Cada linha recebida é parseada pela função <code>processarDadosArduino()</code><br>\n5. Os cards e gráficos atualizam em tempo real<br><br>\n<b>Formatos aceitos:</b><br>\n• Compacto: <code>GAS:46,FOGO:987,LUZ:59,TEMP_RAW:57,TEMP:24.3,BUZZER:0</code><br>\n• Verbose: linhas do Monitor Serial do Arduino IDE<br>\n• JSON: <code>{"gas":46,"fogo":987,...}</code><br><br>\n<b>Compatível com:</b> Google Chrome e Microsoft Edge`,\n\n  // ─────────────────────────────────────────────────────────\n  // AJUDA GERAL\n  // ─────────────────────────────────────────────────────────\n  ajuda: () => `\n<b>💬 Sou ARIA — Assistente Inteligente da Missão EVORTEX</b><br><br>\nTenho conhecimento completo sobre tudo neste sistema. Pergunte sobre:<br><br>\n🛰 <b>missão</b> — o que é a Missão EVORTEX<br>\n🎯 <b>objetivos</b> — o que a missão quer alcançar<br>\n🔧 <b>hardware</b> — Arduino, sensores e componentes<br>\n🔬 <b>sensores</b> — todos os 4 sensores e seus limites<br>\n☁ <b>gás</b> / 🔥 <b>fogo</b> / 💡 <b>luz</b> / 🌡 <b>temperatura</b><br>\n🔔 <b>buzzer</b> — como funciona o alarme físico<br>\n📊 <b>status</b> — dados em tempo real agora<br>\n🚨 <b>alertas</b> — o que está crítico neste momento<br>\n📋 <b>protocolos</b> — ações de emergência detalhadas<br>\n💻 <b>código</b> — explicação do sketch Arduino<br>\n🖥️ <b>interface</b> — como usar todos os botões<br>\n📡 <b>arduino</b> — como conectar o hardware<br>\n📈 <b>gráficos</b> — histórico e exportação<br>\n🔍 <b>análise</b> — painel de estruturas de programação<br>\n⚙ <b>configurações</b> — como ajustar os limites<br>\n🔌 <b>serial</b> — como funciona a Web Serial API<br><br>\nOu me faça qualquer pergunta em texto livre!`,\n};\n\n/* ════════════════════════════════════════════════════════════\n   MOTOR NLP — mapeia perguntas em linguagem natural\n   para as respostas da base de conhecimento\n════════════════════════════════════════════════════════════ */\n\nconst ARIA_NLP = [\n  // Identidade e missão\n  { p: /o que (é|eh|e) (essa|a|esta) miss|evortex|sobre (essa|a|esta) miss|apresent|qual.*miss/i,\n    r: () => ARIA_KB.missao() },\n  { p: /objetiv|propósit|finalidad|pra que serve|para que serve/i,\n    r: () => ARIA_KB.objetivos() },\n\n  // Hardware\n  { p: /hardware|componente|arduino|placa|circuito|montagem/i,\n    r: () => ARIA_KB.hardware() },\n\n  // Sensores em geral\n  { p: /^sensores?$|quais.*sensores|lista.*sensores|todos.*sensores|sensores.*monitora/i,\n    r: () => ARIA_KB.sensores() },\n\n  // Sensor gás\n  { p: /gás|gas|mq.?2|vapores|vazamento/i,\n    r: () => ARIA_KB.gas() },\n\n  // Sensor fogo\n  { p: /fogo|incêndio|incendio|chama|fum[oaçã]|infravermelho|a1/i,\n    r: () => ARIA_KB.fogo() },\n\n  // Sensor luz\n  { p: /luz|ldr|luminosidade|iluminação|iluminacao|a3|led/i,\n    r: () => ARIA_KB.luz() },\n\n  // Temperatura\n  { p: /temperatura|térmico|termico|calor|aquecimento|ntc|termistor|a4/i,\n    r: () => ARIA_KB.temperatura() },\n\n  // Buzzer\n  { p: /buzzer|sirene|alarme|som|barulho|d6|beep|sonoro/i,\n    r: () => ARIA_KB.buzzer() },\n\n  // Alertas e emergência\n  { p: /alerta|emergência|emergencia|crítico|critico|perigo|socorro|o que (fazer|devo|posso)/i,\n    r: () => {\n      const alertas = ARIA_KB.alertas();\n      if (alertas.startsWith(\'✅\')) return alertas;\n      return alertas + \'<br><br>Use o chip <b>protocolos</b> para ver as ações de resposta.\';\n    }},\n\n  // Protocolos\n  { p: /protocol|resposta.*emergência|ação.*critic|o que fazer (se|quando|com)/i,\n    r: () => ARIA_KB.protocolos() },\n\n  // Código Arduino\n  { p: /código|codigo|sketch|função|funcao|loop|setup|c\\+\\+|programação|programacao/i,\n    r: () => ARIA_KB.codigo() },\n\n  // Interface / como usar\n  { p: /como (usar|funciona|operar|mexer|navegar)|botões?|interface|painel|tela/i,\n    r: () => ARIA_KB.interface() },\n\n  // Conectar Arduino\n  { p: /conectar|conexão|conexao|porta com|usb|web serial|baud/i,\n    r: () => ARIA_KB.arduino() },\n\n  // Gráficos e histórico\n  { p: /gráfico|grafico|histórico|historico|csv|exportar|log|eventos/i,\n    r: () => ARIA_KB.graficos() },\n\n  // Painel de análise\n  { p: /análise|analise|estrutura|condicional|repetição|vetor|função.*log/i,\n    r: () => ARIA_KB.analise() },\n\n  // Configurações\n  { p: /configuração|configuracao|limite|ajustar|editar|mudar.*limite|restaurar/i,\n    r: () => ARIA_KB.configuracoes() },\n\n  // Serial\n  { p: /serial|comunicação|comunicacao|transmissão|transmissao|pacote|formato/i,\n    r: () => ARIA_KB.serial() },\n\n  // Simulação\n  { p: /simulação|simulacao|simul|demo|apresentação|apresentacao|sem arduino|teste/i,\n    r: () => `<b>🔄 Modo Simulação EVORTEX:</b><br><br>\nUse quando não tiver o Arduino disponível.<br><br>\n<b>Como ativar:</b> clique no botão <b>Simulação</b> no topo.<br><br>\n<b>Botões disponíveis:</b><br>\n• <b>▶ Auto</b> — alterna cenários automaticamente<br>\n• 🔥 <b>Fogo</b> — sensor A1 cai para valor crítico<br>\n• ☁ <b>Gás</b> — sensor A0 sobe além do limite<br>\n• 💡 <b>Baixa Luz</b> — sensor A3 cai abaixo de ${C.luzLim}<br>\n• 🌡 <b>Temp Alta</b> — temperatura sobe acima de ${C.tempC}°C<br>\n• 📡 <b>Falha Comm</b> — simula perda de sinal por 5 segundos<br>\n• ↺ <b>Restaurar</b> — volta tudo ao normal<br><br>\nOs popups de alerta abrirão automaticamente nos cenários críticos.` },\n];\n\n/* ════════════════════════════════════════════════════════════\n   ROTEADOR PRINCIPAL DA ARIA\n   Recebe a mensagem, tenta identificar o tema e retorna\n   a resposta mais adequada da base de conhecimento.\n════════════════════════════════════════════════════════════ */\n\nfunction _ariaResponderNLP(msg) {\n  const m = msg.trim().toLowerCase();\n\n  // ── Comandos exatos (chips) ──────────────────────────────\n  const exatos = {\n    \'status\': ARIA_KB.status,\n    \'sensores\': ARIA_KB.sensores,\n    \'protocolos\': ARIA_KB.protocolos,\n    \'alertas\': ARIA_KB.alertas,\n    \'alertas ativos\': ARIA_KB.alertas,\n    \'comandos\': ARIA_KB.interface,\n    \'ajuda\': ARIA_KB.ajuda,\n    \'help\': ARIA_KB.ajuda,\n    \'oi\': () => `Olá, operador! Como posso ajudar? Digite <b>ajuda</b> para ver tudo que sei sobre a Missão EVORTEX.`,\n    \'olá\': () => `Olá! Sou ARIA, a IA de controle da Missão EVORTEX. Digite <b>ajuda</b> para ver minha base de conhecimento completa.`,\n    \'ola\': () => `Olá! Sou ARIA, a IA de controle da Missão EVORTEX. Digite <b>ajuda</b> para ver minha base de conhecimento completa.`,\n    \'missao\': ARIA_KB.missao,\n    \'missão\': ARIA_KB.missao,\n    \'objetivos\': ARIA_KB.objetivos,\n    \'hardware\': ARIA_KB.hardware,\n    \'gas\': ARIA_KB.gas,\n    \'gás\': ARIA_KB.gas,\n    \'fogo\': ARIA_KB.fogo,\n    \'luz\': ARIA_KB.luz,\n    \'temperatura\': ARIA_KB.temperatura,\n    \'buzzer\': ARIA_KB.buzzer,\n    \'codigo\': ARIA_KB.codigo,\n    \'código\': ARIA_KB.codigo,\n    \'interface\': ARIA_KB.interface,\n    \'arduino\': ARIA_KB.arduino,\n    \'graficos\': ARIA_KB.graficos,\n    \'gráficos\': ARIA_KB.graficos,\n    \'analise\': ARIA_KB.analise,\n    \'análise\': ARIA_KB.analise,\n    \'configuracoes\': ARIA_KB.configuracoes,\n    \'configurações\': ARIA_KB.configuracoes,\n    \'serial\': ARIA_KB.serial,\n  };\n  if (exatos[m]) return exatos[m]();\n\n  // ── Linguagem natural via NLP ────────────────────────────\n  for (const { p, r } of ARIA_NLP) {\n    if (p.test(msg)) return r();\n  }\n\n  // ── Fallback — lista o que a ARIA sabe ──────────────────\n  return `Não encontrei uma resposta específica para "<b>${msg}</b>".<br><br>\nMinha base de conhecimento cobre:<br>\n<b>missão · objetivos · hardware · sensores · gás · fogo · luz · temperatura · buzzer · status · alertas · protocolos · código · interface · arduino · gráficos · análise · configurações · serial</b><br><br>\nTente uma dessas palavras ou faça uma pergunta mais específica. Exemplo: <i>"como funciona o sensor de fogo?"</i>`;\n}\n\n// ── Inicialização com boas-vindas completa ─────────────────────────────────\nfunction ariaInit() {\n  ariaMsgBot(`<b>⬡ ARIA online — Missão EVORTEX</b><br><br>\nSou a <b>ARIA</b> (Artificial Response & Intelligence Assistant), a IA de controle desta missão.<br><br>\nTenho uma base de dados completa sobre tudo neste sistema:<br>\na missão, os sensores, o Arduino, os alertas, os protocolos de emergência, como usar a interface e muito mais.<br><br>\nUse os botões abaixo ou escreva qualquer pergunta.<br>\nDigite <b>ajuda</b> para ver tudo que sei.`);\n}\n\n\n// ── UI do chat ─────────────────────────────────────────────────────────────────\nfunction ariaToggle() {\n  _ariaOpen = !_ariaOpen;\n  const win = g(\'aria-window\');\n  if (_ariaOpen) {\n    win.classList.add(\'open\');\n    g(\'aria-input\').focus();\n    g(\'aria-notif\').classList.remove(\'show\');\n  } else {\n    win.classList.remove(\'open\');\n  }\n}\n\nfunction ariaSend() {\n  const inp = g(\'aria-input\');\n  const msg = inp.value.trim();\n  if (!msg || _ariaTyping) return;\n  inp.value = \'\';\n  ariaMsgUsr(msg);\n  ariaThink(msg);\n}\n\nfunction ariaChip(cmd) {\n  if (_ariaTyping) return;\n  ariaMsgUsr(cmd);\n  ariaThink(cmd);\n}\n\nfunction ariaMsgUsr(txt) {\n  const msgs = g(\'aria-msgs\');\n  const div = document.createElement(\'div\');\n  div.className = \'msg usr\';\n  div.innerHTML = `<div class="msg-bubble">${txt}</div><div class="msg-icon">👤</div>`;\n  msgs.appendChild(div);\n  msgs.scrollTop = msgs.scrollHeight;\n}\n\nfunction ariaMsgBot(html, isAlert) {\n  const msgs = g(\'aria-msgs\');\n  const div = document.createElement(\'div\');\n  div.className = \'msg bot\' + (isAlert ? \' alert-msg\' : \'\');\n  div.innerHTML = `<div class="msg-icon">🤖</div><div class="msg-bubble">${html}</div>`;\n  msgs.appendChild(div);\n  msgs.scrollTop = msgs.scrollHeight;\n}\n\nfunction ariaThink(msg) {\n  _ariaTyping = true;\n  // Indicador de digitação\n  const msgs = g(\'aria-msgs\');\n  const tyDiv = document.createElement(\'div\');\n  tyDiv.className = \'msg bot\'; tyDiv.id = \'aria-typing\';\n  tyDiv.innerHTML = `<div class="msg-icon">🤖</div>\n    <div class="msg-bubble"><div class="typing">\n      <span></span><span></span><span></span>\n    </div></div>`;\n  msgs.appendChild(tyDiv);\n  msgs.scrollTop = msgs.scrollHeight;\n\n  // Simula latência de IA (300–700ms)\n  setTimeout(() => {\n    const tyEl = g(\'aria-typing\');\n    if (tyEl) tyEl.remove();\n    const resp = _ariaResponderNLP(msg);\n    ariaMsgBot(resp);\n    _ariaTyping = false;\n  }, 300 + Math.random() * 400);\n}\n\n// Chamado quando um popup de alerta é detectado — ARIA notifica proativamente\nfunction ariaReceberAlertas(alertas) {\n  if (!_ariaOpen) {\n    // Mostra bolinha de notificação no FAB\n    g(\'aria-notif\').classList.add(\'show\');\n  }\n  const criticos = alertas.filter(a => a.nivel === \'crit\');\n  if (!criticos.length) return;\n\n  const nomes = criticos.map(a => PROTOCOLOS[a.key]?.nome || a.key).join(\', \');\n  ariaMsgBot(\n    `🚨 <b>Alerta crítico detectado!</b><br>\n     Sistemas afetados: <b>${nomes}</b><br><br>\n     Digite <b>protocolos</b> para ver as ações de resposta, ou <b>alertas</b> para o status completo.`,\n    true\n  );\n}\n\n</script>\n</body>\n</html>\n'

with open('missao_evortex.html', 'w', encoding='utf-8') as f:
    f.write(EVORTEX_HTML_CONTENT)

import os
tamanho = os.path.getsize('missao_evortex.html')
print('=' * 55)
print('  ✅ Arquivo gerado com sucesso!')
print('=' * 55)
print(f'  Arquivo  : missao_evortex.html')
print(f'  Tamanho  : {tamanho:,} bytes ({tamanho//1024} KB)')
print()
print('  Para baixar no Colab:')
print('  1. Painel lateral → ícone de pasta 📁')
print('  2. Clique com botão direito em missao_evortex.html')
print('  3. Selecione "Fazer download"')
print()
print('  Abra no Google Chrome ou Microsoft Edge.')
print('  Use o modo Simulação para testar sem Arduino.')
print('=' * 55)

# Download automático no Colab
try:
    from google.colab import files
    files.download('missao_evortex.html')
    print('  📥 Download iniciado automaticamente.')
except ImportError:
    print('  ℹ️  Execute no Google Colab para download automático.')

---
## 📚 Célula 5 — Referência Técnica Completa

### Arquitetura do sistema

```
Arduino Uno
├── A0 → Sensor de Gás (MQ-2)
├── A1 → Sensor de Fogo (IR, lógica invertida)
├── A3 → Sensor de Luz (LDR)
├── A4 → Temperatura NTC (β=3950)
└── D6 → Buzzer (sirene 600–1400 Hz)
     └── Ativa quando: luz < 50
         
Comunicação: Serial 9600 baud → USB → Chrome/Edge
Formato: GAS:46,FOGO:987,LUZ:59,TEMP_RAW:57,TEMP:24.3,BUZZER:0
```

### Fórmula do termistor NTC

```
Rntc = Rfix × ((1023 / leitura) - 1)
Tk   = 1 / ( 1/(T₀+273.15) + ln(Rntc/R₀) / β )
Tc   = Tk - 273.15
```

### Regras de alerta

| Sensor | Normal | Atenção | Crítico |
|---|---|---|---|
| Gás (A0) | < 140 raw | 140–200 raw | > 200 raw |
| Fogo (A1) | > 350 raw | 300–350 raw | < 300 raw |
| Luz (A3) | ≥ 50 raw | — | < 50 raw |
| Temperatura (A4) | 18–30°C | 31–35°C | > 35°C |

### Funções JavaScript do sistema (70 no total)

**Sensores:** `updateGas()` · `updateFire()` · `updateLuz()` · `updateTemp()` · `calcTempC()`

**Sistema:** `updateStatus()` · `setBuzzer()` · `dispatch()` · `processarDadosArduino()`

**Serial:** `conectarArduino()` · `desconectarArduino()` · `readSerial()` · `checkComm()`

**Simulação:** `simAuto()` · `simGas()` · `simFogo()` · `simLuz()` · `simTemp()` · `simComm()` · `simRestore()`

**Popup:** `abrirPopupAlerta()` · `fecharPopup()` · `_verificarPopupNecessario()`

**ARIA:** `ariaInit()` · `ariaToggle()` · `ariaSend()` · `ariaChip()` · `ariaThink()` · `_ariaResponderNLP()` · `ariaReceberAlertas()`

**Análise:** `updateAnlPanel()` · `fnLog()` · `anlUpdateIf()` · `renderVecCells()`

**Histórico:** `addEvt()` · `renderHist()` · `exportCSV()` · `copyReport()`